# MSCNN-BiLSTM Autoencoder — Unsupervised NIDS
### Two-Stage Architecture (Singh & Jang-Jaccard, 2022)
**Stage 1:** Multi-Scale CNN Autoencoder (spatial feature extraction per-flow)
**Stage 2:** Bidirectional LSTM Autoencoder (temporal pattern learning in latent space)

> **Primary Target:** Generalization to CSE-CIC-IDS-2018 (unseen dataset)
> **Secondary Target:** CIC-IDS-2017 detection performance

## 0. SETUP & CONFIGURATION

In [1]:
# ============================================================
# KONFIGURASI UTAMA — ubah di sini saja, jangan hardcode
# ============================================================
import os, random, json, warnings, hashlib, pickle, gc, time
import re
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# --- Reproducibility / Runtime ---
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# --- Paths ---
DRIVE_ROOT        = "/content/drive/MyDrive"
DATA_ROOT         = f"{DRIVE_ROOT}/nids-data/raw"
CIC2017_DIR       = f"{DATA_ROOT}/CIC-IDS2017"
CSE2018_DIR       = f"{DATA_ROOT}/CSE-CIC-IDS2018"
PROJECT_ROOT      = f"{DRIVE_ROOT}/mscnn-bilstm-ae-28-feb"
PIPELINE_VERSION  = "v5_domainfix"
SOURCE_PIPELINE_VERSION = "v3_sessionfix"
SCORE_PIPELINE_VERSION  = PIPELINE_VERSION
SOURCE_PREPROCESS_VERSION = "v3_sessionfix"
SOURCE_SCORING_VERSION = "v4_scoringfix"

SOURCE_PREPROCESSED_DIR = f"{PROJECT_ROOT}/preprocessed/{SOURCE_PREPROCESS_VERSION}"
SOURCE_ARTIFACTS_DIR    = f"{PROJECT_ROOT}/artifacts/{SOURCE_PREPROCESS_VERSION}"
SOURCE_SCORING_ARTIFACTS_DIR = f"{PROJECT_ROOT}/artifacts/{SOURCE_SCORING_VERSION}"
SCORE_ARTIFACTS_DIR     = f"{PROJECT_ROOT}/artifacts/{SCORE_PIPELINE_VERSION}"
SCORE_RESULTS_DIR       = f"{PROJECT_ROOT}/results/{SCORE_PIPELINE_VERSION}"

# Backward-compatible aliases for existing notebook cells
PREPROCESSED_DIR  = SOURCE_PREPROCESSED_DIR
ARTIFACTS_DIR     = SOURCE_ARTIFACTS_DIR
CHECKPOINT_DIR    = f"{PROJECT_ROOT}/checkpoints"
RESULTS_DIR       = f"{PROJECT_ROOT}/results"

# --- New Runtime Flags ---
AUTO_CACHE_PREPROCESSING = False
CACHE_MODE = "auto"
GPU_ACCEL_MODE = "max_speed"
ENABLE_MIXED_PRECISION = True
ENABLE_XLA = True
ENABLE_TF_DATA_PIPELINE = True
ALLOW_NON_DETERMINISTIC = True
PREFETCH_AUTOTUNE = True

# --- Auto Skip Training by Checkpoint ---
AUTO_SKIP_TRAINING_IF_CHECKPOINT = True
STAGE1_CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/stage1_best_{SOURCE_PREPROCESS_VERSION}.keras"
STAGE2_CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/stage2_best_{SOURCE_PREPROCESS_VERSION}.keras"

# --- Evaluation Stabilization Flags ---
STRICT_FEATURE_MATCH = True
MIN_FEATURE_MATCH_RATIO = 0.95
DROP_HEADER_LEAK_ROWS = True
FEATURE_ALIAS_MAP = {
    # normalized_feature_name: [normalized_alias_1, normalized_alias_2, ...]
    # Add aliases incrementally based on observed CSE schema.
}

# --- Sessionization Quality Gates (v3_sessionfix+) ---
STRICT_SESSION_QUALITY = True
ALLOW_ROW_SPLIT_FALLBACK = False
MIN_UNIQUE_SESSIONS = 10000
MIN_TIMESTAMP_VALID_RATIO = 0.90
MAX_SINGLE_SESSION_SHARE = 0.05
SESSION_KEY_MODE = "5tuple"
SESSION_HASH_LEN = 16

# --- Latent / Score Diagnostics ---
ENABLE_LATENT_HEALTH_CHECK = True
LATENT_STD_MAX = 100.0
ENABLE_SCORE_DIRECTION_DIAGNOSTIC = True
AUTO_APPLY_SCORE_INVERSION = False

# --- Run Flags (manual defaults; may be overridden by auto-cache) ---
RERUN_CLEANING        = False
RERUN_FEATURE_SEL     = False
RERUN_DOMAIN_SHIFT    = True
RERUN_SCALING         = True
RERUN_STAGE1_TRAIN    = True
RERUN_LATENT_EXTRACT  = True
RERUN_WINDOWING       = True
RERUN_STAGE2_TRAIN    = True

# --- Feature Selection ---
VARIANCE_THRESHOLD    = 1e-5
CORRELATION_THRESHOLD = 0.98
KS_SHIFT_THRESHOLD    = 0.3

# --- Windowing ---
WINDOW_SIZE           = 10
MIN_FLOWS_PER_SESSION = 3

# --- Stage 1: MSCNN-AE ---
S1_LATENT_DIM         = 8
S1_DROPOUT            = 0.3
S1_BATCH_SIZE         = 512
S1_MAX_EPOCHS         = 100
S1_LR                 = 1e-3
S1_ES_PATIENCE        = 10
S1_LR_PATIENCE        = 5

# --- Stage 2: LSTM-AE ---
S2_LATENT_DIM         = 4
S2_DROPOUT            = 0.3
S2_BATCH_SIZE         = 256
S2_MAX_EPOCHS         = 100
S2_LR                 = 1e-3
S2_ES_PATIENCE        = 10
S2_LR_PATIENCE        = 5

# --- Threshold ---
K_SIGMA_VALUES        = [1.5, 2.0, 2.5, 3.0]
PERCENTILE_VALUES     = [95, 97, 99, 99.5]

# --- Evaluation ---
CHUNK_SIZE            = 500_000

# --- Anomaly Score Weighting (will be optimized in Section 11) ---
STAGE1_WEIGHT         = 0.5
STAGE2_WEIGHT         = 0.5

# --- Part 4 Scoring Redesign ---
SCORE_MODE = "combined_normalized"  # combined_normalized | stage1_only | stage2_only
INVERT_STAGE2_SCORE = True
SCORE_NORMALIZE_PER_STAGE = True
S1_WEIGHT = 0.5
S2_WEIGHT = 0.5
WEIGHT_GRID_SEARCH = True
W1_CANDIDATES = [0.3, 0.5, 0.7, 0.9, 1.0]
CIC_CALIB_RATIO = 0.10
CIC_CALIB_RANDOM_SEED = RANDOM_SEED


# --- Part 5 Domain Shift Mitigation ---
RUN_PART5_DOMAINFIX = True
DOS_HULK_SAMPLE_SIZE = 10000
CSE_BENIGN_THRESHOLD_SAMPLE = 50000
DOMAIN_SHIFT_KS_CUTOFF = 0.30

HIGH_SHIFT_FEATURES = [
    'Flow IAT Min', 'Bwd Header Length', 'Bwd IAT Min',
    'Bwd IAT Max', 'Bwd IAT Mean', 'min_seg_size_forward',
    'URG Flag Count', 'Fwd Header Length.1',
    'Fwd Packet Length Min', 'Init_Win_bytes_forward'
]
PART5_STRATEGIES = ["mask_high_shift", "clip_high_shift"]
MASK_ANCHOR = "median"  # locked

ENABLE_TARGET_DOMAIN_THRESHOLD = True
TARGET_THRESHOLD_DOMAINS = ["cic_benign", "cse_benign"]
TARGET_THRESHOLD_METHOD = "zscore_k1.5"

ENABLE_DOMAIN_SPECIFIC_SCORE_PROFILE = True
CSE_SCORE_SIGN = "auto_from_part4"

# --- Phase 3 Generalization Experiment ---
ENABLE_DOMAIN_ADAPTATION = True
CSE_BENIGN_ADAPT_MAX_ROWS = 300000
ADAPT_MIX_RATIO_CIC = 0.7
ADAPT_MIX_RATIO_CSE = 0.3
S1_FINETUNE_EPOCHS = 20
S2_FINETUNE_EPOCHS = 20
FINETUNE_LR_SCALE = 0.1
RECALIBRATE_THRESHOLD_ON_MIXED_VAL = True
EXPERIMENT_TAG = "phase3_gen_v1"
RERUN_PHASE3_EXPERIMENTS = False


def normalize_colname(name: str) -> str:
    s = '' if name is None else str(name)
    s = unicodedata.normalize('NFKC', s)
    s = s.replace('\ufeff', '')

    replace_map = {
        '\u2013': '-', '\u2014': '-', '\u2212': '-',
        '\u2018': "'", '\u2019': "'", '\u201c': '"', '\u201d': '"',
    }
    for k, v in replace_map.items():
        s = s.replace(k, v)

    s = ''.join(ch for ch in s if ch.isprintable())
    s = s.strip().lower()
    s = s.replace('_', ' ').replace('-', ' ')
    s = re.sub(r'\s+', ' ', s)
    return s


def build_column_lookup(df_columns):
    lookup = {}
    for c in df_columns:
        n = normalize_colname(c)
        if n and n not in lookup:
            lookup[n] = c
    return lookup


def resolve_feature_column(feature_name, lookup, alias_map):
    nfeat = normalize_colname(feature_name)

    if nfeat in lookup:
        return lookup[nfeat]

    aliases = alias_map.get(nfeat, [])
    for a in aliases:
        na = normalize_colname(a)
        if na in lookup:
            return lookup[na]

    compact = nfeat.replace(' ', '')
    for k, v in lookup.items():
        if k.replace(' ', '') == compact:
            return v

    return None


def summarize_feature_mapping(selected_features, df_columns, alias_map):
    lookup = build_column_lookup(df_columns)
    resolved = {}
    missing = []

    for sf in selected_features:
        col = resolve_feature_column(sf, lookup, alias_map)
        if col is None:
            missing.append(sf)
        else:
            resolved[sf] = col

    matched_count = len(resolved)
    total = len(selected_features)
    ratio = matched_count / total if total > 0 else 0.0

    return {
        'resolved_map': resolved,
        'matched_feature_count': matched_count,
        'missing_feature_count': len(missing),
        'missing_feature_list': missing,
        'match_ratio': ratio,
    }




def resolve_canonical_columns(df_columns):
    lookup = build_column_lookup(df_columns)

    target_aliases = {
        'src_ip': ['src ip', 'source ip', 'sourceip', 'ip src', 'srcip'],
        'dst_ip': ['dst ip', 'destination ip', 'destinationip', 'ip dst', 'dstip'],
        'protocol': ['protocol', 'proto'],
        'timestamp': ['timestamp', 'time stamp', 'time'],
        'label': ['label'],
        'flow_id': ['flow id', 'flowid'],
        'src_port': ['src port', 'source port', 'sport'],
        'dst_port': ['dst port', 'destination port', 'dport'],
    }

    resolved = {}
    for target, aliases in target_aliases.items():
        for a in aliases:
            if a in lookup:
                resolved[target] = lookup[a]
                break
    return resolved


def parse_timestamp_robust(series, source_file=None, row_ids=None):
    series = pd.Series(series)
    n = len(series)
    report = {
        'timestamp_col_detected': True,
        'timestamp_parser_used': None,
        'timestamp_valid_ratio': 0.0,
    }

    ts_candidate = pd.Series([pd.NaT] * n)

    try:
        ts_candidate = pd.to_datetime(series, format='mixed', dayfirst=True, utc=True, errors='coerce')
        report['timestamp_parser_used'] = 'mixed_dayfirst_utc'
    except Exception:
        pass

    valid_ratio = float(ts_candidate.notna().mean())

    if valid_ratio < MIN_TIMESTAMP_VALID_RATIO:
        try:
            ts2 = pd.to_datetime(series, utc=True, errors='coerce')
            if ts2.notna().mean() > ts_candidate.notna().mean():
                ts_candidate = ts2
                report['timestamp_parser_used'] = 'generic_utc'
        except Exception:
            pass
        valid_ratio = float(ts_candidate.notna().mean())

    if valid_ratio < MIN_TIMESTAMP_VALID_RATIO:
        num = pd.to_numeric(series, errors='coerce')
        if float(num.notna().mean()) > 0:
            m = float(num.dropna().abs().median()) if num.notna().any() else 0.0
            if m > 1e16:
                num = num / 1e9
            elif m > 1e13:
                num = num / 1e6
            elif m > 1e11:
                num = num / 1e3
            ts_num = pd.to_datetime(num, unit='s', utc=True, errors='coerce')
            if ts_num.notna().mean() > ts_candidate.notna().mean():
                ts_candidate = ts_num
                report['timestamp_parser_used'] = 'numeric_epoch'
        valid_ratio = float(ts_candidate.notna().mean())

    if row_ids is None:
        row_ids = np.arange(n, dtype=np.int64)
    else:
        row_ids = np.asarray(row_ids, dtype=np.int64)

    if source_file is None:
        src_series = pd.Series(['unknown_source'] * n)
    elif isinstance(source_file, (pd.Series, np.ndarray, list, tuple)):
        src_series = pd.Series(source_file).astype(str)
    else:
        src_series = pd.Series([str(source_file)] * n)

    src_codes = pd.factorize(src_series)[0].astype(np.int64)
    deterministic_fallback = src_codes * 10_000_000_000 + row_ids

    ts_seconds = pd.to_numeric((ts_candidate.view('int64') // 10**9), errors='coerce')
    ts_seconds = ts_seconds.fillna(pd.Series(deterministic_fallback, index=ts_seconds.index)).astype(np.int64)

    report['timestamp_valid_ratio'] = float(valid_ratio)
    if report['timestamp_parser_used'] is None:
        report['timestamp_parser_used'] = 'deterministic_source_row'
    if valid_ratio < MIN_TIMESTAMP_VALID_RATIO:
        report['timestamp_parser_used'] = 'deterministic_source_row'

    return ts_seconds.values, report


def build_session_id(df, canonical_cols, mode='5tuple', hash_len=12):
    mode = str(mode).lower().strip()

    def _series(col_key, default='unk'):
        if col_key in canonical_cols:
            col = canonical_cols[col_key]
            s = df[col]
            if isinstance(s, pd.DataFrame):
                s = s.iloc[:, 0]
            s = s.astype(object)
            s = s.where(s.notna(), default)
            s = s.astype(str).replace({'nan': default, 'None': default})
            return s
        return pd.Series([default] * len(df), index=df.index)

    def _finalize(session_key, label):
        session_id = session_key.map(lambda x: hashlib.md5(str(x).encode()).hexdigest()[:hash_len]).values
        sid_counts = pd.Series(session_id).value_counts()
        largest_share = float(sid_counts.max() / len(session_id)) if len(session_id) > 0 else 0.0
        singleton_ratio = float((sid_counts == 1).mean()) if len(sid_counts) > 0 else 1.0
        return session_id, {
            'fallback_path_used': label,
            'unique_sessions': int(len(sid_counts)),
            'largest_session_share': largest_share,
            'singleton_session_ratio': singleton_ratio,
        }

    src_ip = _series('src_ip')
    dst_ip = _series('dst_ip')
    proto = _series('protocol')
    src_port = _series('src_port')
    dst_port = _series('dst_port')
    src_file = df['__source_file'].astype(str) if '__source_file' in df.columns else pd.Series(['unknown_source'] * len(df), index=df.index)

    if '__row_id_in_file' in df.columns:
        row_raw = pd.to_numeric(df['__row_id_in_file'], errors='coerce')
        if isinstance(row_raw, pd.DataFrame):
            row_raw = row_raw.iloc[:, 0]
        row_raw_np = row_raw.to_numpy(dtype=np.float64, copy=False)
        fallback_np = np.arange(len(df), dtype=np.int64)
        row_idx_np = np.where(np.isfinite(row_raw_np), row_raw_np, fallback_np).astype(np.int64)
        row_idx_num = pd.Series(row_idx_np, index=df.index)
    else:
        row_idx_num = pd.Series(np.arange(len(df), dtype=np.int64), index=df.index)

    has_5tuple = all(k in canonical_cols for k in ['src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port'])
    has_3tuple = all(k in canonical_cols for k in ['src_ip', 'dst_ip', 'protocol'])
    has_proto = 'protocol' in canonical_cols
    has_dport = 'dst_port' in canonical_cols

    lo_ip = np.where(src_ip.values <= dst_ip.values, src_ip.values, dst_ip.values)
    hi_ip = np.where(src_ip.values <= dst_ip.values, dst_ip.values, src_ip.values)
    pair_ip = pd.Series(lo_ip, index=df.index).astype(str) + '|' + pd.Series(hi_ip, index=df.index).astype(str)

    row_bucket_50 = (row_idx_num // 50).astype(str)
    row_bucket_100 = (row_idx_num // 100).astype(str)

    candidates = []

    if mode == '5tuple' and has_5tuple:
        candidates.append((src_ip + '|' + dst_ip + '|' + proto + '|' + src_port + '|' + dst_port, '5tuple'))

    if 'flow_id' in canonical_cols:
        candidates.append((_series('flow_id'), 'flow_id'))

    if has_3tuple:
        candidates.append((src_ip + '|' + dst_ip + '|' + proto, '3tuple_directional'))
        candidates.append((pair_ip + '|' + proto, '3tuple_bidirectional'))
        candidates.append((src_file + '|' + pair_ip + '|' + proto, 'sourcefile_3tuple_bidirectional'))

    if has_dport and has_proto:
        candidates.append((src_file + '|' + dst_port + '|' + proto, 'sourcefile_dport_proto'))
        candidates.append((dst_port + '|' + proto, 'dport_proto_global'))

    if has_dport:
        candidates.append((src_file + '|' + dst_port, 'sourcefile_dport'))

    # Deterministic pseudo-session fallback for datasets missing IP/timestamp metadata.
    candidates.append((src_file + '|' + row_bucket_50, 'sourcefile_rowbucket50'))
    candidates.append((src_file + '|' + row_bucket_100, 'sourcefile_rowbucket100'))
    candidates.append((src_file + '|row|' + row_idx_num.astype(str), 'source_row'))

    evaluated = []
    for key, label in candidates:
        sid, rep = _finalize(key, label)
        evaluated.append((sid, rep))

    selected_sid, selected_rep = evaluated[0]
    for sid, rep in evaluated:
        good_singleton = rep['singleton_session_ratio'] <= 0.70
        good_largest = rep['largest_session_share'] <= MAX_SINGLE_SESSION_SHARE
        if good_singleton and good_largest:
            selected_sid, selected_rep = sid, rep
            break
    else:
        # If no candidate passes strict gates, choose the least bad deterministic option.
        selected_sid, selected_rep = min(
            evaluated,
            key=lambda x: (x[1]['singleton_session_ratio'], x[1]['largest_session_share'])
        )
        selected_rep = dict(selected_rep)
        selected_rep['fallback_path_used'] = selected_rep['fallback_path_used'] + '_degraded'

    return selected_sid, selected_rep

def _exists(path):
    return os.path.exists(path)


def _all_exist(paths):
    return all(_exists(p) for p in paths)


def _checkpoint_candidates(path):
    candidates = [path]
    if path.startswith('/content/drive/MyDrive/'):
        candidates.append(path.replace('/content/drive/MyDrive', '/mydrive', 1))
    elif path.startswith('/mydrive/'):
        candidates.append(path.replace('/mydrive', '/content/drive/MyDrive', 1))
    return list(dict.fromkeys(candidates))


def _resolve_existing_path(path):
    for cand in _checkpoint_candidates(path):
        if os.path.exists(cand):
            return cand
    return path


def apply_checkpoint_auto_skip(log_prefix='Config'):
    global STAGE1_CHECKPOINT_PATH, STAGE2_CHECKPOINT_PATH
    global RERUN_STAGE1_TRAIN, RERUN_STAGE2_TRAIN

    STAGE1_CHECKPOINT_PATH = _resolve_existing_path(STAGE1_CHECKPOINT_PATH)
    STAGE2_CHECKPOINT_PATH = _resolve_existing_path(STAGE2_CHECKPOINT_PATH)

    stage1_ckpt_exists = os.path.exists(STAGE1_CHECKPOINT_PATH)
    stage2_ckpt_exists = os.path.exists(STAGE2_CHECKPOINT_PATH)

    if AUTO_SKIP_TRAINING_IF_CHECKPOINT:
        if stage1_ckpt_exists:
            RERUN_STAGE1_TRAIN = False
        if stage2_ckpt_exists:
            RERUN_STAGE2_TRAIN = False

    print(f"\n=== Checkpoint Training Control ({log_prefix}) ===")
    print(f"STAGE1_CHECKPOINT_PATH      : {STAGE1_CHECKPOINT_PATH}")
    print(f"STAGE2_CHECKPOINT_PATH      : {STAGE2_CHECKPOINT_PATH}")
    print(f"stage1_ckpt_exists          : {stage1_ckpt_exists}")
    print(f"stage2_ckpt_exists          : {stage2_ckpt_exists}")
    print(f"RERUN_STAGE1_TRAIN effective: {RERUN_STAGE1_TRAIN}")
    print(f"RERUN_STAGE2_TRAIN effective: {RERUN_STAGE2_TRAIN}")

    return stage1_ckpt_exists, stage2_ckpt_exists


def resolve_preprocessing_plan():
    cleaning_files = [
        f"{PREPROCESSED_DIR}/cic2017_benign_cleaned.parquet",
        f"{PREPROCESSED_DIR}/cic2017_benign_meta.parquet",
        f"{PREPROCESSED_DIR}/cic2017_all_raw.parquet",
        f"{ARTIFACTS_DIR}/median_values.json",
    ]
    feature_files = [
        f"{ARTIFACTS_DIR}/selected_features.json",
    ]
    scaling_files = [
        f"{PREPROCESSED_DIR}/X_train.npy",
        f"{PREPROCESSED_DIR}/X_val.npy",
        f"{PREPROCESSED_DIR}/train_session_ids.npy",
        f"{PREPROCESSED_DIR}/val_session_ids.npy",
        f"{PREPROCESSED_DIR}/train_timestamps.npy",
        f"{PREPROCESSED_DIR}/val_timestamps.npy",
        f"{ARTIFACTS_DIR}/robust_scaler.pkl",
    ]
    latent_files = [
        f"{PREPROCESSED_DIR}/latent_train.npy",
        f"{PREPROCESSED_DIR}/latent_val.npy",
    ]
    window_files = [
        f"{PREPROCESSED_DIR}/windows_train.npz",
        f"{PREPROCESSED_DIR}/windows_val.npz",
        f"{PREPROCESSED_DIR}/flow_idx_train.npy",
        f"{PREPROCESSED_DIR}/flow_idx_val.npy",
    ]

    status = {
        'cleaning_ready': _all_exist(cleaning_files),
        'feature_ready': _all_exist(feature_files),
        'scaling_ready': _all_exist(scaling_files),
        'latent_ready': _all_exist(latent_files),
        'window_ready': _all_exist(window_files),
    }

    plan = {}
    plan['RERUN_CLEANING'] = not status['cleaning_ready']
    plan['RERUN_FEATURE_SEL'] = plan['RERUN_CLEANING'] or (not status['feature_ready'])
    plan['RERUN_DOMAIN_SHIFT'] = plan['RERUN_FEATURE_SEL'] and (not status['feature_ready'])
    plan['RERUN_SCALING'] = plan['RERUN_FEATURE_SEL'] or (not status['scaling_ready'])
    plan['RERUN_LATENT_EXTRACT'] = plan['RERUN_SCALING'] or (not status['latent_ready'])
    plan['RERUN_WINDOWING'] = plan['RERUN_LATENT_EXTRACT'] or (not status['window_ready'])

    print("\n=== Auto Cache Status ===")
    for k, v in status.items():
        print(f"{k:>18}: {v}")

    print("\n=== Effective Preprocessing Flags ===")
    for k in ['RERUN_CLEANING', 'RERUN_FEATURE_SEL', 'RERUN_DOMAIN_SHIFT',
              'RERUN_SCALING', 'RERUN_LATENT_EXTRACT', 'RERUN_WINDOWING']:
        print(f"{k:>18}: {plan[k]}")

    return plan


if AUTO_CACHE_PREPROCESSING and CACHE_MODE == "auto":
    _plan = resolve_preprocessing_plan()
    RERUN_CLEANING = _plan['RERUN_CLEANING']
    RERUN_FEATURE_SEL = _plan['RERUN_FEATURE_SEL']
    RERUN_DOMAIN_SHIFT = _plan['RERUN_DOMAIN_SHIFT']
    RERUN_SCALING = _plan['RERUN_SCALING']
    RERUN_LATENT_EXTRACT = _plan['RERUN_LATENT_EXTRACT']
    RERUN_WINDOWING = _plan['RERUN_WINDOWING']

stage1_ckpt_exists, stage2_ckpt_exists = apply_checkpoint_auto_skip('Pre-mount')

if GPU_ACCEL_MODE == "max_speed" and ALLOW_NON_DETERMINISTIC:
    os.environ.pop('TF_DETERMINISTIC_OPS', None)
else:
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

import tensorflow as tf
from tensorflow.keras import mixed_precision

tf.random.set_seed(RANDOM_SEED)

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

if gpus and GPU_ACCEL_MODE == "max_speed":
    if ENABLE_MIXED_PRECISION:
        mixed_precision.set_global_policy('mixed_float16')
    if ENABLE_XLA:
        tf.config.optimizer.set_jit(True)
else:
    mixed_precision.set_global_policy('float32')

policy_name = mixed_precision.global_policy().name
xla_status = bool(tf.config.optimizer.get_jit())
deterministic_status = os.environ.get('TF_DETERMINISTIC_OPS', '0') == '1'

print(f"TensorFlow {tf.__version__}")
print("\n=== Runtime Summary ===")
print(f"GPU count           : {len(gpus)}")
print(f"GPU names           : {[g.name for g in gpus] if gpus else []}")
print(f"Mixed precision     : {policy_name}")
print(f"XLA JIT             : {xla_status}")
print(f"Deterministic ops   : {deterministic_status}")

print("\n=== Configuration Loaded ===")
print(f"Random seed: {RANDOM_SEED}")
print(f"Stage 1 latent dim: {S1_LATENT_DIM}")
print(f"Stage 2 latent dim: {S2_LATENT_DIM}")
print(f"Window size: {WINDOW_SIZE}")
print(f"AUTO_CACHE_PREPROCESSING: {AUTO_CACHE_PREPROCESSING}")
print(f"GPU_ACCEL_MODE: {GPU_ACCEL_MODE}")
print(f"AUTO_SKIP_TRAINING_IF_CHECKPOINT: {AUTO_SKIP_TRAINING_IF_CHECKPOINT}")
print(f"STRICT_FEATURE_MATCH={STRICT_FEATURE_MATCH}, MIN_FEATURE_MATCH_RATIO={MIN_FEATURE_MATCH_RATIO:.2f}")
print(f"PIPELINE_VERSION: {PIPELINE_VERSION}")
print(f"SOURCE_PIPELINE_VERSION: {SOURCE_PIPELINE_VERSION}")
print(f"SOURCE_PREPROCESS_VERSION: {SOURCE_PREPROCESS_VERSION}")
print(f"SOURCE_SCORING_VERSION: {SOURCE_SCORING_VERSION}")
print(f"SOURCE_SCORING_ARTIFACTS_DIR: {SOURCE_SCORING_ARTIFACTS_DIR}")
print(f"SOURCE_PREPROCESSED_DIR: {SOURCE_PREPROCESSED_DIR}")
print(f"SCORE_ARTIFACTS_DIR: {SCORE_ARTIFACTS_DIR}")
print(f"SCORE_RESULTS_DIR: {SCORE_RESULTS_DIR}")
print(f"SCORE_MODE={SCORE_MODE}, INVERT_STAGE2_SCORE={INVERT_STAGE2_SCORE}, WEIGHT_GRID_SEARCH={WEIGHT_GRID_SEARCH}")
print(f"STRICT_SESSION_QUALITY={STRICT_SESSION_QUALITY}, MIN_UNIQUE_SESSIONS={MIN_UNIQUE_SESSIONS}")
print(f"MIN_TIMESTAMP_VALID_RATIO={MIN_TIMESTAMP_VALID_RATIO:.2f}, MAX_SINGLE_SESSION_SHARE={MAX_SINGLE_SESSION_SHARE:.2f}")








=== Checkpoint Training Control (Pre-mount) ===
STAGE1_CHECKPOINT_PATH      : /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/checkpoints/stage1_best_v3_sessionfix.keras
STAGE2_CHECKPOINT_PATH      : /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/checkpoints/stage2_best_v3_sessionfix.keras
stage1_ckpt_exists          : False
stage2_ckpt_exists          : False
RERUN_STAGE1_TRAIN effective: True
RERUN_STAGE2_TRAIN effective: True
TensorFlow 2.19.0

=== Runtime Summary ===
GPU count           : 1
GPU names           : ['/physical_device:GPU:0']
Mixed precision     : mixed_float16
XLA JIT             : True
Deterministic ops   : False

=== Configuration Loaded ===
Random seed: 42
Stage 1 latent dim: 8
Stage 2 latent dim: 4
Window size: 10
AUTO_CACHE_PREPROCESSING: False
GPU_ACCEL_MODE: max_speed
AUTO_SKIP_TRAINING_IF_CHECKPOINT: True
STRICT_FEATURE_MATCH=True, MIN_FEATURE_MATCH_RATIO=0.95
PIPELINE_VERSION: v5_domainfix
SOURCE_PIPELINE_VERSION: v3_sessionfix
SOURCE_PREPROCESS_VERSION: v3_ses

## 1. MOUNT GOOGLE DRIVE & DIRECTORY SETUP

In [2]:
from google.colab import drive
drive.mount('/content/drive')

for d in [PREPROCESSED_DIR, CHECKPOINT_DIR, RESULTS_DIR, ARTIFACTS_DIR, SCORE_ARTIFACTS_DIR, SCORE_RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# CSE 2018 abbreviated columns -> CIC 2017 style aliases
FEATURE_ALIAS_MAP.update({
    'total length of fwd packets': ['totlen fwd pkts'],
    'total length of bwd packets': ['totlen bwd pkts'],
    'fwd packet length max': ['fwd pkt len max'],
    'fwd packet length min': ['fwd pkt len min'],
    'fwd packet length std': ['fwd pkt len std'],
    'bwd packet length max': ['bwd pkt len max'],
    'bwd packet length min': ['bwd pkt len min'],
    'bwd packet length std': ['bwd pkt len std'],
    'flow bytes/s': ['flow byts/s'],
    'flow packets/s': ['flow pkts/s'],
    'bwd header length': ['bwd header len'],
    'bwd packets/s': ['bwd pkts/s'],
    'min packet length': ['pkt len min'],
    'max packet length': ['pkt len max'],
    'packet length std': ['pkt len std'],
    'packet length variance': ['pkt len var'],
    'fin flag count': ['fin flag cnt'],
    'syn flag count': ['syn flag cnt'],
    'psh flag count': ['psh flag cnt'],
    'ack flag count': ['ack flag cnt'],
    'urg flag count': ['urg flag cnt'],
    'ece flag count': ['ece flag cnt', 'cwe flag count'],
    'average packet size': ['pkt size avg'],
    'avg fwd segment size': ['fwd seg size avg'],
    'avg bwd segment size': ['bwd seg size avg'],
    'fwd header length.1': ['fwd header len'],
    'init win bytes forward': ['init fwd win byts'],
    'init win bytes backward': ['init bwd win byts'],
    'act data pkt fwd': ['fwd act data pkts'],
    'min seg size forward': ['fwd seg size min'],
})

# Re-check checkpoint existence after Drive is mounted
stage1_ckpt_exists, stage2_ckpt_exists = apply_checkpoint_auto_skip('Post-mount')

print("Drive mounted. Directories ready.")
print(f"CIC-2017 : {CIC2017_DIR}")
print(f"CSE-2018 : {CSE2018_DIR}")
print(f"Project  : {PROJECT_ROOT}")

print(f"Feature alias entries: {len(FEATURE_ALIAS_MAP)}")

cic_files = [f for f in os.listdir(CIC2017_DIR) if f.endswith('.csv')]
print(f"\nCIC-2017 CSV files ({len(cic_files)}):")
for f in sorted(cic_files):
    print(f"  - {f}")

cse_files = [f for f in os.listdir(CSE2018_DIR) if f.endswith('.csv')]
print(f"\nCSE-2018 CSV files ({len(cse_files)}):")
for f in sorted(cse_files):
    print(f"  - {f}")

# Part 4 guard: ensure v3 source cache/checkpoints exist
required_v3_assets = [
    f"{SOURCE_PREPROCESSED_DIR}/X_train.npy",
    f"{SOURCE_PREPROCESSED_DIR}/X_val.npy",
    f"{SOURCE_PREPROCESSED_DIR}/windows_train.npz",
    f"{SOURCE_PREPROCESSED_DIR}/windows_val.npz",
    f"{SOURCE_PREPROCESSED_DIR}/flow_idx_train.npy",
    f"{SOURCE_PREPROCESSED_DIR}/flow_idx_val.npy",
    f"{SOURCE_ARTIFACTS_DIR}/selected_features.json",
    f"{SOURCE_ARTIFACTS_DIR}/median_values.json",
    STAGE1_CHECKPOINT_PATH,
    STAGE2_CHECKPOINT_PATH,
]
missing_v3 = [x for x in required_v3_assets if not os.path.exists(x)]
if missing_v3:
    raise FileNotFoundError(
        "Part 4 requires v3_sessionfix cache/checkpoints. Missing: " + ", ".join(missing_v3)
    )
print(f"[OK] v3 source assets ready: {len(required_v3_assets)} files")



# Part 5 guard: ensure v4 scoring artifacts exist
required_v4_scoring_assets = [
    f"{SOURCE_SCORING_ARTIFACTS_DIR}/score_calibrator.json",
    f"{SOURCE_SCORING_ARTIFACTS_DIR}/score_direction_report_v4.json",
]
missing_v4 = [x for x in required_v4_scoring_assets if not os.path.exists(x)]
if missing_v4:
    raise FileNotFoundError(
        "Part 5 requires v4_scoringfix artifacts. Missing: " + ", ".join(missing_v4)
    )
print(f"[OK] v4 scoring artifacts ready: {len(required_v4_scoring_assets)} files")



Mounted at /content/drive

=== Checkpoint Training Control (Post-mount) ===
STAGE1_CHECKPOINT_PATH      : /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/checkpoints/stage1_best_v3_sessionfix.keras
STAGE2_CHECKPOINT_PATH      : /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/checkpoints/stage2_best_v3_sessionfix.keras
stage1_ckpt_exists          : True
stage2_ckpt_exists          : True
RERUN_STAGE1_TRAIN effective: False
RERUN_STAGE2_TRAIN effective: False
Drive mounted. Directories ready.
CIC-2017 : /content/drive/MyDrive/nids-data/raw/CIC-IDS2017
CSE-2018 : /content/drive/MyDrive/nids-data/raw/CSE-CIC-IDS2018
Project  : /content/drive/MyDrive/mscnn-bilstm-ae-28-feb
Feature alias entries: 30

CIC-2017 CSV files (8):
  - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  - Friday-WorkingHours-Morning.pcap_ISCX.csv
  - Monday-WorkingHours.pcap_ISCX.csv
  - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  - Thursday-Worki

## 2. DATA LOADING & CLEANING

Pipeline: `str('Infinity') → np.inf → np.nan → median imputation`

**Key decisions:**
- Session ID and timestamps are extracted BEFORE dropping columns
- Protocol is retained as a numeric feature (already integer-encoded: 6=TCP, 17=UDP)
- Median imputer is fit ONLY on CIC-2017 benign data

In [3]:
# ============================================================
# 2.1 Load CIC-IDS-2017 - All CSVs, filter BENIGN only
# ============================================================
if RERUN_CLEANING:
    print("=== Loading CIC-IDS-2017 ===")

    csv_files = sorted([
        os.path.join(CIC2017_DIR, f)
        for f in os.listdir(CIC2017_DIR) if f.endswith('.csv')
    ])

    dfs = []
    for fpath in csv_files:
        fname = os.path.basename(fpath)
        print(f"  Loading {fname}...", end=" ")
        try:
            df = pd.read_csv(fpath, low_memory=False, encoding='utf-8')
        except OSError as e:
            msg = str(e)
            if 'Transport endpoint is not connected' in msg:
                raise OSError(
                    "Google Drive mount disconnected while reading CIC CSV. "
                    "Please re-mount Drive (force_remount=True), then rerun from Section 2.1. "
                    f"Failed file: {fname}"
                ) from e
            raise

        df.columns = df.columns.str.strip()

        # Provenance columns for deterministic fallback paths
        df['__source_file'] = fname
        df['__row_id_in_file'] = np.arange(len(df), dtype=np.int64)

        print(f"{len(df)} rows, {len(df.columns)} cols")
        dfs.append(df)

    df_all = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal CIC-2017 rows: {len(df_all):,}")

    canonical_all = resolve_canonical_columns(df_all.columns)
    if 'label' not in canonical_all:
        raise ValueError("Could not resolve Label column from CIC-2017 files.")

    label_col = canonical_all['label']
    print(f"Using label column: {label_col}")
    print(f"Label distribution:\n{df_all[label_col].astype(str).str.strip().value_counts()}")

    df_benign = df_all[df_all[label_col].astype(str).str.strip().str.upper() == 'BENIGN'].copy()
    print(f"\nBenign flows: {len(df_benign):,} ({len(df_benign)/len(df_all)*100:.1f}%)")

    del dfs
    gc.collect()
else:
    print("[SKIP] Data loading - using cached data")


[SKIP] Data loading - using cached data


In [4]:
# ============================================================
# 2.2 Extract metadata columns BEFORE dropping them
# ============================================================
if RERUN_CLEANING:
    canonical = resolve_canonical_columns(df_benign.columns)

    rename_map = {}
    if 'src_ip' in canonical: rename_map[canonical['src_ip']] = 'Src_IP'
    if 'dst_ip' in canonical: rename_map[canonical['dst_ip']] = 'Dst_IP'
    if 'protocol' in canonical: rename_map[canonical['protocol']] = 'Protocol'
    if 'timestamp' in canonical: rename_map[canonical['timestamp']] = 'Timestamp'
    if 'label' in canonical: rename_map[canonical['label']] = 'Label'
    if 'flow_id' in canonical: rename_map[canonical['flow_id']] = 'Flow_ID'
    if 'src_port' in canonical: rename_map[canonical['src_port']] = 'Src_Port'
    if 'dst_port' in canonical: rename_map[canonical['dst_port']] = 'Dst_Port'

    df_benign_renamed = df_benign.rename(columns=rename_map)
    df_all_renamed = df_all.rename(columns=rename_map)

    if '__source_file' not in df_benign_renamed.columns:
        df_benign_renamed['__source_file'] = 'unknown_source'
    if '__row_id_in_file' not in df_benign_renamed.columns:
        df_benign_renamed['__row_id_in_file'] = np.arange(len(df_benign_renamed), dtype=np.int64)

    print("Constructing session IDs...")
    canonical_after = resolve_canonical_columns(df_benign_renamed.columns)
    session_id, sid_report = build_session_id(
        df_benign_renamed,
        canonical_after,
        mode=SESSION_KEY_MODE,
        hash_len=SESSION_HASH_LEN,
    )
    df_benign_renamed['session_id'] = session_id

    ts_report = {
        'timestamp_col_detected': False,
        'timestamp_parser_used': 'deterministic_source_row',
        'timestamp_valid_ratio': 0.0,
    }

    if 'timestamp' in canonical_after:
        ts_col = canonical_after['timestamp']
        ts_vals, ts_report = parse_timestamp_robust(
            df_benign_renamed[ts_col],
            source_file=df_benign_renamed['__source_file'],
            row_ids=df_benign_renamed['__row_id_in_file'],
        )
        df_benign_renamed['ts_parsed'] = ts_vals
        print(f"Timestamp column resolved: {ts_col}")
    else:
        ts_vals, ts_report = parse_timestamp_robust(
            pd.Series([np.nan] * len(df_benign_renamed)),
            source_file=df_benign_renamed['__source_file'],
            row_ids=df_benign_renamed['__row_id_in_file'],
        )
        ts_report['timestamp_col_detected'] = False
        df_benign_renamed['ts_parsed'] = ts_vals
        print("[PERINGATAN] Timestamp column not found - using deterministic source_file+row fallback")

    meta_cols = ['session_id', 'ts_parsed', '__source_file', '__row_id_in_file']
    benign_meta = df_benign_renamed[meta_cols].copy()

    session_counts = benign_meta['session_id'].value_counts()
    unique_sessions = int(session_counts.shape[0])
    largest_session_share = float(session_counts.max() / len(benign_meta)) if len(benign_meta) else 0.0

    session_quality_report = {
        'pipeline_version': PIPELINE_VERSION,
        'session_key_mode': SESSION_KEY_MODE,
        'timestamp_col_detected': bool(ts_report['timestamp_col_detected']),
        'timestamp_parser_used': ts_report['timestamp_parser_used'],
        'timestamp_valid_ratio': float(ts_report['timestamp_valid_ratio']),
        'unique_sessions': unique_sessions,
        'largest_session_share': largest_session_share,
        'singleton_session_ratio': float(sid_report.get('singleton_session_ratio', float('nan'))),
        'fallback_path_used': sid_report['fallback_path_used'],
        'rows': int(len(benign_meta)),
    }

    with open(f"{ARTIFACTS_DIR}/session_quality_report.json", 'w', encoding='utf-8') as f:
        json.dump(session_quality_report, f, indent=2)

    print(f"Unique sessions: {unique_sessions:,}")
    print(f"Largest session share: {largest_session_share:.4f}")
    print(f"Timestamp valid ratio: {ts_report['timestamp_valid_ratio']:.2%}")
    print(f"Timestamp parser: {ts_report['timestamp_parser_used']}")
    print(f"Session fallback path: {sid_report['fallback_path_used']}")
    if 'singleton_session_ratio' in sid_report:
        print(f"Singleton session ratio: {sid_report['singleton_session_ratio']*100:.1f}%")
    print(f"Session ID sample: {df_benign_renamed['session_id'].iloc[:3].tolist()}")


In [5]:
# ============================================================
# 2.3 Drop non-numeric columns & handle Protocol
# ============================================================
if RERUN_CLEANING:
    drop_patterns = ['Label', 'Flow_ID', 'Timestamp', 'Src_IP', 'Dst_IP',
                     'Src_Port', 'Dst_Port', 'session_id', 'ts_parsed', '__source_file', '__row_id_in_file']

    cols_to_drop = [c for c in df_benign_renamed.columns
                    if any(p in c for p in drop_patterns)]

    # [SARAN] Protocol is already numeric (6=TCP, 17=UDP, 1=ICMP).
    # Keeping it as-is since the integer values carry ordinal meaning
    # relevant to network behavior. No encoding needed.
    if 'Protocol' in cols_to_drop:
        cols_to_drop.remove('Protocol')
        print("[SARAN] Keeping 'Protocol' as numeric feature (6=TCP, 17=UDP, 1=ICMP)")

    print(f"Dropping columns: {cols_to_drop}")
    df_numeric = df_benign_renamed.drop(columns=cols_to_drop, errors='ignore')

    # Handle 'Infinity' string values → np.inf → np.nan
    for col in df_numeric.columns:
        if df_numeric[col].dtype == object:
            df_numeric[col] = df_numeric[col].replace(['Infinity', 'infinity', '-Infinity', '-infinity', 'inf', '-inf'], np.nan)
            df_numeric[col] = pd.to_numeric(df_numeric[col], errors='coerce')

    df_numeric = df_numeric.replace([np.inf, -np.inf], np.nan)

    # Force all columns to float
    df_numeric = df_numeric.astype(np.float32)

    nan_pct = (df_numeric.isna().sum() / len(df_numeric) * 100).sort_values(ascending=False)
    print(f"\nTotal features: {len(df_numeric.columns)}")
    print(f"\nNaN percentage (top 10):")
    print(nan_pct.head(10).to_string())

    feature_names_raw = list(df_numeric.columns)
    print(f"\nFeature list ({len(feature_names_raw)}): {feature_names_raw[:10]}...")

In [6]:
# ============================================================
# 2.4 Median Imputation (fit on benign CIC-2017 ONLY)
# ============================================================
if RERUN_CLEANING:
    median_values = df_numeric.median().to_dict()

    with open(f"{ARTIFACTS_DIR}/median_values.json", 'w') as f:
        json.dump({k: float(v) if pd.notna(v) else 0.0 for k, v in median_values.items()}, f, indent=2)
    print(f"Median values saved to {ARTIFACTS_DIR}/median_values.json")

    nan_before = df_numeric.isna().sum().sum()
    df_numeric = df_numeric.fillna(median_values)
    nan_after = df_numeric.isna().sum().sum()
    print(f"NaN imputed: {nan_before:,} → {nan_after:,}")

    assert nan_after == 0, "Still have NaN after imputation!"

    # Save cleaned data
    df_numeric.to_parquet(f"{PREPROCESSED_DIR}/cic2017_benign_cleaned.parquet", index=False)
    benign_meta.to_parquet(f"{PREPROCESSED_DIR}/cic2017_benign_meta.parquet", index=False)
    print(f"\nCleaned data saved: {df_numeric.shape}")
    print(f"Metadata saved: {benign_meta.shape}")

    # Keep full CIC-2017 (all labels) for evaluation in Section 12
    df_all_renamed.to_parquet(f"{PREPROCESSED_DIR}/cic2017_all_raw.parquet", index=False)
    print(f"Full CIC-2017 (all labels) saved for evaluation: {df_all_renamed.shape}")

    del df_all, df_benign, df_all_renamed, df_benign_renamed
    gc.collect()
else:
    required_files = [
        f"{PREPROCESSED_DIR}/cic2017_benign_cleaned.parquet",
        f"{PREPROCESSED_DIR}/cic2017_benign_meta.parquet",
        f"{ARTIFACTS_DIR}/median_values.json",
    ]
    missing = [p for p in required_files if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(
            "Cache incomplete for cleaning: missing " + ", ".join(missing)
        )

    df_numeric = pd.read_parquet(f"{PREPROCESSED_DIR}/cic2017_benign_cleaned.parquet")
    benign_meta = pd.read_parquet(f"{PREPROCESSED_DIR}/cic2017_benign_meta.parquet")
    with open(f"{ARTIFACTS_DIR}/median_values.json") as f:
        median_values = json.load(f)
    print(f"[CACHED] Loaded cleaned data: {df_numeric.shape}")

[CACHED] Loaded cleaned data: (2273097, 77)


## 3. FEATURE ENGINEERING & SELECTION

All selection criteria computed ONLY from benign CIC-2017 data.
1. Near-zero variance filter
2. High correlation filter

In [7]:
# ============================================================
# 3.1 Near-Zero Variance Filter
# ============================================================
if RERUN_FEATURE_SEL:
    variances = df_numeric.var()
    low_var_mask = variances < VARIANCE_THRESHOLD
    low_var_features = variances[low_var_mask].index.tolist()

    print(f"=== Near-Zero Variance Filter (threshold={VARIANCE_THRESHOLD}) ===")
    print(f"Features before: {len(df_numeric.columns)}")
    print(f"Low-variance features dropped ({len(low_var_features)}):")
    for feat in low_var_features:
        print(f"  - {feat} (var={variances[feat]:.2e})")

    df_selected = df_numeric.drop(columns=low_var_features)
    print(f"Features after NZV filter: {len(df_selected.columns)}")

In [8]:
# ============================================================
# 3.2 High Correlation Filter
# ============================================================
if RERUN_FEATURE_SEL:
    corr_matrix = df_selected.corr().abs()
    upper_tri = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    high_corr_pairs = []
    cols_to_drop_corr = set()

    for col in upper_tri.columns:
        correlated = upper_tri.index[upper_tri[col] > CORRELATION_THRESHOLD].tolist()
        for corr_col in correlated:
            pair_corr = corr_matrix.loc[corr_col, col]
            var_col = variances.get(col, 0)
            var_corr = variances.get(corr_col, 0)
            dropped = corr_col if var_corr <= var_col else col
            kept = col if dropped == corr_col else corr_col
            high_corr_pairs.append({
                'feature_1': col, 'feature_2': corr_col,
                'correlation': pair_corr,
                'var_1': var_col, 'var_2': var_corr,
                'dropped': dropped, 'kept': kept
            })
            cols_to_drop_corr.add(dropped)

    print(f"\n=== High Correlation Filter (threshold={CORRELATION_THRESHOLD}) ===")
    print(f"Correlated pairs found: {len(high_corr_pairs)}")
    print(f"Features to drop: {len(cols_to_drop_corr)}")
    for pair in high_corr_pairs[:10]:
        print(f"  |corr|={pair['correlation']:.4f}: {pair['dropped']} (dropped) ↔ {pair['kept']} (kept)")
    if len(high_corr_pairs) > 10:
        print(f"  ... and {len(high_corr_pairs)-10} more pairs")

    df_selected = df_selected.drop(columns=list(cols_to_drop_corr), errors='ignore')
    print(f"Features after correlation filter: {len(df_selected.columns)}")

In [9]:
# ============================================================
# 3.3 Save Feature Selection Results
# ============================================================
if RERUN_FEATURE_SEL:
    selected_features = list(df_selected.columns)

    # Adjust S1_LATENT_DIM based on actual feature count
    n_features = len(selected_features)
    S1_LATENT_DIM = max(4, min(10, n_features // 6))
    print(f"\n[SARAN] Adjusted S1_LATENT_DIM = {S1_LATENT_DIM} "
          f"(n_features={n_features}, compression ratio={n_features/S1_LATENT_DIM:.1f}x)")

    assert S1_LATENT_DIM <= 10, f"Bottleneck too large: {S1_LATENT_DIM} > 10"
    assert n_features / S1_LATENT_DIM >= 4, f"Compression ratio too low: {n_features/S1_LATENT_DIM:.1f}x < 4x"

    with open(f"{ARTIFACTS_DIR}/selected_features.json", 'w') as f:
        json.dump(selected_features, f, indent=2)

    report_rows = []
    for feat in df_numeric.columns:
        status = 'kept' if feat in selected_features else 'dropped_nzv' if feat in low_var_features else 'dropped_corr'
        report_rows.append({'feature': feat, 'variance': float(variances.get(feat, 0)), 'status': status})
    pd.DataFrame(report_rows).to_csv(f"{RESULTS_DIR}/feature_selection_report.csv", index=False)

    print(f"\n=== Feature Selection Summary ===")
    print(f"Original features: {len(df_numeric.columns)}")
    print(f"After NZV filter: {len(df_numeric.columns) - len(low_var_features)}")
    print(f"After corr filter: {len(selected_features)}")
    print(f"Selected features saved to: {ARTIFACTS_DIR}/selected_features.json")
    print(f"Report saved to: {RESULTS_DIR}/feature_selection_report.csv")

    # Apply selection to data
    df_selected = df_numeric[selected_features].copy()
else:
    required_files = [
        f"{ARTIFACTS_DIR}/selected_features.json",
    ]
    missing = [p for p in required_files if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(
            "Cache incomplete for feature selection: missing " + ", ".join(missing)
        )

    with open(f"{ARTIFACTS_DIR}/selected_features.json") as f:
        selected_features = json.load(f)
    n_features = len(selected_features)
    S1_LATENT_DIM = max(4, min(10, n_features // 6))
    df_selected = df_numeric[selected_features].copy()
    print(f"[CACHED] Using {n_features} selected features, S1_LATENT_DIM={S1_LATENT_DIM}")

[CACHED] Using 50 selected features, S1_LATENT_DIM=8


## 4. DOMAIN SHIFT ANALYSIS

Diagnostic only — NO transformations fitted on CSE-2018 data.
Uses KS-test to quantify distribution shift per feature between CIC-2017 and CSE-2018 benign flows.

In [10]:
# ============================================================
# 4.1 Load CSE-2018 Benign Sample
# ============================================================
if RERUN_DOMAIN_SHIFT:
    from scipy.stats import ks_2samp

    print("=== Loading CSE-CIC-IDS-2018 (benign sample) ===")

    cse_csv_files = sorted([
        os.path.join(CSE2018_DIR, f)
        for f in os.listdir(CSE2018_DIR) if f.endswith('.csv')
    ])

    cse_benign_chunks = []
    total_benign = 0
    max_benign = 100_000

    for fpath in cse_csv_files:
        if total_benign >= max_benign:
            break
        print(f"  Reading {os.path.basename(fpath)}...", end=" ")
        for chunk in pd.read_csv(fpath, chunksize=CHUNK_SIZE, low_memory=False, encoding='utf-8'):
            chunk.columns = chunk.columns.str.strip()

            label_col_cse = None
            for c in chunk.columns:
                if normalize_colname(c) == 'label':
                    label_col_cse = c
                    break
            if label_col_cse is None:
                continue

            labels_norm = chunk[label_col_cse].astype(str).map(normalize_colname)
            if DROP_HEADER_LEAK_ROWS:
                valid_label_row = (labels_norm != 'label') & (labels_norm != '') & (labels_norm != 'nan')
                chunk = chunk[valid_label_row]
                labels_norm = labels_norm[valid_label_row]

            benign_mask = labels_norm == 'benign'
            benign_chunk = chunk[benign_mask]

            if len(benign_chunk) > 0:
                remaining = max_benign - total_benign
                benign_chunk = benign_chunk.head(remaining)
                cse_benign_chunks.append(benign_chunk)
                total_benign += len(benign_chunk)

            if total_benign >= max_benign:
                break
        print(f"(total benign so far: {total_benign:,})")

    cse_benign_raw = pd.concat(cse_benign_chunks, ignore_index=True)
    cse_benign_raw.columns = cse_benign_raw.columns.str.strip()

    print(f"\nCSE-2018 benign sample: {len(cse_benign_raw):,} rows")

    map_info = summarize_feature_mapping(
        selected_features,
        cse_benign_raw.columns,
        FEATURE_ALIAS_MAP
    )
    cse_col_map = map_info['resolved_map']

    print(
        f"Feature mapping coverage (domain-shift sample): "
        f"{map_info['matched_feature_count']}/{len(selected_features)} "
        f"({map_info['match_ratio']:.2%})"
    )
    if map_info['missing_feature_count'] > 0:
        print(f"Missing features ({map_info['missing_feature_count']}): {map_info['missing_feature_list'][:10]}")

    if STRICT_FEATURE_MATCH and map_info['match_ratio'] < MIN_FEATURE_MATCH_RATIO:
        raise ValueError(
            f"Domain-shift mapping coverage too low: {map_info['match_ratio']:.2%} < {MIN_FEATURE_MATCH_RATIO:.0%}. "
            f"Missing features: {map_info['missing_feature_list'][:20]}"
        )

    # Apply feature selection and imputation
    cse_data = pd.DataFrame()
    for sf in selected_features:
        if sf in cse_col_map:
            col_data = cse_benign_raw[cse_col_map[sf]].copy()
            if col_data.dtype == object:
                col_data = col_data.replace(
                    ['Infinity', 'infinity', '-Infinity', '-infinity', 'inf', '-inf'], np.nan
                )
                col_data = pd.to_numeric(col_data, errors='coerce')
            col_data = col_data.replace([np.inf, -np.inf], np.nan)
            cse_data[sf] = col_data.astype(np.float32)
        else:
            cse_data[sf] = 0.0

    # Apply median imputation (from CIC-2017 fit)
    for sf in selected_features:
        if sf in median_values:
            cse_data[sf] = cse_data[sf].fillna(float(median_values[sf]))
        else:
            cse_data[sf] = cse_data[sf].fillna(0.0)

    print(f"CSE-2018 benign preprocessed: {cse_data.shape}")

    del cse_benign_chunks, cse_benign_raw
    gc.collect()

=== Loading CSE-CIC-IDS-2018 (benign sample) ===
  Reading 02-14-2018.csv... (total benign so far: 100,000)

CSE-2018 benign sample: 100,000 rows
Feature mapping coverage (domain-shift sample): 50/50 (100.00%)
CSE-2018 benign preprocessed: (100000, 50)


In [11]:
# ============================================================
# 4.2 & 4.3 KS-Test & Visualization
# ============================================================
if RERUN_DOMAIN_SHIFT:
    ks_results = []
    for feat in selected_features:
        cic_vals = df_selected[feat].dropna().values
        cse_vals = cse_data[feat].dropna().values
        if len(cic_vals) == 0 or len(cse_vals) == 0:
            ks_results.append({'feature': feat, 'ks_stat': 1.0, 'p_value': 0.0, 'shift': 'high'})
            continue
        stat, pval = ks_2samp(cic_vals[:50000], cse_vals[:50000])
        category = 'high' if stat >= 0.3 else ('medium' if stat >= 0.1 else 'low')
        ks_results.append({'feature': feat, 'ks_stat': round(stat, 4),
                           'p_value': round(pval, 6), 'shift': category})

    ks_df = pd.DataFrame(ks_results).sort_values('ks_stat', ascending=False)

    n_high = (ks_df['shift'] == 'high').sum()
    n_medium = (ks_df['shift'] == 'medium').sum()
    n_low = (ks_df['shift'] == 'low').sum()

    print(f"\n=== Domain Shift Summary ===")
    print(f"High shift (KS ≥ 0.3)  : {n_high} features")
    print(f"Medium shift (0.1-0.3) : {n_medium} features")
    print(f"Low shift (KS < 0.1)   : {n_low} features")
    print(f"\nTop 10 shifted features:")
    print(ks_df.head(10).to_string(index=False))

    # [SARAN] Recommendation on high-shift features
    if n_high > 0:
        print(f"\n[SARAN] {n_high} features have high domain shift (KS ≥ 0.3).")
        print("  These features may hurt generalization to CSE-2018.")
        print("  However, dropping them risks losing discriminative power on CIC-2017.")
        print("  Strategy: KEEP all features but monitor per-feature contribution")
        print("  in evaluation. RobustScaler + clipping helps mitigate scale differences.")
        if n_high > n_features * 0.5:
            print("  [PERINGATAN] >50% features have high shift — generalization will be challenging.")

    # Bar chart visualization
    fig, ax = plt.subplots(figsize=(14, max(5, len(selected_features) * 0.25)))
    colors = {'high': '#e74c3c', 'medium': '#f39c12', 'low': '#27ae60'}
    bars = ax.barh(range(len(ks_df)), ks_df['ks_stat'].values,
                   color=[colors[s] for s in ks_df['shift'].values])
    ax.set_yticks(range(len(ks_df)))
    ax.set_yticklabels(ks_df['feature'].values, fontsize=7)
    ax.set_xlabel('KS Statistic')
    ax.set_title('Domain Shift: CIC-2017 → CSE-2018 (Benign Distributions)')
    ax.axvline(x=0.3, color='red', linestyle='--', alpha=0.7, label='High shift threshold')
    ax.axvline(x=0.1, color='orange', linestyle='--', alpha=0.7, label='Medium shift threshold')
    ax.legend()
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/domain_shift_analysis.png", dpi=150, bbox_inches='tight')
    plt.show()

    ks_df.to_csv(f"{RESULTS_DIR}/domain_shift_analysis.csv", index=False)
    print(f"\nSaved: domain_shift_analysis.csv & .png")

    del cse_data
    gc.collect()


=== Domain Shift Summary ===
High shift (KS ≥ 0.3)  : 10 features
Medium shift (0.1-0.3) : 34 features
Low shift (KS < 0.1)   : 6 features

Top 10 shifted features:
               feature  ks_stat  p_value shift
          Flow IAT Min   0.4516      0.0  high
     Bwd Header Length   0.4481      0.0  high
           Bwd IAT Min   0.3717      0.0  high
           Bwd IAT Max   0.3535      0.0  high
          Bwd IAT Mean   0.3535      0.0  high
  min_seg_size_forward   0.3239      0.0  high
        URG Flag Count   0.3233      0.0  high
   Fwd Header Length.1   0.3197      0.0  high
 Fwd Packet Length Min   0.3194      0.0  high
Init_Win_bytes_forward   0.3055      0.0  high

[SARAN] 10 features have high domain shift (KS ≥ 0.3).
  These features may hurt generalization to CSE-2018.
  However, dropping them risks losing discriminative power on CIC-2017.
  Strategy: KEEP all features but monitor per-feature contribution
  in evaluation. RobustScaler + clipping helps mitigate scale differ

## 5. SCALING & TRAIN/VAL SPLIT

- Split by `session_id` (not random rows) to prevent session leakage
- RobustScaler fitted ONLY on training split
- Output clipped to [-5, 5]

In [12]:
# ============================================================
# 5.1 Train/Val Split by Session ID (strict session quality)
# ============================================================
if RERUN_SCALING:
    from sklearn.preprocessing import RobustScaler

    if len(benign_meta) != len(df_selected):
        raise ValueError(
            f"Length mismatch: benign_meta={len(benign_meta):,} vs df_selected={len(df_selected):,}. "
            "Re-run cleaning/feature-selection so metadata and features stay aligned."
        )

    session_ids = benign_meta['session_id'].fillna('unknown').astype(str).values
    timestamps = pd.to_numeric(pd.Series(benign_meta['ts_parsed']), errors='coerce')
    unique_sessions = np.unique(session_ids)

    timestamp_valid_ratio = float(timestamps.notna().mean())
    sid_counts = pd.Series(session_ids).value_counts()
    largest_session_share = float(sid_counts.max() / len(session_ids)) if len(session_ids) > 0 else 0.0
    singleton_session_pct = float((sid_counts == 1).mean() * 100) if len(sid_counts) > 0 else 100.0

    if STRICT_SESSION_QUALITY:
        if timestamp_valid_ratio < MIN_TIMESTAMP_VALID_RATIO:
            raise ValueError(
                f"Timestamp valid ratio too low: {timestamp_valid_ratio:.2%} < {MIN_TIMESTAMP_VALID_RATIO:.0%}. "
                "Fix timestamp parsing in Section 2.2 before split."
            )
        if len(unique_sessions) < MIN_UNIQUE_SESSIONS:
            raise ValueError(
                f"Unique sessions too low: {len(unique_sessions):,} < {MIN_UNIQUE_SESSIONS:,}. "
                "Sessionization quality gate failed."
            )
        if largest_session_share > MAX_SINGLE_SESSION_SHARE:
            raise ValueError(
                f"Largest session share too high: {largest_session_share:.2%} > {MAX_SINGLE_SESSION_SHARE:.2%}. "
                "Session distribution is too concentrated."
            )
        if singleton_session_pct > 70.0:
            raise ValueError(
                f"Session singleton ratio too high: {singleton_session_pct:.1f}% > 70.0%. "
                "Session IDs are too granular for windowing; fix sessionization before training."
            )

    split_mode = None
    if len(unique_sessions) >= 2:
        temp = pd.DataFrame({'session_id': session_ids, 'timestamp': timestamps.fillna(0).values})
        session_order = (
            temp.groupby('session_id', as_index=False)['timestamp']
            .min()
            .sort_values('timestamp')
            ['session_id']
            .tolist()
        )

        split_idx = int(len(session_order) * 0.8)
        split_idx = min(max(split_idx, 1), len(session_order) - 1)

        train_sessions = set(session_order[:split_idx])
        val_sessions = set(session_order[split_idx:])

        train_mask = np.array([sid in train_sessions for sid in session_ids], dtype=bool)
        val_mask = np.array([sid in val_sessions for sid in session_ids], dtype=bool)
        split_mode = 'session_time'
    else:
        if not ALLOW_ROW_SPLIT_FALLBACK:
            raise ValueError(
                "Only one unique session found and ALLOW_ROW_SPLIT_FALLBACK=False. "
                "Stop to avoid silent degradation."
            )

        n_rows = len(df_selected)
        if n_rows < 2:
            raise ValueError(f"Not enough rows to split data: n_rows={n_rows}.")

        idx = np.arange(n_rows)
        np.random.shuffle(idx)
        split_idx = int(n_rows * 0.8)
        split_idx = min(max(split_idx, 1), n_rows - 1)

        train_idx = idx[:split_idx]
        train_mask = np.zeros(n_rows, dtype=bool)
        train_mask[train_idx] = True
        val_mask = ~train_mask
        split_mode = 'row_fallback'
        print("[PERINGATAN] Falling back to row-wise split by explicit config.")

    X_train_raw = df_selected.values[train_mask]
    X_val_raw = df_selected.values[val_mask]

    train_session_ids = session_ids[train_mask]
    val_session_ids = session_ids[val_mask]

    train_timestamps = timestamps.values[train_mask]
    val_timestamps = timestamps.values[val_mask]

    if X_train_raw.shape[0] == 0 or X_val_raw.shape[0] == 0:
        raise ValueError(
            f"Empty split produced (mode={split_mode}): train={X_train_raw.shape[0]}, val={X_val_raw.shape[0]}."
        )

    split_report = {
        'split_mode': split_mode,
        'rows_train': int(X_train_raw.shape[0]),
        'rows_val': int(X_val_raw.shape[0]),
        'unique_sessions': int(len(unique_sessions)),
        'timestamp_valid_ratio': float(timestamp_valid_ratio),
        'largest_session_share': float(largest_session_share),
        'train_unique_sessions': int(pd.Series(train_session_ids).nunique()),
        'val_unique_sessions': int(pd.Series(val_session_ids).nunique()),
    }
    with open(f"{ARTIFACTS_DIR}/session_split_report.json", 'w', encoding='utf-8') as f:
        json.dump(split_report, f, indent=2)

    print(f"=== Train/Val Split ({split_mode}) ===")
    print(f"Unique sessions: {len(unique_sessions):,}")
    print(f"Train flows    : {X_train_raw.shape[0]:,}")
    print(f"Val flows      : {X_val_raw.shape[0]:,}")
    print(f"Timestamp valid ratio : {timestamp_valid_ratio:.2%}")
    print(f"Largest session share : {largest_session_share:.2%}")


=== Train/Val Split (session_time) ===
Unique sessions: 50,689
Train flows    : 1,818,854
Val flows      : 454,243
Timestamp valid ratio : 100.00%
Largest session share : 0.00%


In [13]:
# ============================================================
# 5.2 RobustScaler + Clipping
# ============================================================
if RERUN_SCALING:
    if X_train_raw.shape[0] == 0:
        raise ValueError(
            "X_train_raw is empty before scaling. Check split logic in Section 5.1."
        )
    if X_val_raw.shape[0] == 0:
        raise ValueError(
            "X_val_raw is empty before scaling. Check split logic in Section 5.1."
        )

    scaler = RobustScaler()
    X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
    X_val = scaler.transform(X_val_raw).astype(np.float32)

    # Clip to [-5, 5] to limit extreme outliers
    X_train = np.clip(X_train, -5, 5)
    X_val = np.clip(X_val, -5, 5)

    with open(f"{ARTIFACTS_DIR}/robust_scaler.pkl", 'wb') as f:
        pickle.dump(scaler, f)
    print(f"RobustScaler saved to {ARTIFACTS_DIR}/robust_scaler.pkl")

    # Save scaled data
    np.save(f"{PREPROCESSED_DIR}/X_train.npy", X_train)
    np.save(f"{PREPROCESSED_DIR}/X_val.npy", X_val)
    np.save(f"{PREPROCESSED_DIR}/train_session_ids.npy", train_session_ids)
    np.save(f"{PREPROCESSED_DIR}/val_session_ids.npy", val_session_ids)
    np.save(f"{PREPROCESSED_DIR}/train_timestamps.npy", train_timestamps)
    np.save(f"{PREPROCESSED_DIR}/val_timestamps.npy", val_timestamps)

    print(f"\nX_train: {X_train.shape}, range [{X_train.min():.2f}, {X_train.max():.2f}]")
    print(f"X_val  : {X_val.shape}, range [{X_val.min():.2f}, {X_val.max():.2f}]")
    n_features = X_train.shape[1]
    print(f"n_features = {n_features}")
else:
    required_files = [
        f"{PREPROCESSED_DIR}/X_train.npy",
        f"{PREPROCESSED_DIR}/X_val.npy",
        f"{PREPROCESSED_DIR}/train_session_ids.npy",
        f"{PREPROCESSED_DIR}/val_session_ids.npy",
        f"{PREPROCESSED_DIR}/train_timestamps.npy",
        f"{PREPROCESSED_DIR}/val_timestamps.npy",
        f"{ARTIFACTS_DIR}/robust_scaler.pkl",
    ]
    missing = [p for p in required_files if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(
            "Cache incomplete for scaling: missing " + ", ".join(missing)
        )

    X_train = np.load(f"{PREPROCESSED_DIR}/X_train.npy")
    X_val = np.load(f"{PREPROCESSED_DIR}/X_val.npy")
    train_session_ids = np.load(f"{PREPROCESSED_DIR}/train_session_ids.npy", allow_pickle=True)
    val_session_ids = np.load(f"{PREPROCESSED_DIR}/val_session_ids.npy", allow_pickle=True)
    train_timestamps = np.load(f"{PREPROCESSED_DIR}/train_timestamps.npy", allow_pickle=True)
    val_timestamps = np.load(f"{PREPROCESSED_DIR}/val_timestamps.npy", allow_pickle=True)
    with open(f"{ARTIFACTS_DIR}/robust_scaler.pkl", 'rb') as f:
        scaler = pickle.load(f)
    n_features = X_train.shape[1]
    S1_LATENT_DIM = max(4, min(10, n_features // 6))
    print(f"[CACHED] X_train={X_train.shape}, X_val={X_val.shape}, n_features={n_features}")

RobustScaler saved to /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/artifacts/v3_sessionfix/robust_scaler.pkl

X_train: (1818854, 50), range [-5.00, 5.00]
X_val  : (454243, 50), range [-5.00, 5.00]
n_features = 50


## 6. STAGE 1 — MSCNN-AE: TRAINING

Multi-Scale CNN Autoencoder for spatial feature extraction.
- Input: per-flow feature vector `(n_features, 1)` — no windowing
- Multi-scale parallel Conv1D branches (kernel 3, 5, 7)
- Bottleneck: `S1_LATENT_DIM` dimensions with LINEAR activation
- Output: LINEAR activation (input is not bounded [0,1])

In [14]:
# ============================================================
# 6.1 Build MSCNN-AE Architecture
# ============================================================
from tensorflow.keras.layers import (Input, Conv1D, Dense, Flatten, Reshape,
                                      Dropout, BatchNormalization, Concatenate,
                                      Bidirectional, LSTM, RepeatVector,
                                      TimeDistributed, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

def build_mscnn_ae(n_features, latent_dim, dropout_rate):
    """
    Stage 1: Multi-Scale CNN Autoencoder.
    Uses GlobalAveragePooling1D instead of Flatten to reduce parameter count
    and avoid the LARANGAN of Flatten before bottleneck on high-dim intermediate.
    """
    inputs = Input(shape=(n_features, 1), name='input')

    # === ENCODER — Multi-Scale Parallel Branches ===
    b1 = Conv1D(16, kernel_size=3, padding='same', activation='relu')(inputs)
    b1 = BatchNormalization()(b1)

    b2 = Conv1D(16, kernel_size=5, padding='same', activation='relu')(inputs)
    b2 = BatchNormalization()(b2)

    b3 = Conv1D(16, kernel_size=7, padding='same', activation='relu')(inputs)
    b3 = BatchNormalization()(b3)

    x = Concatenate()([b1, b2, b3])  # (batch, n_features, 48)

    x = Conv1D(32, kernel_size=3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)

    # GlobalAveragePooling1D compresses spatial dimension without Flatten explosion
    # [SARAN] Using GAP instead of Flatten avoids overparameterization
    # (Flatten would create n_features*32 parameters → too many for small bottleneck)
    x = GlobalAveragePooling1D()(x)  # (batch, 32)

    # Bottleneck — LINEAR activation preserves full value range
    bottleneck = Dense(latent_dim, activation='linear', name='bottleneck')(x)

    # === DECODER ===
    x = Dense(32, activation='relu')(bottleneck)
    x = BatchNormalization()(x)
    x = Dense(n_features * 16, activation='relu')(x)
    x = Reshape((n_features, 16))(x)
    x = Conv1D(16, kernel_size=3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    outputs = Conv1D(1, kernel_size=1, activation='linear', name='output')(x)
    outputs = Reshape((n_features,), name='output_reshape')(outputs)

    model = Model(inputs, outputs, name='MSCNN_AE_Stage1')
    encoder = Model(inputs, bottleneck, name='MSCNN_Encoder')
    return model, encoder

model_s1, encoder_s1 = build_mscnn_ae(n_features, S1_LATENT_DIM, S1_DROPOUT)

print("=== MSCNN-AE Architecture ===")
model_s1.summary()
total_params = model_s1.count_params()
compression = n_features / S1_LATENT_DIM
print(f"\nTotal parameters   : {total_params:,}")
print(f"Compression ratio  : {n_features} / {S1_LATENT_DIM} = {compression:.1f}x")
print(f"Bottleneck dim     : {S1_LATENT_DIM}")

assert S1_LATENT_DIM <= 10, f"[LARANGAN] Bottleneck too large: {S1_LATENT_DIM} > 10"
assert compression >= 4, f"[LARANGAN] Compression ratio too low: {compression:.1f}x < 4x"
print("\n✓ Architecture constraints satisfied")

=== MSCNN-AE Architecture ===


Model: "MSCNN_AE_Stage1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 50, 1)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 50, 16)    │         64 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 50, 16)    │         96 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 50, 16)    │        128 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 50, 16)    │         64 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 50, 16)    │         64 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 50, 16)    │         64 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 50, 48)    │          0 │ batch_normalizat… │
│ (Concatenate)       │                   │            │ batch_normalizat… │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 50, 32)    │      4,640 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 50, 32)    │        128 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 50, 32)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ dropout[0][0]     │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bottleneck (Dense)  │ (None, 8)         │        264 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │        288 │ bottleneck[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32)        │        128 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 800)       │     26,400 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 50, 16)    │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 50, 16)    │        784 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 50, 16)    │         64 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Conv1D)     │ (None, 50, 1)     │         17 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 33,193 (129.66 KB)

 Trainable params: 32,937 (128.66 KB)

 Non-trainable params: 256 (1.00 KB)


Total parameters   : 33,193
Compression ratio  : 50 / 8 = 6.2x
Bottleneck dim     : 8

✓ Architecture constraints satisfied


In [15]:
# ============================================================
# 6.2 Train Stage 1
# ============================================================
if RERUN_STAGE1_TRAIN:
    model_s1.compile(optimizer=Adam(learning_rate=S1_LR), loss='mse')

    callbacks_s1 = [
        EarlyStopping(monitor='val_loss', patience=S1_ES_PATIENCE,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=S1_LR_PATIENCE, min_lr=1e-6, verbose=1),
        ModelCheckpoint(STAGE1_CHECKPOINT_PATH,
                        monitor='val_loss', save_best_only=True, verbose=1)
    ]

    X_train_s1 = X_train[..., np.newaxis]  # (n, n_features) → (n, n_features, 1)
    X_val_s1 = X_val[..., np.newaxis]

    print(f"Training Stage 1 MSCNN-AE...")
    print(f"  Train: {X_train_s1.shape}, Val: {X_val_s1.shape}")

    if ENABLE_TF_DATA_PIPELINE:
        train_ds_s1 = tf.data.Dataset.from_tensor_slices((X_train_s1, X_train))
        val_ds_s1 = tf.data.Dataset.from_tensor_slices((X_val_s1, X_val))

        shuffle_buffer = min(len(X_train_s1), 200000)
        if shuffle_buffer > 1:
            train_ds_s1 = train_ds_s1.shuffle(
                shuffle_buffer,
                seed=RANDOM_SEED,
                reshuffle_each_iteration=True
            )

        train_ds_s1 = train_ds_s1.batch(S1_BATCH_SIZE, drop_remainder=False)
        val_ds_s1 = val_ds_s1.batch(S1_BATCH_SIZE, drop_remainder=False)

        if PREFETCH_AUTOTUNE:
            train_ds_s1 = train_ds_s1.prefetch(tf.data.AUTOTUNE)
            val_ds_s1 = val_ds_s1.prefetch(tf.data.AUTOTUNE)

        history_s1 = model_s1.fit(
            train_ds_s1,
            validation_data=val_ds_s1,
            epochs=S1_MAX_EPOCHS,
            callbacks=callbacks_s1,
            verbose=1
        )
    else:
        history_s1 = model_s1.fit(
            X_train_s1, X_train,
            validation_data=(X_val_s1, X_val),
            batch_size=S1_BATCH_SIZE,
            epochs=S1_MAX_EPOCHS,
            callbacks=callbacks_s1,
            verbose=1
        )

    # Diagnostics
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history_s1.history['loss'], label='Train Loss')
    ax.plot(history_s1.history['val_loss'], label='Val Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('Stage 1 MSCNN-AE Training Curve')
    ax.legend()
    ax.set_yscale('log')
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/stage1_training_curve.png", dpi=150, bbox_inches='tight')
    plt.show()

    final_train_loss = history_s1.history['loss'][-1]
    final_val_loss = history_s1.history['val_loss'][-1]
    best_val_loss = min(history_s1.history['val_loss'])
    best_epoch = history_s1.history['val_loss'].index(best_val_loss) + 1

    print(f"\n=== Stage 1 Training Summary ===")
    print(f"Best epoch     : {best_epoch}")
    print(f"Best val_loss  : {best_val_loss:.6f}")
    print(f"Final train    : {final_train_loss:.6f}")
    print(f"Final val      : {final_val_loss:.6f}")

    if final_val_loss > final_train_loss * 2:
        print("[PERINGATAN] val_loss > 2x train_loss — possible overfitting!")

    # Check convergence in first 10 epochs
    early_losses = history_s1.history['loss'][:10]
    if len(early_losses) >= 10 and early_losses[-1] > early_losses[0] * 0.95:
        print("[PERINGATAN] Loss barely decreased in first 10 epochs — check learning rate or architecture")

else:
    if not os.path.exists(STAGE1_CHECKPOINT_PATH):
        raise FileNotFoundError(f"Checkpoint missing for Stage 1: {STAGE1_CHECKPOINT_PATH}")
    model_s1.load_weights(STAGE1_CHECKPOINT_PATH)
    print(f"[CACHED] Stage 1 weights loaded from checkpoint: {STAGE1_CHECKPOINT_PATH}")

[CACHED] Stage 1 weights loaded from checkpoint: /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/checkpoints/stage1_best_v3_sessionfix.keras


## 7. STAGE 1 — LATENT VECTOR EXTRACTION

Extract latent representations from Stage 1 encoder for all splits.
These become the input for session-based windowing and Stage 2.

In [16]:
# ============================================================
# 7. Extract Latent Vectors + Health Check
# ============================================================
if RERUN_LATENT_EXTRACT:
    def extract_latent(encoder, X, batch_size=S1_BATCH_SIZE):
        X_reshaped = X[..., np.newaxis]
        latent = encoder.predict(X_reshaped, batch_size=batch_size, verbose=0)
        return np.asarray(latent, dtype=np.float32)

    latent_train = extract_latent(encoder_s1, X_train)
    latent_val = extract_latent(encoder_s1, X_val)
else:
    required_files = [
        f"{PREPROCESSED_DIR}/latent_train.npy",
        f"{PREPROCESSED_DIR}/latent_val.npy",
    ]
    missing = [p for p in required_files if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(
            "Cache incomplete for latent extraction: missing " + ", ".join(missing)
        )

    latent_train = np.load(f"{PREPROCESSED_DIR}/latent_train.npy").astype(np.float32, copy=False)
    latent_val = np.load(f"{PREPROCESSED_DIR}/latent_val.npy").astype(np.float32, copy=False)
    print(f"[CACHED] Latent: train={latent_train.shape}, val={latent_val.shape}")

# Always normalize dtype before health check to avoid fp16 overflow in reductions.
latent_train = np.asarray(latent_train, dtype=np.float32)
latent_val = np.asarray(latent_val, dtype=np.float32)

train_finite = np.isfinite(latent_train)
val_finite = np.isfinite(latent_val)
train_non_finite = int((~train_finite).sum())
val_non_finite = int((~val_finite).sum())

if train_non_finite == 0 and val_non_finite == 0:
    latent_std = np.std(latent_train.astype(np.float64), axis=0)
    latent_std_mean = float(np.mean(latent_std))
    latent_std_max = float(np.max(latent_std))
else:
    latent_std = np.full(latent_train.shape[1], np.nan, dtype=np.float64)
    latent_std_mean = float('nan')
    latent_std_max = float('nan')

latent_health_report = {
    'pipeline_version': PIPELINE_VERSION,
    'latent_train_shape': list(latent_train.shape),
    'latent_val_shape': list(latent_val.shape),
    'train_non_finite_count': train_non_finite,
    'val_non_finite_count': val_non_finite,
    'latent_min': float(np.nanmin(latent_train)),
    'latent_max': float(np.nanmax(latent_train)),
    'latent_std_mean': latent_std_mean,
    'latent_std_max': latent_std_max,
    'latent_std_threshold': float(LATENT_STD_MAX),
}

with open(f"{ARTIFACTS_DIR}/latent_health_report.json", 'w', encoding='utf-8') as f:
    json.dump(latent_health_report, f, indent=2)

if ENABLE_LATENT_HEALTH_CHECK:
    if train_non_finite > 0 or val_non_finite > 0:
        raise ValueError(
            "Latent health check failed: found NaN/Inf in latent vectors. "
            f"train_non_finite={train_non_finite}, val_non_finite={val_non_finite}. "
            "Check scaling/clipping and mixed precision stability."
        )
    if not np.isfinite(latent_std_max) or latent_std_max > LATENT_STD_MAX:
        raise ValueError(
            "Latent health check failed: latent std too large/invalid. "
            f"latent_std_max={latent_std_max} > LATENT_STD_MAX={LATENT_STD_MAX:.4f}. "
            "Check scaling/clipping and mixed precision overflow."
        )

if RERUN_LATENT_EXTRACT:
    np.save(f"{PREPROCESSED_DIR}/latent_train.npy", latent_train)
    np.save(f"{PREPROCESSED_DIR}/latent_val.npy", latent_val)

print(f"=== Latent Vectors Ready ===")
print(f"Latent train : {latent_train.shape}")
print(f"Latent val   : {latent_val.shape}")
print(f"Latent range : [{latent_health_report['latent_min']:.3f}, {latent_health_report['latent_max']:.3f}]")
print(f"Latent std   : mean={latent_std_mean:.4f}, max={latent_std_max:.4f}")

collapsed = int((latent_std < 1e-4).sum()) if np.isfinite(latent_std).all() else -1
if collapsed > 0:
    print(f"[PERINGATAN] {collapsed}/{S1_LATENT_DIM} latent dimensions nearly collapsed (std < 1e-4)")
elif collapsed == 0:
    print(f"[OK] All {S1_LATENT_DIM} latent dimensions active")



=== Latent Vectors Ready ===
Latent train : (1818854, 8)
Latent val   : (454243, 8)
Latent range : [-5.996, 15.344]
Latent std   : mean=1.5347, max=2.4843
[OK] All 8 latent dimensions active


## 8. SESSION-BASED WINDOWING (in Latent Space)

**Fundamental change from previous implementation:**
Windowing happens in **latent space**, not raw feature space.
Windows are grouped by session and sorted by timestamp.

Catatan run-order: meskipun `RERUN_WINDOWING=False` (pakai cache), jalankan cell 8.2 sekali per sesi kernel agar definisi fungsi `create_session_windows` tersedia untuk evaluasi (Section 12/13).

In [17]:
# ============================================================
# 8.1 Session Length Analysis
# ============================================================
if RERUN_WINDOWING:
    session_lengths_train = pd.Series(train_session_ids).value_counts()

    print("=== Session Length Distribution (Train) ===")
    print(session_lengths_train.describe())
    print(f"\nMedian session length      : {session_lengths_train.median():.1f}")
    print(f"% Sessions with 1 flow     : {(session_lengths_train == 1).mean()*100:.1f}%")
    print(f"% Sessions with < 3 flows  : {(session_lengths_train < 3).mean()*100:.1f}%")
    print(f"% Sessions with >= 10 flows: {(session_lengths_train >= 10).mean()*100:.1f}%")

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(session_lengths_train.clip(upper=50).values, bins=50, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Flows per Session')
    ax.set_ylabel('Count')
    ax.set_title('Session Length Distribution (CIC-2017 Benign, Train Split)')
    ax.axvline(x=WINDOW_SIZE, color='red', linestyle='--', label=f'Window size = {WINDOW_SIZE}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/session_length_distribution.png", dpi=150, bbox_inches='tight')
    plt.show()

    pct_single = (session_lengths_train == 1).mean() * 100
    median_len = float(session_lengths_train.median())

    if STRICT_SESSION_QUALITY and (median_len < MIN_FLOWS_PER_SESSION or pct_single > 70):
        raise ValueError(
            "Session quality gate failed before windowing: "
            f"median_len={median_len:.1f}, single_flow_pct={pct_single:.1f}%. "
            "Fix sessionization instead of falling back silently."
        )

    print(f"\n? Session-based windowing enforced (median={median_len:.0f}, single-flow={pct_single:.1f}%)")

=== Session Length Distribution (Train) ===
count    40551.000000
mean        44.853493
std         13.862393
min          1.000000
25%         49.000000
50%         50.000000
75%         50.000000
max         50.000000
Name: count, dtype: float64

Median session length      : 50.0
% Sessions with 1 flow     : 4.3%
% Sessions with < 3 flows  : 6.4%
% Sessions with >= 10 flows: 90.9%

? Session-based windowing enforced (median=50, single-flow=4.3%)


In [18]:
# ============================================================
# 8.2 Create Windowed Latent Sequences
# ============================================================
def create_session_windows(latent_vectors, session_ids, timestamps,
                            window_size, min_flows):
    """
    Create non-overlapping windows from latent vectors,
    grouped by session and sorted by timestamp.
    """
    if window_size < 1:
        raise ValueError(f"window_size must be >= 1, got {window_size}")

    if len(latent_vectors) != len(session_ids) or len(latent_vectors) != len(timestamps):
        raise ValueError(
            f"Length mismatch: latent={len(latent_vectors)}, sids={len(session_ids)}, ts={len(timestamps)}"
        )

    ts_series = pd.to_numeric(pd.Series(timestamps), errors='coerce')
    ts_valid_ratio = float(ts_series.notna().mean())

    if STRICT_SESSION_QUALITY and ts_valid_ratio < MIN_TIMESTAMP_VALID_RATIO:
        raise ValueError(
            f"Timestamp valid ratio too low for windowing: {ts_valid_ratio:.2%} < {MIN_TIMESTAMP_VALID_RATIO:.0%}."
        )

    ts_filled = ts_series.ffill().bfill()
    if ts_filled.isna().any():
        ts_filled = pd.Series(np.arange(len(latent_vectors), dtype=np.int64))

    windows, masks, flow_idx_list = [], [], []
    dropped_sessions = 0
    sessions_used = 0

    df_temp = pd.DataFrame({
        'session_id': pd.Series(session_ids).astype(str).values,
        'timestamp': ts_filled.values,
        'latent_idx': np.arange(len(latent_vectors), dtype=np.int64),
    })

    for _, group in df_temp.groupby('session_id', sort=False):
        group = group.sort_values('timestamp')
        indices = group['latent_idx'].values

        if len(indices) < min_flows:
            dropped_sessions += 1
            continue

        sessions_used += 1
        for start in range(0, len(indices), window_size):
            chunk = indices[start:start + window_size]
            w = latent_vectors[chunk]
            mask = np.zeros(window_size, dtype=bool)

            if len(chunk) < window_size:
                pad_len = window_size - len(chunk)
                w = np.vstack([w, np.zeros((pad_len, latent_vectors.shape[1]), dtype=latent_vectors.dtype)])
                mask[len(chunk):] = True

            windows.append(w)
            masks.append(mask)

            padded_chunk = np.full(window_size, -1, dtype=np.int64)
            padded_chunk[:len(chunk)] = chunk
            flow_idx_list.append(padded_chunk)

    if len(windows) == 0:
        raise ValueError(
            "No windows generated. Check sessionization quality/min_flows/window_size settings."
        )

    windows_np = np.array(windows)
    masks_np = np.array(masks)
    flow_idx_np = np.array(flow_idx_list)

    create_session_windows.last_stats = {
        'sessions_used': int(sessions_used),
        'sessions_dropped_lt_min_flows': int(dropped_sessions),
        'window_count': int(len(windows_np)),
        'padded_ratio': float(masks_np.any(axis=1).mean()) if len(masks_np) > 0 else 0.0,
        'timestamp_valid_ratio': float(ts_valid_ratio),
    }

    return windows_np, masks_np, flow_idx_np


if RERUN_WINDOWING:
    windows_train, masks_train, flow_idx_train = create_session_windows(
        latent_train, train_session_ids, train_timestamps, WINDOW_SIZE, MIN_FLOWS_PER_SESSION
    )
    stats_train = getattr(create_session_windows, 'last_stats', {})

    windows_val, masks_val, flow_idx_val = create_session_windows(
        latent_val, val_session_ids, val_timestamps, WINDOW_SIZE, MIN_FLOWS_PER_SESSION
    )
    stats_val = getattr(create_session_windows, 'last_stats', {})

    print(f"\n=== Windowed Latent Sequences ===")
    print(f"Window size       : {WINDOW_SIZE}")
    print(f"Windows train     : {windows_train.shape}")
    print(f"Windows val       : {windows_val.shape}")
    print(f"% Padded windows  : {masks_train.any(axis=1).mean()*100:.2f}% (train)")

    windowing_quality_report = {
        'pipeline_version': PIPELINE_VERSION,
        'n_windows_train': int(windows_train.shape[0]),
        'n_windows_val': int(windows_val.shape[0]),
        'padded_ratio_train': float(masks_train.any(axis=1).mean()),
        'padded_ratio_val': float(masks_val.any(axis=1).mean()),
        'sessions_used_train': int(stats_train.get('sessions_used', 0)),
        'sessions_used_val': int(stats_val.get('sessions_used', 0)),
        'sessions_dropped_lt_min_flows_train': int(stats_train.get('sessions_dropped_lt_min_flows', 0)),
        'sessions_dropped_lt_min_flows_val': int(stats_val.get('sessions_dropped_lt_min_flows', 0)),
        'timestamp_valid_ratio_train': float(stats_train.get('timestamp_valid_ratio', 0.0)),
        'timestamp_valid_ratio_val': float(stats_val.get('timestamp_valid_ratio', 0.0)),
    }
    with open(f"{ARTIFACTS_DIR}/windowing_quality_report.json", 'w', encoding='utf-8') as f:
        json.dump(windowing_quality_report, f, indent=2)

    np.save(f"{PREPROCESSED_DIR}/flow_idx_train.npy", flow_idx_train)
    np.save(f"{PREPROCESSED_DIR}/flow_idx_val.npy", flow_idx_val)
else:
    print("[SKIP] Window generation - using cached windows from Section 9")


=== Windowed Latent Sequences ===
Window size       : 10
Windows train     : (184156, 10, 8)
Windows val       : (46000, 10, 8)
% Padded windows  : 4.44% (train)


## 9. SAVE/LOAD CHECKPOINT

In [19]:
# ============================================================
# 9. Checkpoint Management
# ============================================================
def save_preprocessed():
    np.savez_compressed(f"{PREPROCESSED_DIR}/windows_train.npz",
                        X=windows_train, mask=masks_train)
    np.savez_compressed(f"{PREPROCESSED_DIR}/windows_val.npz",
                        X=windows_val, mask=masks_val)
    # Save windowing config
    use_windowing_flag = bool(globals().get('USE_WINDOWING', True))
    wconfig = {'window_size': WINDOW_SIZE, 'min_flows': MIN_FLOWS_PER_SESSION,
               'use_windowing': use_windowing_flag, 'n_windows_train': len(windows_train),
               'n_windows_val': len(windows_val)}
    with open(f"{ARTIFACTS_DIR}/windowing_config.json", 'w') as f:
        json.dump(wconfig, f, indent=2)
    print("Preprocessed windows saved to Drive.")

def load_preprocessed():
    global windows_train, masks_train, windows_val, masks_val, flow_idx_train, flow_idx_val
    d = np.load(f"{PREPROCESSED_DIR}/windows_train.npz")
    windows_train, masks_train = d['X'], d['mask']
    d = np.load(f"{PREPROCESSED_DIR}/windows_val.npz")
    windows_val, masks_val = d['X'], d['mask']
    flow_idx_train = np.load(f"{PREPROCESSED_DIR}/flow_idx_train.npy")
    flow_idx_val = np.load(f"{PREPROCESSED_DIR}/flow_idx_val.npy")
    print(f"Loaded: train={windows_train.shape}, val={windows_val.shape}")

if RERUN_WINDOWING:
    save_preprocessed()
else:
    required_files = [
        f"{PREPROCESSED_DIR}/windows_train.npz",
        f"{PREPROCESSED_DIR}/windows_val.npz",
        f"{PREPROCESSED_DIR}/flow_idx_train.npy",
        f"{PREPROCESSED_DIR}/flow_idx_val.npy",
    ]
    missing = [p for p in required_files if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(
            "Cache incomplete for windowing: missing " + ", ".join(missing)
        )
    load_preprocessed()

# Guard ringan: flow index harus ada sebelum threshold/evaluation
if 'flow_idx_val' not in globals():
    raise RuntimeError(
        "flow_idx_val is not loaded. Run Section 8.2 then Section 9 before Sections 11/12."
    )


Preprocessed windows saved to Drive.


## 10. STAGE 2 — LSTM-AE: TRAINING

Bidirectional LSTM Autoencoder for temporal pattern learning in latent space.
- Input: windowed latent vectors `(window_size, S1_LATENT_DIM)`
- Bottleneck: `S2_LATENT_DIM` (further compression of already-compressed vectors)
- Output: reconstructed latent sequence with LINEAR activation

In [20]:
# ============================================================
# 10.1 Build LSTM-AE Architecture
# ============================================================
def build_lstm_ae(window_size, latent_dim_in, latent_dim_out, dropout_rate):
    """
    Stage 2: Bidirectional LSTM Autoencoder.
    Learns temporal patterns from sequences of Stage 1 latent vectors.
    """
    inputs = Input(shape=(window_size, latent_dim_in), name='seq_input')

    # === ENCODER ===
    x = Bidirectional(LSTM(32, return_sequences=True))(inputs)
    x = Dropout(dropout_rate)(x)
    x = Bidirectional(LSTM(16, return_sequences=False))(x)
    x = Dropout(dropout_rate)(x)

    bottleneck = Dense(latent_dim_out, activation='linear',
                       name='temporal_bottleneck')(x)

    # === DECODER ===
    x = RepeatVector(window_size)(bottleneck)
    x = Bidirectional(LSTM(16, return_sequences=True))(x)
    x = Dropout(dropout_rate)(x)
    x = Bidirectional(LSTM(32, return_sequences=True))(x)
    outputs = TimeDistributed(
        Dense(latent_dim_in, activation='linear')
    )(x)

    model = Model(inputs, outputs, name='BiLSTM_AE_Stage2')
    return model

actual_window = windows_train.shape[1]
actual_latent = windows_train.shape[2]
model_s2 = build_lstm_ae(actual_window, actual_latent, S2_LATENT_DIM, S2_DROPOUT)

print("=== BiLSTM-AE Architecture ===")
model_s2.summary()
s2_params = model_s2.count_params()
s2_compression = (actual_window * actual_latent) / S2_LATENT_DIM
print(f"\nTotal parameters   : {s2_params:,}")
print(f"Temporal compression: ({actual_window}×{actual_latent}) / {S2_LATENT_DIM} = {s2_compression:.1f}x")

=== BiLSTM-AE Architecture ===


Model: "BiLSTM_AE_Stage2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ seq_input (InputLayer)          │ (None, 10, 8)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 10, 64)         │        10,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 32)             │        10,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ temporal_bottleneck (Dense)     │ (None, 4)              │           132 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 10, 4)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 10, 32)         │         2,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 10, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 10, 64)         │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 10, 8)          │           520 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,844 (159.55 KB)

 Trainable params: 40,844 (159.55 KB)

 Non-trainable params: 0 (0.00 B)


Total parameters   : 40,844
Temporal compression: (10×8) / 4 = 20.0x


In [21]:
# ============================================================
# 10.2 Train Stage 2
# ============================================================
if RERUN_STAGE2_TRAIN:
    model_s2.compile(optimizer=Adam(learning_rate=S2_LR), loss='mse')

    callbacks_s2 = [
        EarlyStopping(monitor='val_loss', patience=S2_ES_PATIENCE,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=S2_LR_PATIENCE, min_lr=1e-6, verbose=1),
        ModelCheckpoint(STAGE2_CHECKPOINT_PATH,
                        monitor='val_loss', save_best_only=True, verbose=1)
    ]

    print(f"Training Stage 2 BiLSTM-AE...")
    print(f"  Train: {windows_train.shape}, Val: {windows_val.shape}")

    if ENABLE_TF_DATA_PIPELINE:
        train_ds_s2 = tf.data.Dataset.from_tensor_slices((windows_train, windows_train))
        val_ds_s2 = tf.data.Dataset.from_tensor_slices((windows_val, windows_val))

        shuffle_buffer = min(len(windows_train), 100000)
        if shuffle_buffer > 1:
            train_ds_s2 = train_ds_s2.shuffle(
                shuffle_buffer,
                seed=RANDOM_SEED,
                reshuffle_each_iteration=True
            )

        train_ds_s2 = train_ds_s2.batch(S2_BATCH_SIZE, drop_remainder=False)
        val_ds_s2 = val_ds_s2.batch(S2_BATCH_SIZE, drop_remainder=False)

        if PREFETCH_AUTOTUNE:
            train_ds_s2 = train_ds_s2.prefetch(tf.data.AUTOTUNE)
            val_ds_s2 = val_ds_s2.prefetch(tf.data.AUTOTUNE)

        history_s2 = model_s2.fit(
            train_ds_s2,
            validation_data=val_ds_s2,
            epochs=S2_MAX_EPOCHS,
            callbacks=callbacks_s2,
            verbose=1
        )
    else:
        history_s2 = model_s2.fit(
            windows_train, windows_train,
            validation_data=(windows_val, windows_val),
            batch_size=S2_BATCH_SIZE,
            epochs=S2_MAX_EPOCHS,
            callbacks=callbacks_s2,
            verbose=1
        )

    # Diagnostics
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(history_s2.history['loss'], label='Train Loss')
    axes[0].plot(history_s2.history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('MSE Loss')
    axes[0].set_title('Stage 2 BiLSTM-AE Training Curve')
    axes[0].legend()
    axes[0].set_yscale('log')

    # Compare Stage 1 vs Stage 2 convergence
    try:
        axes[1].plot(history_s1.history['val_loss'], label='Stage 1 Val Loss', alpha=0.7)
    except Exception:
        pass
    axes[1].plot(history_s2.history['val_loss'], label='Stage 2 Val Loss', alpha=0.7)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MSE Loss')
    axes[1].set_title('Stage 1 vs Stage 2 Convergence')
    axes[1].legend()
    axes[1].set_yscale('log')

    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/stage2_training_curve.png", dpi=150, bbox_inches='tight')
    plt.show()

    s2_final_train = history_s2.history['loss'][-1]
    s2_final_val = history_s2.history['val_loss'][-1]
    s2_best_val = min(history_s2.history['val_loss'])
    s2_best_epoch = history_s2.history['val_loss'].index(s2_best_val) + 1

    print(f"\n=== Stage 2 Training Summary ===")
    print(f"Best epoch     : {s2_best_epoch}")
    print(f"Best val_loss  : {s2_best_val:.6f}")
    print(f"Final train    : {s2_final_train:.6f}")
    print(f"Final val      : {s2_final_val:.6f}")

    if s2_final_val > s2_final_train * 2:
        print("[PERINGATAN] val_loss > 2x train_loss — possible overfitting!")
else:
    if not os.path.exists(STAGE2_CHECKPOINT_PATH):
        raise FileNotFoundError(f"Checkpoint missing for Stage 2: {STAGE2_CHECKPOINT_PATH}")
    model_s2.load_weights(STAGE2_CHECKPOINT_PATH)
    print(f"[CACHED] Stage 2 weights loaded from checkpoint: {STAGE2_CHECKPOINT_PATH}")

[CACHED] Stage 2 weights loaded from checkpoint: /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/checkpoints/stage2_best_v3_sessionfix.keras


## 10.5 ? ANOMALY SCORE REDESIGN

Part 4 redesign untuk normalisasi per-stage, direction fix Stage 2, dan weight tuning berbasis CIC calibration split.


In [22]:

# ============================================================
# 10.5 Scoring Redesign Helpers (Part 4)
# ============================================================
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split


def fit_stage_score_stats(err_s1_val, err_s2_val_perflow, eps=1e-8):
    s1_mean = float(np.mean(err_s1_val))
    s1_std = float(max(np.std(err_s1_val), eps))
    s2_mean = float(np.mean(err_s2_val_perflow))
    s2_std = float(max(np.std(err_s2_val_perflow), eps))
    return {'s1_mean': s1_mean, 's1_std': s1_std, 's2_mean': s2_mean, 's2_std': s2_std, 'eps': eps}


def normalize_stage_errors(err_s1, err_s2, stats):
    eps = float(stats.get('eps', 1e-8))
    s1_norm = (np.asarray(err_s1) - float(stats['s1_mean'])) / (float(stats['s1_std']) + eps)
    s2_norm = (np.asarray(err_s2) - float(stats['s2_mean'])) / (float(stats['s2_std']) + eps)
    return s1_norm, s2_norm


def combine_anomaly_score(s1_norm, s2_norm, w1, w2, mode='combined_normalized', invert_s2=True, shift_ref_min=None):
    if mode == 'stage1_only':
        combined = np.asarray(s1_norm)
    elif mode == 'stage2_only':
        combined = -np.asarray(s2_norm) if invert_s2 else np.asarray(s2_norm)
    else:
        s2_contrib = -np.asarray(s2_norm) if invert_s2 else np.asarray(s2_norm)
        combined = float(w1) * np.asarray(s1_norm) + float(w2) * s2_contrib

    combined = np.clip(combined, -5.0, 5.0)
    ref_min = float(np.min(combined) if shift_ref_min is None else shift_ref_min)
    shifted = combined - ref_min
    return shifted, ref_min


def direction_diagnostic(y_true, scores):
    auc_raw = float(roc_auc_score(y_true, scores))
    auc_inv = float(roc_auc_score(y_true, -np.asarray(scores)))
    return {'auc_raw': auc_raw, 'auc_inverted': auc_inv, 'direction_issue': bool(auc_raw < auc_inv), 'delta': float(auc_inv - auc_raw)}


def run_weight_grid_search(s1_errors, s2_errors, y_true, stage_stats, w1_candidates, mode='combined_normalized', invert_s2=True):
    rows = []
    best_row = None

    s1_norm, s2_norm = normalize_stage_errors(s1_errors, s2_errors, stage_stats)
    for w1 in w1_candidates:
        w2 = 1.0 - float(w1)
        scores, shift_ref = combine_anomaly_score(
            s1_norm, s2_norm, w1, w2,
            mode=mode,
            invert_s2=invert_s2,
            shift_ref_min=None,
        )
        auc = float(roc_auc_score(y_true, scores))
        ap = float(average_precision_score(y_true, scores))

        benign_scores = scores[np.asarray(y_true) == 0]
        if len(benign_scores) > 0:
            t95 = float(np.percentile(benign_scores, 95))
            fpr95 = float((benign_scores > t95).mean())
        else:
            t95 = float('nan')
            fpr95 = float('nan')

        row = {
            'w1': float(w1),
            'w2': float(w2),
            'roc_auc': auc,
            'pr_auc': ap,
            'benign_fpr95': fpr95,
            'shift_ref_min': float(shift_ref),
        }
        rows.append(row)

        if best_row is None or (row['roc_auc'], row['pr_auc'], -row['benign_fpr95']) > (best_row['roc_auc'], best_row['pr_auc'], -best_row['benign_fpr95']):
            best_row = row

    return pd.DataFrame(rows), best_row

print('Part 4 scoring helper functions ready.')



Part 4 scoring helper functions ready.


## 11. THRESHOLD DETERMINATION

Thresholds determined ONLY from benign validation data reconstruction errors.
No test set information is used.

**Anomaly score mapping strategy:**
- Stage 1 error: per-flow MSE
- Stage 2 error: per-window MSE → mapped back to each flow in the window
- Combined: weighted sum of Stage 1 and Stage 2 errors per flow

In [23]:

# ============================================================
# 11.1 Compute Reconstruction Errors (Validation) + Fit v4 Scorer
# ============================================================
def _align_recon_tensors(x_true, x_pred):
    x_true = np.asarray(x_true)
    x_pred = np.asarray(x_pred)

    if x_true.shape != x_pred.shape:
        if x_true.ndim == x_pred.ndim + 1 and x_true.shape[-1] == 1:
            x_true = np.squeeze(x_true, axis=-1)
        elif x_pred.ndim == x_true.ndim + 1 and x_pred.shape[-1] == 1:
            x_pred = np.squeeze(x_pred, axis=-1)

    if x_true.shape != x_pred.shape:
        raise ValueError(
            f"Reconstruction shape mismatch after alignment: true={x_true.shape}, pred={x_pred.shape}"
        )

    return x_true, x_pred


def compute_recon_error(model, X, batch_size=512):
    X_arr = np.asarray(X)
    input_shape = model.input_shape[0] if isinstance(model.input_shape, list) else model.input_shape
    expected_ndim = len(input_shape)
    X_model = X_arr

    if X_model.ndim + 1 == expected_ndim and input_shape[-1] == 1:
        X_model = X_model[..., np.newaxis]

    X_pred = model.predict(X_model, batch_size=batch_size, verbose=0)
    X_true_aligned, X_pred_aligned = _align_recon_tensors(X_arr, X_pred)

    axes = tuple(range(1, X_true_aligned.ndim))
    return np.mean((X_true_aligned - X_pred_aligned) ** 2, axis=axes)


def _compute_s1_s2_components(X_scaled, sids, ts):
    err_s1 = compute_recon_error(model_s1, X_scaled, batch_size=S1_BATCH_SIZE)

    X_s1 = X_scaled[..., np.newaxis]
    latent = encoder_s1.predict(X_s1, batch_size=S1_BATCH_SIZE, verbose=0)

    windows, masks, flow_indices = create_session_windows(
        latent, sids, ts, WINDOW_SIZE, MIN_FLOWS_PER_SESSION
    )

    if len(windows) > 0:
        err_s2_windows = compute_recon_error(model_s2, windows, batch_size=S2_BATCH_SIZE)
        err_s2_perflow = np.full(len(X_scaled), np.nan, dtype=np.float32)
        for w_idx in range(len(flow_indices)):
            fi = flow_indices[w_idx]
            valid = fi[fi >= 0]
            err_s2_perflow[valid] = err_s2_windows[w_idx]

        not_windowed = np.isnan(err_s2_perflow)
        if not_windowed.any():
            err_s2_perflow[not_windowed] = np.nanmedian(err_s2_perflow)
    else:
        err_s2_perflow = np.zeros(len(X_scaled), dtype=np.float32)

    return err_s1, err_s2_perflow


def _preprocess_eval_core(df_raw, dataset_name='cic-calib'):
    df = df_raw.copy()
    df.columns = df.columns.str.strip()

    if '__source_file' not in df.columns:
        df['__source_file'] = dataset_name
    if '__row_id_in_file' not in df.columns:
        df['__row_id_in_file'] = np.arange(len(df), dtype=np.int64)

    canonical = resolve_canonical_columns(df.columns)
    rename_map = {}
    if 'src_ip' in canonical: rename_map[canonical['src_ip']] = 'Src_IP'
    if 'dst_ip' in canonical: rename_map[canonical['dst_ip']] = 'Dst_IP'
    if 'protocol' in canonical: rename_map[canonical['protocol']] = 'Protocol'
    if 'timestamp' in canonical: rename_map[canonical['timestamp']] = 'Timestamp'
    if 'label' in canonical: rename_map[canonical['label']] = 'Label'
    if 'flow_id' in canonical: rename_map[canonical['flow_id']] = 'Flow_ID'
    if 'src_port' in canonical: rename_map[canonical['src_port']] = 'Src_Port'
    if 'dst_port' in canonical: rename_map[canonical['dst_port']] = 'Dst_Port'
    df = df.rename(columns=rename_map)

    if 'Label' in df.columns:
        labels_raw = df['Label'].astype(str).str.strip()
    else:
        labels_raw = pd.Series(['Unknown'] * len(df), index=df.index)

    y_binary = (~labels_raw.str.upper().isin(['BENIGN'])).astype(int).values

    canonical_after = resolve_canonical_columns(df.columns)
    sids, _ = build_session_id(df, canonical_after, mode=SESSION_KEY_MODE, hash_len=SESSION_HASH_LEN)

    if 'timestamp' in canonical_after:
        ts, _ = parse_timestamp_robust(df[canonical_after['timestamp']], source_file=df['__source_file'], row_ids=df['__row_id_in_file'])
    else:
        ts, _ = parse_timestamp_robust(pd.Series([np.nan] * len(df)), source_file=df['__source_file'], row_ids=df['__row_id_in_file'])

    drop_cols = ['Label', 'Flow_ID', 'Timestamp', 'Src_IP', 'Dst_IP', 'Src_Port', 'Dst_Port', 'session_id', 'ts_parsed', '__source_file', '__row_id_in_file']
    df_feat = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

    for col in df_feat.columns:
        if df_feat[col].dtype == object:
            df_feat[col] = df_feat[col].replace(['Infinity', 'infinity', '-Infinity', '-infinity', 'inf', '-inf'], np.nan)
            df_feat[col] = pd.to_numeric(df_feat[col], errors='coerce')
    df_feat = df_feat.replace([np.inf, -np.inf], np.nan).astype(np.float32)

    for col in df_feat.columns:
        if col in median_values:
            df_feat[col] = df_feat[col].fillna(float(median_values[col]))
        else:
            df_feat[col] = df_feat[col].fillna(0.0)

    map_info = summarize_feature_mapping(selected_features, df_feat.columns, FEATURE_ALIAS_MAP)
    if STRICT_FEATURE_MATCH and map_info['match_ratio'] < MIN_FEATURE_MATCH_RATIO:
        raise ValueError(f"[cic-calib] Feature mapping too low: {map_info['match_ratio']:.2%}")

    X_df = pd.DataFrame()
    for sf in selected_features:
        if sf in map_info['resolved_map']:
            X_df[sf] = df_feat[map_info['resolved_map'][sf]]
        else:
            X_df[sf] = 0.0

    X_scaled = scaler.transform(X_df.values).astype(np.float32)
    X_scaled = np.clip(X_scaled, -5, 5)
    return X_scaled, y_binary, sids, ts


# Stage 1: per-flow error on validation (benign val)
err_s1_val = compute_recon_error(model_s1, X_val, batch_size=S1_BATCH_SIZE)
print(f"Stage 1 val error: mean={err_s1_val.mean():.6f}, std={err_s1_val.std():.6f}")

# Stage 2: per-window -> per-flow error on validation (benign val)
err_s2_val_windows = compute_recon_error(model_s2, windows_val, batch_size=S2_BATCH_SIZE)
print(f"Stage 2 val error: mean={err_s2_val_windows.mean():.6f}, std={err_s2_val_windows.std():.6f}")

err_s2_val_perflow = np.full(len(X_val), np.nan, dtype=np.float32)
for w_idx in range(len(flow_idx_val)):
    flow_idxs = flow_idx_val[w_idx]
    valid = flow_idxs[flow_idxs >= 0]
    err_s2_val_perflow[valid] = err_s2_val_windows[w_idx]
not_windowed = np.isnan(err_s2_val_perflow)
if not_windowed.any():
    err_s2_val_perflow[not_windowed] = np.nanmedian(err_s2_val_perflow)

stage_score_stats = fit_stage_score_stats(err_s1_val, err_s2_val_perflow)

# Build CIC calibration split for weight tuning (10% stratified)
df_cic_all_for_calib = pd.read_parquet(f"{PREPROCESSED_DIR}/cic2017_all_raw.parquet")
X_cic_all_cal, y_cic_all_cal, sids_cic_all_cal, ts_cic_all_cal = _preprocess_eval_core(df_cic_all_for_calib, dataset_name='cic-calib-split')

all_idx = np.arange(len(y_cic_all_cal))
calib_idx, holdout_idx = train_test_split(
    all_idx,
    train_size=CIC_CALIB_RATIO,
    random_state=CIC_CALIB_RANDOM_SEED,
    stratify=y_cic_all_cal,
)

X_cic_calib = X_cic_all_cal[calib_idx]
y_cic_calib = y_cic_all_cal[calib_idx]
sids_cic_calib = sids_cic_all_cal[calib_idx]
ts_cic_calib = ts_cic_all_cal[calib_idx]

err_s1_calib, err_s2_calib = _compute_s1_s2_components(X_cic_calib, sids_cic_calib, ts_cic_calib)

if WEIGHT_GRID_SEARCH and SCORE_MODE == 'combined_normalized':
    weight_search_df, best_weight_row = run_weight_grid_search(
        err_s1_calib,
        err_s2_calib,
        y_cic_calib,
        stage_score_stats,
        W1_CANDIDATES,
        mode=SCORE_MODE,
        invert_s2=INVERT_STAGE2_SCORE,
    )
    best_w1 = float(best_weight_row['w1'])
    best_w2 = float(best_weight_row['w2'])
    print(f"[WEIGHT SEARCH] best_w1={best_w1:.2f}, best_w2={best_w2:.2f}, auc={best_weight_row['roc_auc']:.4f}")
else:
    weight_search_df = pd.DataFrame([{'w1': S1_WEIGHT, 'w2': S2_WEIGHT, 'roc_auc': np.nan, 'pr_auc': np.nan, 'benign_fpr95': np.nan, 'shift_ref_min': np.nan}])
    best_w1, best_w2 = float(S1_WEIGHT), float(S2_WEIGHT)

weight_search_df.to_csv(f"{SCORE_ARTIFACTS_DIR}/weight_grid_search_results.csv", index=False)

s1_val_norm, s2_val_norm = normalize_stage_errors(err_s1_val, err_s2_val_perflow, stage_score_stats)
combined_val, shift_ref_min = combine_anomaly_score(
    s1_val_norm,
    s2_val_norm,
    best_w1,
    best_w2,
    mode=SCORE_MODE,
    invert_s2=INVERT_STAGE2_SCORE,
    shift_ref_min=None,
)

score_calibrator = {
    'score_pipeline_version': SCORE_PIPELINE_VERSION,
    'source_pipeline_version': SOURCE_PIPELINE_VERSION,
    'mode': SCORE_MODE,
    'invert_stage2': bool(INVERT_STAGE2_SCORE),
    'score_normalize_per_stage': bool(SCORE_NORMALIZE_PER_STAGE),
    'w1': float(best_w1),
    'w2': float(best_w2),
    'shift_ref_min': float(shift_ref_min),
    'stage_stats': stage_score_stats,
    'cic_calib_ratio': float(CIC_CALIB_RATIO),
    'cic_calib_random_seed': int(CIC_CALIB_RANDOM_SEED),
}
with open(f"{SCORE_ARTIFACTS_DIR}/score_calibrator.json", 'w', encoding='utf-8') as f:
    json.dump(score_calibrator, f, indent=2)

# Lock selected weights for downstream cells
S1_WEIGHT = best_w1
S2_WEIGHT = best_w2

print(f"Combined val score (v4): mean={combined_val.mean():.6f}, std={combined_val.std():.6f}")
print(f"Combined val range: [{combined_val.min():.6f}, {combined_val.max():.6f}]")
print(f"Score calibrator saved: {SCORE_ARTIFACTS_DIR}/score_calibrator.json")



Stage 1 val error: mean=0.025940, std=0.075551
Stage 2 val error: mean=0.893805, std=0.639846
[WEIGHT SEARCH] best_w1=0.90, best_w2=0.10, auc=0.8421
Combined val score (v4): mean=1.105938, std=0.630739
Combined val range: [0.000000, 6.125345]
Score calibrator saved: /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/artifacts/v5_domainfix/score_calibrator.json


In [24]:

# ============================================================
# 11.2 Threshold Strategies (v4 score)
# ============================================================
from scipy.stats import norm

threshold_results = {}

# --- 1. Z-Score ---
mu, sigma = combined_val.mean(), combined_val.std()
for k in K_SIGMA_VALUES:
    t = mu + k * sigma
    fpr = (combined_val > t).mean()
    threshold_results[f'zscore_k{k}'] = {'threshold': float(t), 'fpr': float(fpr), 'method': 'zscore', 'k': k}

# --- 2. Percentile ---
for p in PERCENTILE_VALUES:
    t = np.percentile(combined_val, p)
    fpr = (combined_val > t).mean()
    threshold_results[f'percentile_{p}'] = {'threshold': float(t), 'fpr': float(fpr), 'method': 'percentile', 'p': p}

# --- 3. Gaussian Fit ---
gauss_mu, gauss_sigma = norm.fit(combined_val)
for k in K_SIGMA_VALUES:
    t = gauss_mu + k * gauss_sigma
    fpr = (combined_val > t).mean()
    threshold_results[f'gaussian_k{k}'] = {'threshold': float(t), 'fpr': float(fpr), 'method': 'gaussian', 'k': k}

# --- 4. IQR ---
q1 = np.percentile(combined_val, 25)
q3 = np.percentile(combined_val, 75)
iqr = q3 - q1
for k in [1.5, 2.0, 3.0]:
    t = q3 + k * iqr
    fpr = (combined_val > t).mean()
    threshold_results[f'iqr_k{k}'] = {'threshold': float(t), 'fpr': float(fpr), 'method': 'iqr', 'k': k}

# --- 5. MAD ---
median_err = np.median(combined_val)
mad = np.median(np.abs(combined_val - median_err))
for k in K_SIGMA_VALUES:
    t = median_err + k * 1.4826 * mad
    fpr = (combined_val > t).mean()
    threshold_results[f'mad_k{k}'] = {'threshold': float(t), 'fpr': float(fpr), 'method': 'mad', 'k': k}

print("=== Threshold Comparison (v4) ===")
print(f"{'Method':<20} {'Threshold':>10} {'FPR':>8}")
print("-" * 40)
for name, res in sorted(threshold_results.items(), key=lambda x: x[1]['fpr']):
    print(f"{name:<20} {res['threshold']:>10.6f} {res['fpr']:>8.4f}")

best_threshold_name = None
best_threshold_val = None
for k in sorted(K_SIGMA_VALUES):
    key = f'zscore_k{k}'
    if threshold_results[key]['fpr'] < 0.05:
        best_threshold_name = key
        best_threshold_val = threshold_results[key]['threshold']
        break

if best_threshold_val is None:
    best_threshold_name = 'percentile_95'
    best_threshold_val = threshold_results['percentile_95']['threshold']
    print("[INFO] No Z-Score threshold achieved FPR < 0.05, using percentile_95")

best_t = float(best_threshold_val)
print(f"Selected threshold (v4): {best_threshold_name} = {best_t:.6f} (FPR={threshold_results[best_threshold_name]['fpr']:.4f})")

with open(f"{SCORE_ARTIFACTS_DIR}/threshold_results_v4.json", 'w', encoding='utf-8') as f:
    json.dump(threshold_results, f, indent=2)
with open(f"{SCORE_ARTIFACTS_DIR}/best_threshold_v4.json", 'w', encoding='utf-8') as f:
    json.dump({'name': best_threshold_name, 'value': best_t}, f, indent=2)
print(f"Saved: {SCORE_ARTIFACTS_DIR}/threshold_results_v4.json")



=== Threshold Comparison (v4) ===
Method                Threshold      FPR
----------------------------------------
percentile_99.5        6.098609   0.0050
percentile_99          4.413470   0.0100
zscore_k3.0            2.998156   0.0220
gaussian_k3.0          2.998156   0.0220
zscore_k2.5            2.682786   0.0277
gaussian_k2.5          2.682786   0.0277
percentile_97          2.579163   0.0300
zscore_k2.0            2.367417   0.0359
gaussian_k2.0          2.367417   0.0359
zscore_k1.5            2.052047   0.0491
gaussian_k1.5          2.052047   0.0491
percentile_95          2.036287   0.0500
iqr_k3.0               1.681911   0.0760
iqr_k2.0               1.478730   0.1023
iqr_k1.5               1.377139   0.1204
mad_k3.0               1.356697   0.1247
mad_k2.5               1.288037   0.1410
mad_k2.0               1.219377   0.1620
mad_k1.5               1.150717   0.1911
Selected threshold (v4): zscore_k1.5 = 2.052047 (FPR=0.0491)
Saved: /content/drive/MyDrive/mscnn-bilstm-a

In [25]:
# ============================================================
# 11.3 Visualize Error Distribution + Thresholds
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Stage 1 error distribution
axes[0].hist(err_s1_val, bins=100, density=True, alpha=0.7, color='steelblue')
axes[0].set_title('Stage 1 Reconstruction Error (Val)')
axes[0].set_xlabel('MSE')
axes[0].set_ylabel('Density')
axes[0].set_xlim(0, np.percentile(err_s1_val, 99.5))

# Stage 2 error distribution
axes[1].hist(err_s2_val_perflow[~np.isnan(err_s2_val_perflow)], bins=100,
             density=True, alpha=0.7, color='coral')
axes[1].set_title('Stage 2 Reconstruction Error (Val, per-flow)')
axes[1].set_xlabel('MSE')
axes[1].set_ylabel('Density')
axes[1].set_xlim(0, np.percentile(err_s2_val_perflow[~np.isnan(err_s2_val_perflow)], 99.5))

# Combined error + thresholds
axes[2].hist(combined_val, bins=100, density=True, alpha=0.7, color='mediumpurple')
axes[2].axvline(x=best_threshold_val, color='red', linewidth=2, linestyle='--',
                label=f'Best: {best_threshold_name}')
for name, res in threshold_results.items():
    if 'zscore' in name:
        axes[2].axvline(x=res['threshold'], color='gray', linewidth=0.5, alpha=0.5)
axes[2].set_title('Combined Error + Thresholds (Val)')
axes[2].set_xlabel('Weighted Score')
axes[2].set_ylabel('Density')
axes[2].set_xlim(0, np.percentile(combined_val, 99.9))
axes[2].legend()

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/threshold_distributions.png", dpi=150, bbox_inches='tight')
plt.show()

## 12. EVALUATION — CIC-IDS-2017 (Secondary)

Full evaluation pipeline on CIC-IDS-2017 with all attack labels.
Uses saved artifacts (scaler, imputer, feature list) — no re-fitting.

In [26]:
# ============================================================
# 12.0 Evaluation Helper Functions
# ============================================================
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                              f1_score, roc_auc_score, average_precision_score,
                              roc_curve, precision_recall_curve)

def preprocess_eval_dataset(df_raw, selected_features, median_values, scaler,
                             col_name_map=None, return_meta=False, dataset_name='dataset'):
    """
    Apply saved preprocessing pipeline to evaluation dataset.
    Returns: X_scaled (np.array), labels (np.array), session_ids, timestamps
    If return_meta=True, also returns mapping metadata dict.
    """
    df = df_raw.copy()
    df.columns = df.columns.str.strip()

    if '__source_file' not in df.columns:
        df['__source_file'] = dataset_name
    if '__row_id_in_file' not in df.columns:
        df['__row_id_in_file'] = np.arange(len(df), dtype=np.int64)

    canonical = resolve_canonical_columns(df.columns)

    rename_map = {}
    if 'src_ip' in canonical: rename_map[canonical['src_ip']] = 'Src_IP'
    if 'dst_ip' in canonical: rename_map[canonical['dst_ip']] = 'Dst_IP'
    if 'protocol' in canonical: rename_map[canonical['protocol']] = 'Protocol'
    if 'timestamp' in canonical: rename_map[canonical['timestamp']] = 'Timestamp'
    if 'label' in canonical: rename_map[canonical['label']] = 'Label'
    if 'flow_id' in canonical: rename_map[canonical['flow_id']] = 'Flow_ID'
    if 'src_port' in canonical: rename_map[canonical['src_port']] = 'Src_Port'
    if 'dst_port' in canonical: rename_map[canonical['dst_port']] = 'Dst_Port'
    df = df.rename(columns=rename_map)

    dropped_header_rows = 0

    if 'Label' in df.columns:
        labels_raw = df['Label'].astype(str).str.strip()

        if DROP_HEADER_LEAK_ROWS:
            labels_norm = labels_raw.map(normalize_colname)
            valid_mask = (labels_norm != 'label') & (labels_norm != '') & (labels_norm != 'nan')
            dropped_header_rows = int((~valid_mask).sum())
            if dropped_header_rows > 0:
                df = df[valid_mask].copy()
                labels_raw = labels_raw[valid_mask]

        labels_raw = labels_raw.astype(str)
    else:
        labels_raw = pd.Series(['Unknown'] * len(df), index=df.index)

    y_binary = (~labels_raw.str.upper().isin(['BENIGN'])).astype(int).values
    attack_types = labels_raw.values

    canonical_after = resolve_canonical_columns(df.columns)
    sids, sid_report = build_session_id(
        df,
        canonical_after,
        mode=SESSION_KEY_MODE,
        hash_len=SESSION_HASH_LEN,
    )

    if 'timestamp' in canonical_after:
        ts_col = canonical_after['timestamp']
        ts, ts_report = parse_timestamp_robust(
            df[ts_col],
            source_file=df['__source_file'],
            row_ids=df['__row_id_in_file'],
        )
    else:
        ts, ts_report = parse_timestamp_robust(
            pd.Series([np.nan] * len(df)),
            source_file=df['__source_file'],
            row_ids=df['__row_id_in_file'],
        )
        ts_report['timestamp_col_detected'] = False

    drop_cols = ['Label', 'Flow_ID', 'Timestamp', 'Src_IP', 'Dst_IP',
                 'Src_Port', 'Dst_Port', 'session_id', 'ts_parsed',
                 '__source_file', '__row_id_in_file']
    df_feat = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

    for col in df_feat.columns:
        if df_feat[col].dtype == object:
            df_feat[col] = df_feat[col].replace(
                ['Infinity', 'infinity', '-Infinity', '-infinity', 'inf', '-inf'], np.nan
            )
            df_feat[col] = pd.to_numeric(df_feat[col], errors='coerce')
    df_feat = df_feat.replace([np.inf, -np.inf], np.nan)
    df_feat = df_feat.astype(np.float32)

    for col in df_feat.columns:
        if col in median_values:
            df_feat[col] = df_feat[col].fillna(float(median_values[col]))
        else:
            df_feat[col] = df_feat[col].fillna(0.0)

    map_info = summarize_feature_mapping(selected_features, df_feat.columns, FEATURE_ALIAS_MAP)
    resolved_map = map_info['resolved_map']

    print(
        f"[{dataset_name}] Feature mapping coverage: "
        f"{map_info['matched_feature_count']}/{len(selected_features)} "
        f"({map_info['match_ratio']:.2%})"
    )
    if map_info['missing_feature_count'] > 0:
        print(f"[{dataset_name}] Missing features ({map_info['missing_feature_count']}): {map_info['missing_feature_list'][:10]}")

    if STRICT_FEATURE_MATCH and map_info['match_ratio'] < MIN_FEATURE_MATCH_RATIO:
        raise ValueError(
            f"[{dataset_name}] Feature match ratio too low: {map_info['match_ratio']:.2%} < {MIN_FEATURE_MATCH_RATIO:.0%}. "
            f"Missing: {map_info['missing_feature_list'][:20]}"
        )

    X_df = pd.DataFrame()
    for sf in selected_features:
        if sf in resolved_map:
            X_df[sf] = df_feat[resolved_map[sf]]
        else:
            X_df[sf] = 0.0

    X_scaled = scaler.transform(X_df.values).astype(np.float32)
    X_scaled = np.clip(X_scaled, -5, 5)

    meta = {
        'matched_feature_count': map_info['matched_feature_count'],
        'missing_feature_count': map_info['missing_feature_count'],
        'missing_feature_list': map_info['missing_feature_list'],
        'match_ratio': map_info['match_ratio'],
        'dropped_header_rows': dropped_header_rows,
        'timestamp_col_detected': bool(ts_report['timestamp_col_detected']),
        'timestamp_valid_ratio': float(ts_report['timestamp_valid_ratio']),
        'timestamp_parser_used': ts_report['timestamp_parser_used'],
        'unique_sessions_eval': int(sid_report['unique_sessions']),
        'largest_session_share_eval': float(sid_report['largest_session_share']),
        'session_fallback_path': sid_report['fallback_path_used'],
    }

    if return_meta:
        return X_scaled, y_binary, attack_types, sids, ts, meta
    return X_scaled, y_binary, attack_types, sids, ts


def compute_eval_scores(X_scaled, sids, ts, model_s1, encoder_s1, model_s2,
                         err_s1_scaler=None, err_s2_scaler=None,
                         window_size=10, min_flows=3,
                         score_calibrator=None, return_components=False):
    """Compute anomaly scores with Part 4 scorer (v4)."""
    if 'create_session_windows' not in globals():
        raise NameError(
            "create_session_windows is not defined. Run Section 8.2 before evaluation."
        )

    if score_calibrator is None:
        score_calibrator = globals().get('score_calibrator')
    if score_calibrator is None:
        raise RuntimeError('score_calibrator is missing. Run Section 11.1 first.')

    err_s1 = compute_recon_error(model_s1, X_scaled, batch_size=S1_BATCH_SIZE)

    X_s1 = X_scaled[..., np.newaxis]
    latent = encoder_s1.predict(X_s1, batch_size=S1_BATCH_SIZE, verbose=0)

    windows, masks, flow_indices = create_session_windows(
        latent, sids, ts, window_size, min_flows
    )

    if len(windows) > 0:
        err_s2_windows = compute_recon_error(model_s2, windows, batch_size=S2_BATCH_SIZE)
        err_s2_perflow = np.full(len(X_scaled), np.nan, dtype=np.float32)
        for w_idx in range(len(flow_indices)):
            fi = flow_indices[w_idx]
            valid = fi[fi >= 0]
            err_s2_perflow[valid] = err_s2_windows[w_idx]

        not_windowed = np.isnan(err_s2_perflow)
        if not_windowed.any():
            err_s2_perflow[not_windowed] = np.nanmedian(err_s2_perflow)
    else:
        err_s2_perflow = np.zeros(len(X_scaled), dtype=np.float32)

    stats = score_calibrator['stage_stats']
    s1_norm, s2_norm = normalize_stage_errors(err_s1, err_s2_perflow, stats)

    combined, _ = combine_anomaly_score(
        s1_norm,
        s2_norm,
        score_calibrator['w1'],
        score_calibrator['w2'],
        mode=score_calibrator.get('mode', SCORE_MODE),
        invert_s2=score_calibrator.get('invert_stage2', INVERT_STAGE2_SCORE),
        shift_ref_min=score_calibrator.get('shift_ref_min', None),
    )

    if return_components:
        return combined, err_s1, err_s2_perflow, s1_norm, s2_norm
    return combined, err_s1, err_s2_perflow


def evaluate_with_thresholds(y_true, y_score, threshold_results, dataset_name):
    """Evaluate all threshold strategies and return metrics."""
    all_metrics = {}

    for t_name, t_info in threshold_results.items():
        t_val = t_info['threshold']
        y_pred = (y_score > t_val).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        fpr_val = fp / (fp + tn) if (fp + tn) > 0 else 0

        all_metrics[t_name] = {
            'threshold': t_val,
            'precision': float(precision_score(y_true, y_pred, zero_division=0)),
            'recall': float(recall_score(y_true, y_pred, zero_division=0)),
            'f1': float(f1_score(y_true, y_pred, zero_division=0)),
            'fpr': float(fpr_val),
            'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn),
        }

    try:
        roc_auc = float(roc_auc_score(y_true, y_score))
    except Exception:
        roc_auc = 0.0
    try:
        pr_auc = float(average_precision_score(y_true, y_score))
    except Exception:
        pr_auc = 0.0

    for k in all_metrics:
        all_metrics[k]['roc_auc'] = roc_auc
        all_metrics[k]['pr_auc'] = pr_auc

    return all_metrics, roc_auc, pr_auc

print("Evaluation helper functions defined.")


Evaluation helper functions defined.


In [27]:

# ============================================================
# 12.1 Evaluate CIC-IDS-2017 (All Labels) - Calibration/Holdout (Part 4)
# ============================================================
print("=== Evaluating CIC-IDS-2017 (All Labels) [Part 4] ===")

with open(f"{SCORE_ARTIFACTS_DIR}/score_calibrator.json", 'r', encoding='utf-8') as f:
    score_calibrator = json.load(f)

# Load full CIC-2017 data
df_cic_all = pd.read_parquet(f"{PREPROCESSED_DIR}/cic2017_all_raw.parquet")
print(f"CIC-2017 all labels: {len(df_cic_all):,} rows")
print(f"Label distribution:\n{df_cic_all['Label'].value_counts() if 'Label' in df_cic_all.columns else 'N/A'}")

X_cic_all, y_cic_all, attacks_cic_all, sids_cic_all, ts_cic_all = preprocess_eval_dataset(
    df_cic_all, selected_features, median_values, scaler
)
print(f"Preprocessed: {X_cic_all.shape}, attack types: {len(np.unique(attacks_cic_all))}")

all_idx = np.arange(len(y_cic_all))
calib_idx, holdout_idx = train_test_split(
    all_idx,
    train_size=CIC_CALIB_RATIO,
    random_state=CIC_CALIB_RANDOM_SEED,
    stratify=y_cic_all,
)

# Calibration metrics (for traceability)
X_cic_calib = X_cic_all[calib_idx]
y_cic_calib = y_cic_all[calib_idx]
attacks_cic_calib = attacks_cic_all[calib_idx]
sids_cic_calib = sids_cic_all[calib_idx]
ts_cic_calib = ts_cic_all[calib_idx]

scores_cic_calib, err_s1_cic_calib, err_s2_cic_calib = compute_eval_scores(
    X_cic_calib, sids_cic_calib, ts_cic_calib,
    model_s1, encoder_s1, model_s2,
    window_size=WINDOW_SIZE, min_flows=MIN_FLOWS_PER_SESSION,
    score_calibrator=score_calibrator,
)
metrics_cic_calib, roc_auc_cic_calib, pr_auc_cic_calib = evaluate_with_thresholds(
    y_cic_calib, scores_cic_calib, threshold_results, 'CIC-2017-calibration'
)

# Final holdout metrics
X_cic = X_cic_all[holdout_idx]
y_cic = y_cic_all[holdout_idx]
attacks_cic = attacks_cic_all[holdout_idx]
sids_cic = sids_cic_all[holdout_idx]
ts_cic = ts_cic_all[holdout_idx]

scores_cic, err_s1_cic, err_s2_cic = compute_eval_scores(
    X_cic, sids_cic, ts_cic,
    model_s1, encoder_s1, model_s2,
    window_size=WINDOW_SIZE, min_flows=MIN_FLOWS_PER_SESSION,
    score_calibrator=score_calibrator,
)
print(f"Holdout scores computed: mean={scores_cic.mean():.4f}, std={scores_cic.std():.4f}")

metrics_cic, roc_auc_cic, pr_auc_cic = evaluate_with_thresholds(
    y_cic, scores_cic, threshold_results, 'CIC-2017-holdout'
)

diag_cic = direction_diagnostic(y_cic, scores_cic)
print(f"ROC-AUC raw     : {diag_cic['auc_raw']:.4f}")
print(f"ROC-AUC inverted: {diag_cic['auc_inverted']:.4f}")
print(f"Direction issue : {diag_cic['direction_issue']}")

best_m = metrics_cic[best_threshold_name]
print(f"\n=== CIC-2017 Holdout Results (best threshold: {best_threshold_name}) ===")
print(f"ROC-AUC   : {roc_auc_cic:.4f}")
print(f"PR-AUC    : {pr_auc_cic:.4f}")
print(f"Precision : {best_m['precision']:.4f}")
print(f"Recall    : {best_m['recall']:.4f}")
print(f"F1-Score  : {best_m['f1']:.4f}")
print(f"FPR       : {best_m['fpr']:.4f}")

cic_calib_payload = {
    'n_rows': int(len(y_cic_calib)),
    'roc_auc': float(roc_auc_cic_calib),
    'pr_auc': float(pr_auc_cic_calib),
    'best_threshold_name': best_threshold_name,
    'best_threshold_metrics': metrics_cic_calib[best_threshold_name],
}
with open(f"{SCORE_ARTIFACTS_DIR}/cic_calibration_metrics.json", 'w', encoding='utf-8') as f:
    json.dump(cic_calib_payload, f, indent=2)

cic_holdout_payload = {
    'n_rows': int(len(y_cic)),
    'roc_auc': float(roc_auc_cic),
    'pr_auc': float(pr_auc_cic),
    'best_threshold_name': best_threshold_name,
    'best_threshold_metrics': best_m,
    'direction_diagnostic': diag_cic,
}
with open(f"{SCORE_ARTIFACTS_DIR}/cic_holdout_metrics.json", 'w', encoding='utf-8') as f:
    json.dump(cic_holdout_payload, f, indent=2)

print(f"Saved: {SCORE_ARTIFACTS_DIR}/cic_calibration_metrics.json")
print(f"Saved: {SCORE_ARTIFACTS_DIR}/cic_holdout_metrics.json")

del df_cic_all, X_cic_all, y_cic_all, attacks_cic_all, sids_cic_all, ts_cic_all
gc.collect()




=== Evaluating CIC-IDS-2017 (All Labels) [Part 4] ===
CIC-2017 all labels: 2,830,743 rows
Label distribution:
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64
[dataset] Feature mapping coverage: 50/50 (100.00%)
Preprocessed: (2830743, 50), attack types: 15
Holdout scores computed: mean=1.5807, std=1.5458
ROC-AUC raw     : 0.8633
ROC-AUC inverted: 0.1367
Direction issue : False

=== CIC-2017 Holdout Results (best threshold: zscore_k1.5) ===
ROC

1383

In [28]:
# ============================================================
# 12.2 CIC-2017 Per-Attack Detection Rate
# ============================================================
best_t = threshold_results[best_threshold_name]['threshold']
y_pred_cic = (scores_cic > best_t).astype(int)

print("=== CIC-2017: Per-Attack Detection Rate ===")
attack_df_cic = pd.DataFrame({'attack': attacks_cic, 'y_true': y_cic, 'y_pred': y_pred_cic})

det_rates_cic = {}
for atype, grp in attack_df_cic.groupby('attack'):
    if atype.strip().upper() == 'BENIGN':
        continue
    dr = grp['y_pred'].mean()
    det_rates_cic[atype] = {'detection_rate': dr, 'count': len(grp)}
    status = '✓' if dr > 0.7 else ('⚠' if dr > 0.3 else '✗')
    print(f"  {status} {atype:<30} DR={dr:.4f}  (n={len(grp):,})")

# Benign FPR
benign_grp = attack_df_cic[attack_df_cic['attack'].str.strip().str.upper() == 'BENIGN']
benign_fpr = benign_grp['y_pred'].mean()
print(f"\n  Benign FPR: {benign_fpr:.4f}")

=== CIC-2017: Per-Attack Detection Rate ===
  ⚠ Bot                            DR=0.3781  (n=1,756)
  ⚠ DDoS                           DR=0.6901  (n=115,267)
  ⚠ DoS GoldenEye                  DR=0.6921  (n=9,261)
  ✓ DoS Hulk                       DR=0.7005  (n=208,005)
  ⚠ DoS Slowhttptest               DR=0.4894  (n=4,914)
  ⚠ DoS slowloris                  DR=0.6249  (n=5,222)
  ⚠ FTP-Patator                    DR=0.5054  (n=7,176)
  ✓ Heartbleed                     DR=1.0000  (n=11)
  ✓ Infiltration                   DR=0.9375  (n=32)
  ⚠ PortScan                       DR=0.5036  (n=142,978)
  ⚠ SSH-Patator                    DR=0.4797  (n=5,278)
  ✗ Web Attack � Brute Force       DR=0.0993  (n=1,369)
  ⚠ Web Attack � Sql Injection     DR=0.5556  (n=18)
  ✗ Web Attack � XSS               DR=0.0303  (n=594)

  Benign FPR: 0.0481


In [29]:
# ============================================================
# 12.3 CIC-2017 Visualizations
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Error distribution per class (violin)
unique_attacks = [a for a in np.unique(attacks_cic) if a.strip().upper() != 'BENIGN']
plot_data = [scores_cic[attacks_cic == 'BENIGN']] if 'BENIGN' in attacks_cic else []
plot_labels = ['BENIGN'] if 'BENIGN' in attacks_cic else []

# Use canonical benign label
benign_mask_cic = pd.Series(attacks_cic).str.strip().str.upper() == 'BENIGN'
plot_data = [scores_cic[benign_mask_cic.values]]
plot_labels = ['BENIGN']

for a in sorted(unique_attacks)[:8]:
    mask = attacks_cic == a
    if mask.sum() > 10:
        plot_data.append(scores_cic[mask])
        plot_labels.append(a[:20])

bp = axes[0, 0].boxplot(plot_data, labels=plot_labels, vert=True, patch_artist=True)
for i, box in enumerate(bp['boxes']):
    box.set_facecolor('lightblue' if i == 0 else 'salmon')
axes[0, 0].axhline(y=best_t, color='red', linestyle='--', label='Threshold')
axes[0, 0].set_title('CIC-2017: Error Distribution by Class')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].set_ylabel('Anomaly Score')
axes[0, 0].legend()

# 2. ROC Curve
fpr_curve, tpr_curve, _ = roc_curve(y_cic, scores_cic)
axes[0, 1].plot(fpr_curve, tpr_curve, 'b-', label=f'CIC-2017 (AUC={roc_auc_cic:.4f})')
axes[0, 1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].set_title('ROC Curve — CIC-2017')
axes[0, 1].legend()

# 3. PR Curve
prec_curve, rec_curve, _ = precision_recall_curve(y_cic, scores_cic)
axes[1, 0].plot(rec_curve, prec_curve, 'b-', label=f'CIC-2017 (AP={pr_auc_cic:.4f})')
axes[1, 0].set_xlabel('Recall')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_title('Precision-Recall Curve — CIC-2017')
axes[1, 0].legend()

# 4. Confusion Matrix
cm = confusion_matrix(y_cic, y_pred_cic, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 1],
            xticklabels=['Benign', 'Attack'], yticklabels=['Benign', 'Attack'])
axes[1, 1].set_xlabel('Predicted')
axes[1, 1].set_ylabel('Actual')
axes[1, 1].set_title(f'CIC-2017 Confusion Matrix ({best_threshold_name})')

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/cic2017_evaluation.png", dpi=150, bbox_inches='tight')
plt.show()

# Detection rate bar chart
fig2, ax2 = plt.subplots(figsize=(12, 5))
dr_names = list(det_rates_cic.keys())
dr_vals = [det_rates_cic[k]['detection_rate'] for k in dr_names]
colors_dr = ['#27ae60' if v > 0.7 else '#f39c12' if v > 0.3 else '#e74c3c' for v in dr_vals]
ax2.barh(dr_names, dr_vals, color=colors_dr)
ax2.axvline(x=0.7, color='green', linestyle='--', alpha=0.5, label='Good (>0.7)')
ax2.axvline(x=0.3, color='red', linestyle='--', alpha=0.5, label='Poor (<0.3)')
ax2.set_xlabel('Detection Rate')
ax2.set_title('CIC-2017: Detection Rate per Attack Category')
ax2.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/cic2017_detection_rates.png", dpi=150, bbox_inches='tight')
plt.show()

# Save CIC results
pd.DataFrame(metrics_cic).T.to_csv(f"{RESULTS_DIR}/cic2017_metrics.csv")
print("CIC-2017 evaluation complete. Results saved.")

/tmp/ipython-input-1743/3860817239.py:22: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = axes[0, 0].boxplot(plot_data, labels=plot_labels, vert=True, patch_artist=True)


CIC-2017 evaluation complete. Results saved.


## 13. EVALUATION — CSE-CIC-IDS-2018 (Primary)

**This is the primary evaluation target** — model has never seen this dataset.
Chunk-based loading to handle large file sizes.

In [30]:
# ============================================================
# 13.1 Load & Evaluate CSE-CIC-IDS-2018
# ============================================================
print("=== Evaluating CSE-CIC-IDS-2018 (All Labels) ===")

cse_csv_files = sorted([
    os.path.join(CSE2018_DIR, f)
    for f in os.listdir(CSE2018_DIR) if f.endswith('.csv')
])

all_X_cse, all_y_cse, all_attacks_cse, all_sids_cse, all_ts_cse = [], [], [], [], []
cse_file_meta = []
failed_files = []

for fpath in cse_csv_files:
    fname = os.path.basename(fpath)
    print(f"  Processing {fname}...", end=" ")

    try:
        chunks_data = []
        dropped_header_rows_file = 0

        for chunk in pd.read_csv(fpath, chunksize=CHUNK_SIZE, low_memory=False, encoding='utf-8'):
            chunk.columns = chunk.columns.str.strip()

            # Early header-leak cleaning at chunk level
            label_col = None
            for c in chunk.columns:
                if normalize_colname(c) == 'label':
                    label_col = c
                    break

            if label_col is not None and DROP_HEADER_LEAK_ROWS:
                labels_norm = chunk[label_col].astype(str).map(normalize_colname)
                valid_mask = (labels_norm != 'label') & (labels_norm != '') & (labels_norm != 'nan')
                dropped_header_rows_file += int((~valid_mask).sum())
                chunk = chunk[valid_mask]

            if len(chunk) > 0:
                chunks_data.append(chunk)

        if len(chunks_data) == 0:
            msg = "no valid rows after header cleaning"
            failed_files.append((fname, msg))
            print(f"SKIPPED ({msg})")
            continue

        df_cse_file = pd.concat(chunks_data, ignore_index=True)

        X_cse_f, y_cse_f, atk_cse_f, sid_cse_f, ts_cse_f, meta_f = preprocess_eval_dataset(
            df_cse_file,
            selected_features,
            median_values,
            scaler,
            return_meta=True,
            dataset_name=f"CSE-{fname}"
        )

        meta_f['file'] = fname
        meta_f['rows'] = int(len(df_cse_file))
        meta_f['dropped_header_rows_file'] = dropped_header_rows_file
        cse_file_meta.append(meta_f)

        all_X_cse.append(X_cse_f)
        all_y_cse.append(y_cse_f)
        all_attacks_cse.append(atk_cse_f)
        all_sids_cse.append(sid_cse_f)
        all_ts_cse.append(ts_cse_f)
        print(
            f"{len(df_cse_file):,} rows "
            f"(benign: {(y_cse_f==0).sum():,}, attack: {(y_cse_f==1).sum():,}, "
            f"drop_header: {dropped_header_rows_file})"
        )

        del df_cse_file, chunks_data
        gc.collect()
    except Exception as e:
        failed_files.append((fname, str(e)))
        print(f"ERROR: {e}")
        continue

if len(all_X_cse) == 0:
    details = "\n".join([f"  - {f}: {err}" for f, err in failed_files])
    raise RuntimeError(
        "All CSE files failed preprocessing.\n"
        "Likely causes: feature alias map incomplete or strict feature match too high.\n"
        f"STRICT_FEATURE_MATCH={STRICT_FEATURE_MATCH}, MIN_FEATURE_MATCH_RATIO={MIN_FEATURE_MATCH_RATIO}.\n"
        "Failed files:\n"
        f"{details}\n"
        "Action: update FEATURE_ALIAS_MAP in setup cell, then rerun cell 4 -> 13.1."
    )

X_cse = np.vstack(all_X_cse)
y_cse = np.concatenate(all_y_cse)
attacks_cse = np.concatenate(all_attacks_cse)
sids_cse = np.concatenate(all_sids_cse)
ts_cse = np.concatenate(all_ts_cse)

# Aggregate metadata
total_dropped_header_rows = int(sum(m['dropped_header_rows_file'] for m in cse_file_meta))
min_match_ratio = float(min(m['match_ratio'] for m in cse_file_meta)) if cse_file_meta else 0.0
max_missing_features = int(max(m['missing_feature_count'] for m in cse_file_meta)) if cse_file_meta else 0
min_timestamp_valid_ratio_eval = float(min(m.get('timestamp_valid_ratio', 0.0) for m in cse_file_meta)) if cse_file_meta else 0.0
max_largest_session_share_eval = float(max(m.get('largest_session_share_eval', 0.0) for m in cse_file_meta)) if cse_file_meta else 0.0
min_unique_sessions_eval = int(min(m.get('unique_sessions_eval', 0) for m in cse_file_meta)) if cse_file_meta else 0

cse_eval_meta = {
    'total_dropped_header_rows': total_dropped_header_rows,
    'min_match_ratio': min_match_ratio,
    'max_missing_features': max_missing_features,
    'min_timestamp_valid_ratio_eval': min_timestamp_valid_ratio_eval,
    'max_largest_session_share_eval': max_largest_session_share_eval,
    'min_unique_sessions_eval': min_unique_sessions_eval,
    'file_meta': cse_file_meta,
    'failed_files': failed_files,
}

print(f"\nCSE-2018 total: {len(X_cse):,} rows")
print(f"Benign: {(y_cse == 0).sum():,}, Attack: {(y_cse == 1).sum():,}")
print(f"Total dropped header-leak rows: {total_dropped_header_rows:,}")
print(f"Min feature match ratio across files: {min_match_ratio:.2%}")
print(f"Max missing features across files: {max_missing_features}")
print(f"Min timestamp valid ratio (eval): {min_timestamp_valid_ratio_eval:.2%}")
print(f"Max largest-session share (eval): {max_largest_session_share_eval:.2%}")
print(f"Min unique sessions (eval files): {min_unique_sessions_eval:,}")
if failed_files:
    print("\nFailed files summary:")
    for f, err in failed_files:
        print(f"  - {f}: {err}")

print(f"Attack types: {len(np.unique(attacks_cse[y_cse == 1]))}:")
for atype in np.unique(attacks_cse):
    n = (attacks_cse == atype).sum()
    print(f"  {atype}: {n:,}")

del all_X_cse, all_y_cse, all_attacks_cse, all_sids_cse, all_ts_cse
gc.collect()

=== Evaluating CSE-CIC-IDS-2018 (All Labels) ===
  Processing 02-14-2018.csv... [CSE-02-14-2018.csv] Feature mapping coverage: 50/50 (100.00%)
1,048,575 rows (benign: 667,626, attack: 380,949, drop_header: 0)
  Processing 02-15-2018.csv... [CSE-02-15-2018.csv] Feature mapping coverage: 50/50 (100.00%)
1,048,575 rows (benign: 996,077, attack: 52,498, drop_header: 0)
  Processing 02-16-2018.csv... [CSE-02-16-2018.csv] Feature mapping coverage: 50/50 (100.00%)
1,048,574 rows (benign: 446,772, attack: 601,802, drop_header: 1)

CSE-2018 total: 3,145,724 rows
Benign: 2,110,475, Attack: 1,035,249
Total dropped header-leak rows: 1
Min feature match ratio across files: 100.00%
Max missing features across files: 0
Min timestamp valid ratio (eval): 100.00%
Max largest-session share (eval): 0.00%
Min unique sessions (eval files): 20,972
Attack types: 6:
  Benign: 2,110,475
  DoS attacks-GoldenEye: 41,508
  DoS attacks-Hulk: 461,912
  DoS attacks-SlowHTTPTest: 139,890
  DoS attacks-Slowloris: 10,99

0

In [31]:

# ============================================================
# 13.2 Compute Scores & Metrics for CSE-2018 (Part 4)
# ============================================================
print("=== CSE Pre-Scoring Validation ===")
if 'cse_eval_meta' not in globals():
    raise RuntimeError("cse_eval_meta not found. Run Section 13.1 first.")

with open(f"{SCORE_ARTIFACTS_DIR}/score_calibrator.json", 'r', encoding='utf-8') as f:
    score_calibrator = json.load(f)

labels_norm_unique = {normalize_colname(x) for x in np.unique(attacks_cse.astype(str))}
assert 'label' not in labels_norm_unique, (
    "Invalid class 'Label' detected after preprocessing. Check header leakage cleaning."
)

if STRICT_FEATURE_MATCH:
    assert cse_eval_meta['max_missing_features'] == 0, (
        f"Missing features still found (max_missing_features={cse_eval_meta['max_missing_features']}). "
        "Fix mapping before scoring."
    )

assert cse_eval_meta['min_match_ratio'] >= MIN_FEATURE_MATCH_RATIO, (
    f"Feature match ratio below threshold: {cse_eval_meta['min_match_ratio']:.2%} "
    f"< {MIN_FEATURE_MATCH_RATIO:.0%}"
)

if STRICT_SESSION_QUALITY:
    assert cse_eval_meta['min_timestamp_valid_ratio_eval'] >= MIN_TIMESTAMP_VALID_RATIO, (
        f"CSE timestamp valid ratio too low: {cse_eval_meta['min_timestamp_valid_ratio_eval']:.2%} "
        f"< {MIN_TIMESTAMP_VALID_RATIO:.0%}"
    )
    assert cse_eval_meta['max_largest_session_share_eval'] <= MAX_SINGLE_SESSION_SHARE, (
        f"CSE largest session share too high: {cse_eval_meta['max_largest_session_share_eval']:.2%} "
        f"> {MAX_SINGLE_SESSION_SHARE:.2%}"
    )

print(f"Rows total         : {len(X_cse):,}")
print(f"Benign / Attack    : {(y_cse == 0).sum():,} / {(y_cse == 1).sum():,}")
print(f"Dropped header rows: {cse_eval_meta['total_dropped_header_rows']:,}")
print(f"Min match ratio    : {cse_eval_meta['min_match_ratio']:.2%}")
print(f"Min ts valid ratio : {cse_eval_meta['min_timestamp_valid_ratio_eval']:.2%}")
print(f"Max session share  : {cse_eval_meta['max_largest_session_share_eval']:.2%}")
print("\nTop 10 labels:")
print(pd.Series(attacks_cse).value_counts().head(10).to_string())
print("\n[OK] Pre-scoring validation passed.")

scores_cse, err_s1_cse, err_s2_cse = compute_eval_scores(
    X_cse, sids_cse, ts_cse,
    model_s1, encoder_s1, model_s2,
    window_size=WINDOW_SIZE, min_flows=MIN_FLOWS_PER_SESSION,
    score_calibrator=score_calibrator,
)
print(f"Scores computed: mean={scores_cse.mean():.4f}, std={scores_cse.std():.4f}")

metrics_cse, roc_auc_cse, pr_auc_cse = evaluate_with_thresholds(
    y_cse, scores_cse, threshold_results, 'CSE-2018'
)

diag_cse = direction_diagnostic(y_cse, scores_cse)
print(f"ROC-AUC raw     : {diag_cse['auc_raw']:.4f}")
print(f"ROC-AUC inverted: {diag_cse['auc_inverted']:.4f}")
print(f"Direction issue : {diag_cse['direction_issue']}")

score_direction_report = {
    'score_pipeline_version': SCORE_PIPELINE_VERSION,
    'mode': score_calibrator.get('mode', SCORE_MODE),
    'invert_stage2': bool(score_calibrator.get('invert_stage2', INVERT_STAGE2_SCORE)),
    'w1': float(score_calibrator['w1']),
    'w2': float(score_calibrator['w2']),
    'roc_auc_raw': float(diag_cse['auc_raw']),
    'roc_auc_inverted': float(diag_cse['auc_inverted']),
    'delta': float(diag_cse['delta']),
    'direction_issue': bool(diag_cse['direction_issue']),
}
with open(f"{SCORE_ARTIFACTS_DIR}/score_direction_report_v4.json", 'w', encoding='utf-8') as f:
    json.dump(score_direction_report, f, indent=2)

best_m_cse = metrics_cse[best_threshold_name]
print(f"\n=== CSE-2018 Results (best threshold: {best_threshold_name}) ===")
print(f"ROC-AUC   : {roc_auc_cse:.4f}")
print(f"PR-AUC    : {pr_auc_cse:.4f}")
print(f"Precision : {best_m_cse['precision']:.4f}")
print(f"Recall    : {best_m_cse['recall']:.4f}")
print(f"F1-Score  : {best_m_cse['f1']:.4f}")
print(f"FPR       : {best_m_cse['fpr']:.4f}")

cse_payload = {
    'n_rows': int(len(y_cse)),
    'roc_auc': float(roc_auc_cse),
    'pr_auc': float(pr_auc_cse),
    'best_threshold_name': best_threshold_name,
    'best_threshold_metrics': best_m_cse,
    'direction_diagnostic': diag_cse,
}
with open(f"{SCORE_ARTIFACTS_DIR}/cse_metrics.json", 'w', encoding='utf-8') as f:
    json.dump(cse_payload, f, indent=2)

print(f"Saved: {SCORE_ARTIFACTS_DIR}/score_direction_report_v4.json")
print(f"Saved: {SCORE_ARTIFACTS_DIR}/cse_metrics.json")




=== CSE Pre-Scoring Validation ===
Rows total         : 3,145,724
Benign / Attack    : 2,110,475 / 1,035,249
Dropped header rows: 1
Min match ratio    : 100.00%
Min ts valid ratio : 100.00%
Max session share  : 0.00%

Top 10 labels:
Benign                      2110475
DoS attacks-Hulk             461912
FTP-BruteForce               193360
SSH-Bruteforce               187589
DoS attacks-SlowHTTPTest     139890
DoS attacks-GoldenEye         41508
DoS attacks-Slowloris         10990

[OK] Pre-scoring validation passed.
Scores computed: mean=3.1122, std=2.1388
ROC-AUC raw     : 0.2770
ROC-AUC inverted: 0.7230
Direction issue : True

=== CSE-2018 Results (best threshold: zscore_k1.5) ===
ROC-AUC   : 0.2770
PR-AUC    : 0.2288
Precision : 0.2529
Recall    : 0.4545
F1-Score  : 0.3250
FPR       : 0.6586
Saved: /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/artifacts/v5_domainfix/score_direction_report_v4.json
Saved: /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/artifacts/v5_domainfix/cse_metrics

In [32]:
# ============================================================
# 13.3 CSE-2018 Per-Attack Detection Rate
# ============================================================
y_pred_cse = (scores_cse > best_t).astype(int)

print("=== CSE-2018: Per-Attack Detection Rate ===")
attack_df_cse = pd.DataFrame({'attack': attacks_cse, 'y_true': y_cse, 'y_pred': y_pred_cse})

det_rates_cse = {}
for atype, grp in attack_df_cse.groupby('attack'):
    if atype.strip().upper() == 'BENIGN':
        continue
    dr = grp['y_pred'].mean()
    det_rates_cse[atype] = {'detection_rate': dr, 'count': len(grp)}
    status = '✓' if dr > 0.7 else ('⚠' if dr > 0.3 else '✗')
    print(f"  {status} {atype:<30} DR={dr:.4f}  (n={len(grp):,})")

# Benign FPR
benign_grp_cse = attack_df_cse[attack_df_cse['attack'].str.strip().str.upper() == 'BENIGN']
benign_fpr_cse = benign_grp_cse['y_pred'].mean() if len(benign_grp_cse) > 0 else float('nan')
print(f"\n  Benign FPR: {benign_fpr_cse:.4f}")

=== CSE-2018: Per-Attack Detection Rate ===
  ⚠ DoS attacks-GoldenEye          DR=0.4146  (n=41,508)
  ✗ DoS attacks-Hulk               DR=0.0400  (n=461,912)
  ✓ DoS attacks-SlowHTTPTest       DR=0.9999  (n=139,890)
  ✓ DoS attacks-Slowloris          DR=0.7105  (n=10,990)
  ✓ FTP-BruteForce                 DR=0.9999  (n=193,360)
  ⚠ SSH-Bruteforce                 DR=0.5003  (n=187,589)

  Benign FPR: 0.6586


In [33]:
# ============================================================
# 13.4 CSE-2018 Visualizations
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Error distribution by class
benign_mask_cse = pd.Series(attacks_cse).str.strip().str.upper() == 'BENIGN'
unique_attacks_cse = [a for a in np.unique(attacks_cse)
                       if a.strip().upper() != 'BENIGN']

plot_data_cse = [scores_cse[benign_mask_cse.values]]
plot_labels_cse = ['BENIGN']
for a in sorted(unique_attacks_cse)[:8]:
    mask = attacks_cse == a
    if mask.sum() > 10:
        plot_data_cse.append(scores_cse[mask])
        plot_labels_cse.append(a[:20])

bp = axes[0, 0].boxplot(plot_data_cse, labels=plot_labels_cse, vert=True, patch_artist=True)
for i, box in enumerate(bp['boxes']):
    box.set_facecolor('lightblue' if i == 0 else 'salmon')
axes[0, 0].axhline(y=best_t, color='red', linestyle='--', label='Threshold')
axes[0, 0].set_title('CSE-2018: Error Distribution by Class')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].set_ylabel('Anomaly Score')
axes[0, 0].legend()

# 2. ROC Curve (both datasets)
fpr_cse_curve, tpr_cse_curve, _ = roc_curve(y_cse, scores_cse)
fpr_cic_curve, tpr_cic_curve, _ = roc_curve(y_cic, scores_cic)
axes[0, 1].plot(fpr_cic_curve, tpr_cic_curve, 'b-', label=f'CIC-2017 (AUC={roc_auc_cic:.4f})', alpha=0.7)
axes[0, 1].plot(fpr_cse_curve, tpr_cse_curve, 'r-', label=f'CSE-2018 (AUC={roc_auc_cse:.4f})', alpha=0.7)
axes[0, 1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].set_title('ROC Curves — CIC vs CSE')
axes[0, 1].legend()

# 3. PR Curve (both datasets)
prec_cse, rec_cse, _ = precision_recall_curve(y_cse, scores_cse)
prec_cic_c, rec_cic_c, _ = precision_recall_curve(y_cic, scores_cic)
axes[1, 0].plot(rec_cic_c, prec_cic_c, 'b-', label=f'CIC-2017 (AP={pr_auc_cic:.4f})', alpha=0.7)
axes[1, 0].plot(rec_cse, prec_cse, 'r-', label=f'CSE-2018 (AP={pr_auc_cse:.4f})', alpha=0.7)
axes[1, 0].set_xlabel('Recall')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_title('Precision-Recall Curves — CIC vs CSE')
axes[1, 0].legend()

# 4. Confusion Matrix CSE
cm_cse = confusion_matrix(y_cse, y_pred_cse, labels=[0, 1])
sns.heatmap(cm_cse, annot=True, fmt='d', cmap='Reds', ax=axes[1, 1],
            xticklabels=['Benign', 'Attack'], yticklabels=['Benign', 'Attack'])
axes[1, 1].set_xlabel('Predicted')
axes[1, 1].set_ylabel('Actual')
axes[1, 1].set_title(f'CSE-2018 Confusion Matrix ({best_threshold_name})')

plt.tight_layout()
plt.savefig(f"{SCORE_RESULTS_DIR}/cse2018_evaluation.png", dpi=150, bbox_inches='tight')
plt.show()

# Detection rate bar chart
fig2, ax2 = plt.subplots(figsize=(12, 5))
dr_names_cse = list(det_rates_cse.keys())
dr_vals_cse = [det_rates_cse[k]['detection_rate'] for k in dr_names_cse]
colors_dr_cse = ['#27ae60' if v > 0.7 else '#f39c12' if v > 0.3 else '#e74c3c' for v in dr_vals_cse]
ax2.barh(dr_names_cse, dr_vals_cse, color=colors_dr_cse)
ax2.axvline(x=0.7, color='green', linestyle='--', alpha=0.5, label='Good (>0.7)')
ax2.axvline(x=0.3, color='red', linestyle='--', alpha=0.5, label='Poor (<0.3)')
ax2.set_xlabel('Detection Rate')
ax2.set_title('CSE-2018: Detection Rate per Attack Category')
ax2.legend()
plt.tight_layout()
plt.savefig(f"{SCORE_RESULTS_DIR}/cse2018_detection_rates.png", dpi=150, bbox_inches='tight')
plt.show()

# Save CSE results
pd.DataFrame(metrics_cse).T.to_csv(f"{SCORE_RESULTS_DIR}/cse2018_metrics.csv")
print("CSE-2018 evaluation complete. Results saved.")

# Part 4 summary report
part4_summary_lines = [
    '# Part 4 Scoring Fix Summary',
    f'Date: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    '',
    '## Configuration',
    f'- SCORE_PIPELINE_VERSION: {SCORE_PIPELINE_VERSION}',
    f'- SOURCE_PIPELINE_VERSION: {SOURCE_PIPELINE_VERSION}',
    f'- SCORE_MODE: {score_calibrator.get("mode", SCORE_MODE)}',
    f'- INVERT_STAGE2_SCORE: {score_calibrator.get("invert_stage2", INVERT_STAGE2_SCORE)}',
    f'- Weights: w1={score_calibrator["w1"]:.2f}, w2={score_calibrator["w2"]:.2f}',
    '',
    '## CIC Holdout',
    f'- ROC-AUC: {roc_auc_cic:.4f}',
    f'- PR-AUC: {pr_auc_cic:.4f}',
    f'- F1: {best_m["f1"]:.4f}',
    f'- FPR: {best_m["fpr"]:.4f}',
    '',
    '## CSE',
    f'- ROC-AUC: {roc_auc_cse:.4f}',
    f'- PR-AUC: {pr_auc_cse:.4f}',
    f'- F1: {best_m_cse["f1"]:.4f}',
    f'- FPR: {best_m_cse["fpr"]:.4f}',
    '',
    '## Direction Diagnostics',
    f'- CIC direction_issue: {diag_cic["direction_issue"]}',
    f'- CSE direction_issue: {diag_cse["direction_issue"]}',
    '',
    '## Gates (Pragmatic)',
    f'- G1 CSE ROC-AUC > 0.5: {roc_auc_cse > 0.5}',
    f'- G2 direction_issue=False (CSE): {not diag_cse["direction_issue"]}',
    f'- G3 CIC ROC-AUC >= 0.592 (baseline 0.622 - 0.03): {roc_auc_cic >= 0.592}',
    f'- G4 CSE FPR < 0.20: {best_m_cse["fpr"] < 0.20}',
]
part4_summary_path = f"{SCORE_RESULTS_DIR}/part4_summary.md"
with open(part4_summary_path, 'w', encoding='utf-8') as f:
    f.write('\\n'.join(part4_summary_lines))
print(f"Part 4 summary saved: {part4_summary_path}")




/tmp/ipython-input-1743/1635562554.py:19: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = axes[0, 0].boxplot(plot_data_cse, labels=plot_labels_cse, vert=True, patch_artist=True)


CSE-2018 evaluation complete. Results saved.
Part 4 summary saved: /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/results/v5_domainfix/part4_summary.md


## 13.5 PART 5 ? DOMAIN SHIFT MITIGATION + DOMAIN THRESHOLD CALIBRATION

Part 5 menjalankan DoS Hulk deep dive, strategi mitigasi high-shift, kalibrasi threshold per-domain, dan evaluasi 4 kombinasi.


In [34]:

# ============================================================
# 13.5 Part 5 Execution (v5_domainfix)
# ============================================================
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, confusion_matrix

if RUN_PART5_DOMAINFIX:
    # -------- bootstrap --------
    with open(f"{SOURCE_SCORING_ARTIFACTS_DIR}/score_calibrator.json", 'r', encoding='utf-8') as f:
        score_calibrator_v4 = json.load(f)
    with open(f"{SOURCE_SCORING_ARTIFACTS_DIR}/score_direction_report_v4.json", 'r', encoding='utf-8') as f:
        score_direction_v4 = json.load(f)

    run_manifest = {
        'pipeline_version': PIPELINE_VERSION,
        'source_preprocess_version': SOURCE_PREPROCESS_VERSION,
        'source_scoring_version': SOURCE_SCORING_VERSION,
        'source_preprocessed_dir': SOURCE_PREPROCESSED_DIR,
        'source_scoring_artifacts_dir': SOURCE_SCORING_ARTIFACTS_DIR,
        'v5_artifacts_dir': SCORE_ARTIFACTS_DIR,
        'v5_results_dir': SCORE_RESULTS_DIR,
        'high_shift_features': HIGH_SHIFT_FEATURES,
        'part5_strategies': PART5_STRATEGIES,
        'mask_anchor': MASK_ANCHOR,
        'target_threshold_domains': TARGET_THRESHOLD_DOMAINS,
        'target_threshold_method': TARGET_THRESHOLD_METHOD,
    }
    with open(f"{SCORE_ARTIFACTS_DIR}/v5_run_manifest.json", 'w', encoding='utf-8') as f:
        json.dump(run_manifest, f, indent=2)

    # -------- helpers --------
    feature_norm_map = {normalize_colname(f): i for i, f in enumerate(selected_features)}

    def _resolve_feature_index(name):
        n = normalize_colname(name)
        if n in feature_norm_map:
            return feature_norm_map[n]
        nn = n.replace('_', ' ')
        if nn in feature_norm_map:
            return feature_norm_map[nn]
        return None

    high_shift_indices = []
    high_shift_used = []
    for f in HIGH_SHIFT_FEATURES:
        idx = _resolve_feature_index(f)
        if idx is not None:
            high_shift_indices.append(idx)
            high_shift_used.append(selected_features[idx])

    # CIC-based anchor/bounds from benign train (scaled domain)
    cic_median_scaled = np.median(X_train, axis=0)
    cic_p1_scaled = np.percentile(X_train, 1, axis=0)
    cic_p99_scaled = np.percentile(X_train, 99, axis=0)

    profile = {
        'high_shift_indices': [int(i) for i in high_shift_indices],
        'high_shift_features_used': high_shift_used,
        'mask_anchor': MASK_ANCHOR,
        'cic_median_scaled': {selected_features[i]: float(cic_median_scaled[i]) for i in high_shift_indices},
        'cic_p1_scaled': {selected_features[i]: float(cic_p1_scaled[i]) for i in high_shift_indices},
        'cic_p99_scaled': {selected_features[i]: float(cic_p99_scaled[i]) for i in high_shift_indices},
    }
    with open(f"{SCORE_ARTIFACTS_DIR}/high_shift_feature_profile.json", 'w', encoding='utf-8') as f:
        json.dump(profile, f, indent=2)

    def apply_high_shift_mask_median(X):
        Xo = X.copy()
        for i in high_shift_indices:
            Xo[:, i] = cic_median_scaled[i]
        return Xo

    def apply_high_shift_clip_bounds(X):
        Xo = X.copy()
        for i in high_shift_indices:
            Xo[:, i] = np.clip(Xo[:, i], cic_p1_scaled[i], cic_p99_scaled[i])
        return Xo

    # score profile lock from v4
    delta = float(score_direction_v4.get('delta', 0.0))
    direction_issue_v4 = bool(score_direction_v4.get('direction_issue', False))
    if ENABLE_DOMAIN_SPECIFIC_SCORE_PROFILE and CSE_SCORE_SIGN == 'auto_from_part4' and direction_issue_v4 and delta >= 0.10:
        cse_score_sign = -1.0
    else:
        cse_score_sign = 1.0

    score_profile_v5 = {
        'base_w1': float(score_calibrator_v4['w1']),
        'base_w2': float(score_calibrator_v4['w2']),
        'invert_stage2': bool(score_calibrator_v4.get('invert_stage2', True)),
        'score_mode': score_calibrator_v4.get('mode', 'combined_normalized'),
        'cse_score_sign': float(cse_score_sign),
        'source_direction_issue_v4': direction_issue_v4,
        'source_delta_v4': delta,
    }
    with open(f"{SCORE_ARTIFACTS_DIR}/score_profile_v5.json", 'w', encoding='utf-8') as f:
        json.dump(score_profile_v5, f, indent=2)

    def compute_scores_with_profile(X_scaled, sids, ts, domain='cic', strategy='none'):
        if strategy == 'mask_high_shift':
            X_use = apply_high_shift_mask_median(X_scaled)
        elif strategy == 'clip_high_shift':
            X_use = apply_high_shift_clip_bounds(X_scaled)
        else:
            X_use = X_scaled

        scores, err_s1, err_s2 = compute_eval_scores(
            X_use, sids, ts,
            model_s1, encoder_s1, model_s2,
            window_size=WINDOW_SIZE, min_flows=MIN_FLOWS_PER_SESSION,
            score_calibrator=score_calibrator_v4,
        )

        if domain == 'cse':
            scores = score_profile_v5['cse_score_sign'] * scores

        return scores, err_s1, err_s2, X_use

    def build_domain_thresholds(benign_scores):
        th = {}
        mu, sigma = float(np.mean(benign_scores)), float(np.std(benign_scores) + 1e-12)
        for k in K_SIGMA_VALUES:
            t = mu + k * sigma
            th[f'zscore_k{k}'] = {'threshold': float(t), 'fpr': float((benign_scores > t).mean())}
        for p in PERCENTILE_VALUES:
            t = float(np.percentile(benign_scores, p))
            th[f'percentile_{p}'] = {'threshold': t, 'fpr': float((benign_scores > t).mean())}

        if TARGET_THRESHOLD_METHOD in th:
            best_name = TARGET_THRESHOLD_METHOD
        elif 'percentile_95' in th:
            best_name = 'percentile_95'
        else:
            best_name = sorted(th.keys())[0]
        return th, best_name, float(th[best_name]['threshold'])

    def metrics_at_threshold(y_true, scores, threshold):
        y_pred = (scores > threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        return {
            'precision': float(precision_score(y_true, y_pred, zero_division=0)),
            'recall': float(recall_score(y_true, y_pred, zero_division=0)),
            'f1': float(f1_score(y_true, y_pred, zero_division=0)),
            'fpr': float(fpr),
            'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn),
        }

    # -------- DoS Hulk deep dive --------
    hulk_mask = np.array([normalize_colname(a) == normalize_colname('DoS attacks-Hulk') for a in attacks_cse])
    benign_mask = (y_cse == 0)

    rng = np.random.default_rng(RANDOM_SEED)
    benign_idx = np.where(benign_mask)[0]
    hulk_idx = np.where(hulk_mask)[0]
    n_b = min(DOS_HULK_SAMPLE_SIZE, len(benign_idx))
    n_h = min(DOS_HULK_SAMPLE_SIZE, len(hulk_idx))
    sel_b = rng.choice(benign_idx, size=n_b, replace=False)
    sel_h = rng.choice(hulk_idx, size=n_h, replace=False)

    scores_b, err_s1_b, err_s2_b, Xb_used = compute_scores_with_profile(X_cse[sel_b], sids_cse[sel_b], ts_cse[sel_b], domain='cse', strategy='none')
    scores_h, err_s1_h, err_s2_h, Xh_used = compute_scores_with_profile(X_cse[sel_h], sids_cse[sel_h], ts_cse[sel_h], domain='cse', strategy='none')

    mean_b, std_b = float(np.mean(err_s1_b)), float(np.std(err_s1_b) + 1e-12)
    mean_h = float(np.mean(err_s1_h))

    if mean_h < mean_b:
        hulk_outcome = 'known_limitation_hulk_looks_normal'
    elif abs(mean_h - mean_b) <= 0.10 * std_b:
        hulk_outcome = 'fundamental_indistinguishable'
    else:
        hulk_outcome = 'threshold_transfer_failure'

    residual_b = np.abs(Xb_used - X_cse[sel_b])
    residual_h = np.abs(Xh_used - X_cse[sel_h])
    residual_diff = np.mean(residual_h, axis=0) - np.mean(residual_b, axis=0)
    top_idx = np.argsort(-np.abs(residual_diff))[:20]
    residual_df = pd.DataFrame({
        'feature': [selected_features[i] for i in top_idx],
        'residual_diff_hulk_minus_benign': residual_diff[top_idx],
    })
    residual_df.to_csv(f"{SCORE_ARTIFACTS_DIR}/dos_hulk_residual_features.csv", index=False)

    plt.figure(figsize=(10,4))
    sns.kdeplot(err_s1_b, label='Benign S1 error', fill=True)
    sns.kdeplot(err_s1_h, label='Hulk S1 error', fill=True)
    plt.title('DoS Hulk vs Benign - Stage1 Reconstruction Error')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{SCORE_RESULTS_DIR}/dos_hulk_error_distributions.png", dpi=150)
    plt.close()

    dos_hulk_report = {
        'n_benign_sample': int(n_b),
        'n_hulk_sample': int(n_h),
        'benign_s1_mean': mean_b,
        'benign_s1_std': float(np.std(err_s1_b)),
        'hulk_s1_mean': mean_h,
        'hulk_s1_std': float(np.std(err_s1_h)),
        'benign_s2_mean': float(np.mean(err_s2_b)),
        'hulk_s2_mean': float(np.mean(err_s2_h)),
        'benign_score_mean': float(np.mean(scores_b)),
        'hulk_score_mean': float(np.mean(scores_h)),
        'decision_outcome': hulk_outcome,
    }
    with open(f"{SCORE_ARTIFACTS_DIR}/dos_hulk_deep_dive.json", 'w', encoding='utf-8') as f:
        json.dump(dos_hulk_report, f, indent=2)

    # -------- thresholds per domain --------
    # CIC benign (holdout subset)
    cic_benign_holdout = np.where(y_cic == 0)[0]
    cic_scores_benign, _, _, _ = compute_scores_with_profile(X_cic[cic_benign_holdout], sids_cic[cic_benign_holdout], ts_cic[cic_benign_holdout], domain='cic', strategy='none')

    # CSE benign sample
    cse_benign_idx = np.where(y_cse == 0)[0]
    n_cse_b = min(CSE_BENIGN_THRESHOLD_SAMPLE, len(cse_benign_idx))
    cse_b_sel = rng.choice(cse_benign_idx, size=n_cse_b, replace=False)
    cse_scores_benign, _, _, _ = compute_scores_with_profile(X_cse[cse_b_sel], sids_cse[cse_b_sel], ts_cse[cse_b_sel], domain='cse', strategy='none')

    th_cic, th_cic_name, th_cic_val = build_domain_thresholds(cic_scores_benign)
    th_cse, th_cse_name, th_cse_val = build_domain_thresholds(cse_scores_benign)

    with open(f"{SCORE_ARTIFACTS_DIR}/thresholds_cic_benign_v5.json", 'w', encoding='utf-8') as f:
        json.dump({'thresholds': th_cic, 'selected': th_cic_name, 'value': th_cic_val}, f, indent=2)
    with open(f"{SCORE_ARTIFACTS_DIR}/thresholds_cse_benign_v5.json", 'w', encoding='utf-8') as f:
        json.dump({'thresholds': th_cse, 'selected': th_cse_name, 'value': th_cse_val}, f, indent=2)

    drift_report = {
        'cic_benign_score_mean': float(np.mean(cic_scores_benign)),
        'cic_benign_score_std': float(np.std(cic_scores_benign)),
        'cse_benign_score_mean': float(np.mean(cse_scores_benign)),
        'cse_benign_score_std': float(np.std(cse_scores_benign)),
        'normalized_mean_shift': float((np.mean(cse_scores_benign)-np.mean(cic_scores_benign)) / (np.std(cic_scores_benign)+1e-12)),
    }
    with open(f"{SCORE_ARTIFACTS_DIR}/threshold_transfer_drift_report.json", 'w', encoding='utf-8') as f:
        json.dump(drift_report, f, indent=2)

    # -------- strategy preview --------
    preview_rows = []
    preview_n = min(200, len(X_cse))
    X_preview = X_cse[:preview_n]
    X_mask = apply_high_shift_mask_median(X_preview)
    X_clip = apply_high_shift_clip_bounds(X_preview)
    for i in high_shift_indices[:10]:
        preview_rows.append({
            'feature': selected_features[i],
            'orig_mean': float(np.mean(X_preview[:, i])),
            'mask_mean': float(np.mean(X_mask[:, i])),
            'clip_mean': float(np.mean(X_clip[:, i])),
        })
    pd.DataFrame(preview_rows).to_csv(f"{SCORE_ARTIFACTS_DIR}/strategy_inputs_preview.csv", index=False)

    # -------- evaluation matrix 2x2 --------
    combos = [
        ('mask_high_shift', 'cic_benign_threshold'),
        ('mask_high_shift', 'cse_benign_threshold'),
        ('clip_high_shift', 'cic_benign_threshold'),
        ('clip_high_shift', 'cse_benign_threshold'),
    ]

    eval_rows = []
    for strategy, th_source in combos:
        # strategy applied to CSE; CIC left untouched for compatibility guard
        scores_cic_s, _, _, _ = compute_scores_with_profile(X_cic, sids_cic, ts_cic, domain='cic', strategy='none')
        scores_cse_s, _, _, _ = compute_scores_with_profile(X_cse, sids_cse, ts_cse, domain='cse', strategy=strategy)

        if th_source == 'cse_benign_threshold':
            threshold = th_cse_val
        else:
            threshold = th_cic_val

        m_cic = metrics_at_threshold(y_cic, scores_cic_s, threshold)
        m_cse = metrics_at_threshold(y_cse, scores_cse_s, threshold)

        auc_cic_raw = float(roc_auc_score(y_cic, scores_cic_s))
        auc_cic_inv = float(roc_auc_score(y_cic, -scores_cic_s))
        auc_cse_raw = float(roc_auc_score(y_cse, scores_cse_s))
        auc_cse_inv = float(roc_auc_score(y_cse, -scores_cse_s))
        pr_cic = float(average_precision_score(y_cic, scores_cic_s))
        pr_cse = float(average_precision_score(y_cse, scores_cse_s))

        hulk_subset = (np.array([normalize_colname(a) == normalize_colname('DoS attacks-Hulk') for a in attacks_cse]))
        if hulk_subset.any():
            hulk_dr = float((scores_cse_s[hulk_subset] > threshold).mean())
        else:
            hulk_dr = float('nan')

        eval_rows.append({
            'strategy': strategy,
            'threshold_source': th_source,
            'threshold_value': float(threshold),
            'cic_roc_auc_raw': auc_cic_raw,
            'cic_roc_auc_inverted': auc_cic_inv,
            'cic_pr_auc': pr_cic,
            'cic_f1': m_cic['f1'],
            'cic_fpr': m_cic['fpr'],
            'cse_roc_auc_raw': auc_cse_raw,
            'cse_roc_auc_inverted': auc_cse_inv,
            'cse_pr_auc': pr_cse,
            'cse_f1': m_cse['f1'],
            'cse_fpr': m_cse['fpr'],
            'dos_hulk_dr': hulk_dr,
            'direction_issue_cse': bool(auc_cse_raw < auc_cse_inv),
        })

    eval_df = pd.DataFrame(eval_rows)
    eval_df.to_csv(f"{SCORE_ARTIFACTS_DIR}/part5_eval_matrix.csv", index=False)

    # selection with CIC guard
    eval_df_guard = eval_df[eval_df['cic_roc_auc_raw'] >= 0.8333].copy()
    if len(eval_df_guard) == 0:
        eval_df_guard = eval_df.copy()

    eval_df_guard = eval_df_guard.sort_values(
        by=['cse_roc_auc_raw', 'dos_hulk_dr', 'cse_fpr'],
        ascending=[False, False, True]
    )
    best = eval_df_guard.iloc[0].to_dict()

    with open(f"{SCORE_ARTIFACTS_DIR}/part5_best_config.json", 'w', encoding='utf-8') as f:
        json.dump(best, f, indent=2)

    # gates
    gates = {
        'G1_cse_auc_inverted_ge_0_72': bool(best['cse_roc_auc_inverted'] >= 0.72),
        'G2_cse_auc_raw_gt_0_60': bool(best['cse_roc_auc_raw'] > 0.60),
        'G3_cse_fpr_lt_0_20': bool(best['cse_fpr'] < 0.20),
        'G4_dos_hulk_dr_gt_0_15': bool(best['dos_hulk_dr'] > 0.15),
        'G5_cic_auc_ge_0_8333': bool(best['cic_roc_auc_raw'] >= 0.8333),
    }

    if gates['G2_cse_auc_raw_gt_0_60']:
        outcome = 'advance_to_part6_or_phase3'
        recommendation = 'Part 5 gates satisfied for raw AUC target. Continue to Phase 3 improvements.'
    elif gates['G1_cse_auc_inverted_ge_0_72']:
        outcome = 'declare_calibration_limitation_continue_phase3_without_v5b_retrain'
        recommendation = 'Use inverted-score signal in Phase 3 (Isolation Forest); do not trigger v5b retrain.'
    else:
        outcome = 'trigger_v5b_retrain'
        recommendation = 'Both raw and inverted performance insufficient; retrain branch required.'

    gate_report = {
        'best_config': best,
        'gates': gates,
        'outcome': outcome,
        'recommendation': recommendation,
        'dos_hulk_outcome': hulk_outcome,
    }
    with open(f"{SCORE_ARTIFACTS_DIR}/part5_gate_report.json", 'w', encoding='utf-8') as f:
        json.dump(gate_report, f, indent=2)

    summary_lines = [
        '# Part 5 Domain Fix Summary',
        f'Date: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
        '',
        '## Best Configuration',
        f"- strategy: {best['strategy']}",
        f"- threshold_source: {best['threshold_source']}",
        f"- cse_roc_auc_raw: {best['cse_roc_auc_raw']:.4f}",
        f"- cse_roc_auc_inverted: {best['cse_roc_auc_inverted']:.4f}",
        f"- cse_fpr: {best['cse_fpr']:.4f}",
        f"- dos_hulk_dr: {best['dos_hulk_dr']:.4f}",
        '',
        '## Hulk Deep Dive',
        f"- outcome: {hulk_outcome}",
        f"- benign_s1_mean: {dos_hulk_report['benign_s1_mean']:.6f}",
        f"- hulk_s1_mean: {dos_hulk_report['hulk_s1_mean']:.6f}",
        '',
        '## Gates',
        f"- G1: {gates['G1_cse_auc_inverted_ge_0_72']}",
        f"- G2: {gates['G2_cse_auc_raw_gt_0_60']}",
        f"- G3: {gates['G3_cse_fpr_lt_0_20']}",
        f"- G4: {gates['G4_dos_hulk_dr_gt_0_15']}",
        f"- G5: {gates['G5_cic_auc_ge_0_8333']}",
        '',
        '## Outcome',
        f"- {outcome}",
        f"- {recommendation}",
    ]
    with open(f"{SCORE_RESULTS_DIR}/part5_summary.md", 'w', encoding='utf-8') as f:
        f.write('\n'.join(summary_lines))

    print('[Part5] Completed')
    print(f"Best strategy: {best['strategy']} + {best['threshold_source']}")
    print(f"CSE AUC raw/inv: {best['cse_roc_auc_raw']:.4f}/{best['cse_roc_auc_inverted']:.4f}")
    print(f"CSE FPR: {best['cse_fpr']:.4f} | Hulk DR: {best['dos_hulk_dr']:.4f}")
    print(f"Outcome: {outcome}")
    print(f"Saved: {SCORE_ARTIFACTS_DIR}/part5_gate_report.json")
    print(f"Saved: {SCORE_RESULTS_DIR}/part5_summary.md")
else:
    print('[INFO] RUN_PART5_DOMAINFIX=False. Skipping Part 5 execution.')



[Part5] Completed
Best strategy: clip_high_shift + cse_benign_threshold
CSE AUC raw/inv: 0.6223/0.3777
CSE FPR: 0.0003 | Hulk DR: 0.0062
Outcome: advance_to_part6_or_phase3
Saved: /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/artifacts/v5_domainfix/part5_gate_report.json
Saved: /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/results/v5_domainfix/part5_summary.md


## 14. GENERALIZATION ANALYSIS

Quantitative comparison of model performance across datasets.
Determines whether the model generalizes, partially generalizes, or is CIC-specific.

In [35]:
# ============================================================
# 14.1 Side-by-Side Comparison
# ============================================================
print("=" * 70)
print("GENERALIZATION ANALYSIS: CIC-2017 vs CSE-2018")
print("=" * 70)

best_cic = metrics_cic[best_threshold_name]
best_cse = metrics_cse[best_threshold_name]

comparison_rows = []
targets = {
    'ROC-AUC':   (roc_auc_cic, roc_auc_cse, 0.85, 0.75),
    'PR-AUC':    (pr_auc_cic, pr_auc_cse, 0.70, 0.60),
    'F1-Score':  (best_cic['f1'], best_cse['f1'], 0.65, 0.50),
    'Recall':    (best_cic['recall'], best_cse['recall'], 0.65, 0.50),
    'Precision': (best_cic['precision'], best_cse['precision'], 0.65, 0.50),
    'FPR':       (best_cic['fpr'], best_cse['fpr'], 0.05, 0.10),
}

print(f"\n{'Metric':<12} {'CIC-2017':>10} {'CSE-2018':>10} {'Gap':>8} {'Target CIC':>11} {'Target CSE':>11} {'Status':>8}")
print("-" * 75)

statuses = []
for metric, (cic_val, cse_val, target_cic, target_cse) in targets.items():
    gap = cse_val - cic_val
    if metric == 'FPR':
        cic_ok = cic_val <= target_cic
        cse_ok = cse_val <= target_cse
    else:
        cic_ok = cic_val >= target_cic
        cse_ok = cse_val >= target_cse

    status = '✓' if cse_ok else ('⚠' if abs(cse_val - target_cse) < 0.1 else '✗')
    statuses.append(status)

    print(f"{metric:<12} {cic_val:>10.4f} {cse_val:>10.4f} {gap:>+8.4f} "
          f"{'<' if metric == 'FPR' else '>'}{target_cic:>10.2f} "
          f"{'<' if metric == 'FPR' else '>'}{target_cse:>10.2f} {status:>8}")

    comparison_rows.append({
        'metric': metric, 'cic_2017': cic_val, 'cse_2018': cse_val,
        'gap': gap, 'target_cic': target_cic, 'target_cse': target_cse,
        'status': status
    })

pd.DataFrame(comparison_rows).to_csv(f"{RESULTS_DIR}/generalization_comparison.csv", index=False)

GENERALIZATION ANALYSIS: CIC-2017 vs CSE-2018

Metric         CIC-2017   CSE-2018      Gap  Target CIC  Target CSE   Status
---------------------------------------------------------------------------
ROC-AUC          0.8633     0.2770  -0.5863 >      0.85 >      0.75        ✗
PR-AUC           0.7356     0.2288  -0.5068 >      0.70 >      0.60        ✗
F1-Score         0.6902     0.3250  -0.3652 >      0.65 >      0.50        ✗
Recall           0.6303     0.4545  -0.1758 >      0.65 >      0.50        ⚠
Precision        0.7626     0.2529  -0.5097 >      0.65 >      0.50        ✗
FPR              0.0481     0.6586  +0.6104 <      0.05 <      0.10        ✗


In [36]:
# ============================================================
# 14.2 Attack-Level Analysis
# ============================================================
print("\n=== Attack-Level Analysis ===")

# High detection attacks
high_det_cic = {k: v for k, v in det_rates_cic.items() if v['detection_rate'] > 0.7}
low_det_cic = {k: v for k, v in det_rates_cic.items() if v['detection_rate'] < 0.3}
high_det_cse = {k: v for k, v in det_rates_cse.items() if v['detection_rate'] > 0.7}
low_det_cse = {k: v for k, v in det_rates_cse.items() if v['detection_rate'] < 0.3}

print(f"\nCIC-2017 — Successfully detected (DR > 0.7): {len(high_det_cic)} attack types")
for k, v in sorted(high_det_cic.items(), key=lambda x: -x[1]['detection_rate']):
    print(f"  ✓ {k:<30} DR={v['detection_rate']:.4f}")

print(f"\nCIC-2017 — Failed to detect (DR < 0.3): {len(low_det_cic)} attack types")
for k, v in sorted(low_det_cic.items(), key=lambda x: x[1]['detection_rate']):
    print(f"  ✗ {k:<30} DR={v['detection_rate']:.4f}")

print(f"\nCSE-2018 — Successfully detected (DR > 0.7): {len(high_det_cse)} attack types")
for k, v in sorted(high_det_cse.items(), key=lambda x: -x[1]['detection_rate']):
    print(f"  ✓ {k:<30} DR={v['detection_rate']:.4f}")

print(f"\nCSE-2018 — Failed to detect (DR < 0.3): {len(low_det_cse)} attack types")
for k, v in sorted(low_det_cse.items(), key=lambda x: x[1]['detection_rate']):
    print(f"  ✗ {k:<30} DR={v['detection_rate']:.4f}")

# Correlate with domain shift
if os.path.exists(f"{RESULTS_DIR}/domain_shift_analysis.csv"):
    ks_df = pd.read_csv(f"{RESULTS_DIR}/domain_shift_analysis.csv")
    high_shift_feats = ks_df[ks_df['shift'] == 'high']['feature'].tolist()
    print(f"\n[ANALISIS] Features with high domain shift: {len(high_shift_feats)}")
    print(f"  These features may explain poor CSE detection for some attack types.")
    print(f"  High-shift features: {high_shift_feats[:5]}{'...' if len(high_shift_feats) > 5 else ''}")


=== Attack-Level Analysis ===

CIC-2017 — Successfully detected (DR > 0.7): 3 attack types
  ✓ Heartbleed                     DR=1.0000
  ✓ Infiltration                   DR=0.9375
  ✓ DoS Hulk                       DR=0.7005

CIC-2017 — Failed to detect (DR < 0.3): 2 attack types
  ✗ Web Attack � XSS               DR=0.0303
  ✗ Web Attack � Brute Force       DR=0.0993

CSE-2018 — Successfully detected (DR > 0.7): 3 attack types
  ✓ DoS attacks-SlowHTTPTest       DR=0.9999
  ✓ FTP-BruteForce                 DR=0.9999
  ✓ DoS attacks-Slowloris          DR=0.7105

CSE-2018 — Failed to detect (DR < 0.3): 1 attack types
  ✗ DoS attacks-Hulk               DR=0.0400

[ANALISIS] Features with high domain shift: 10
  These features may explain poor CSE detection for some attack types.
  High-shift features: ['Flow IAT Min', 'Bwd Header Length', 'Bwd IAT Min', 'Bwd IAT Max', 'Bwd IAT Mean']...


In [37]:
# ============================================================
# 14.3 Generalization Verdict
# ============================================================
n_pass = statuses.count('✓')
n_warn = statuses.count('⚠')
n_fail = statuses.count('✗')

if n_pass >= 4 and roc_auc_cse > 0.70:
    verdict = "GENERAL"
    verdict_desc = ("Model successfully generalizes to CSE-2018. "
                    "Performance on unseen dataset is close to training dataset.")
elif roc_auc_cse > 0.55 and n_pass + n_warn >= 3:
    verdict = "PARTIAL"
    verdict_desc = ("Model partially generalizes. Some attack types are detected, "
                    "but significant performance gap exists between CIC and CSE.")
else:
    verdict = "CIC-SPECIFIC"
    verdict_desc = ("Model is overfit to CIC-2017 data distribution. "
                    "CSE-2018 performance is far below targets. "
                    "Domain adaptation techniques required for Phase 3.")

print(f"\n{'='*70}")
print(f"GENERALIZATION VERDICT: {verdict}")
print(f"{'='*70}")
print(f"\n{verdict_desc}")
print(f"\nMetric summary: {n_pass} pass, {n_warn} warning, {n_fail} fail")
print(f"ROC-AUC gap: CIC={roc_auc_cic:.4f} → CSE={roc_auc_cse:.4f} (Δ={roc_auc_cse - roc_auc_cic:+.4f})")

if roc_auc_cse < 0.5:
    print("\n[PERINGATAN KRITIS] ROC-AUC CSE < 0.5 — JANGAN lanjut ke Fase 3!")
    print("  Model perlu diagnosis fundamental sebelum refinement.")
    print("  Kemungkinan root cause:")
    print("  1. Domain shift terlalu besar pada fitur-fitur kunci")
    print("  2. Preprocessing tidak konsisten antara CIC dan CSE")
    print("  3. Model belajar noise CIC-specific, bukan anomaly patterns universal")


GENERALIZATION VERDICT: CIC-SPECIFIC

Model is overfit to CIC-2017 data distribution. CSE-2018 performance is far below targets. Domain adaptation techniques required for Phase 3.

Metric summary: 0 pass, 1 warning, 5 fail
ROC-AUC gap: CIC=0.8633 → CSE=0.2770 (Δ=-0.5863)

[PERINGATAN KRITIS] ROC-AUC CSE < 0.5 — JANGAN lanjut ke Fase 3!
  Model perlu diagnosis fundamental sebelum refinement.
  Kemungkinan root cause:
  1. Domain shift terlalu besar pada fitur-fitur kunci
  2. Preprocessing tidak konsisten antara CIC dan CSE
  3. Model belajar noise CIC-specific, bukan anomaly patterns universal


## 15. LAPORAN AKHIR

In [38]:
# ============================================================
# 15. Generate Final Report
# ============================================================
report_date = datetime.now().strftime('%Y-%m-%d %H:%M')

# Gather training info
try:
    s1_best_epoch = history_s1.history['val_loss'].index(min(history_s1.history['val_loss'])) + 1
    s1_best_val = min(history_s1.history['val_loss'])
except Exception:
    s1_best_epoch = 'N/A'
    s1_best_val = 'N/A'

try:
    s2_best_epoch = history_s2.history['val_loss'].index(min(history_s2.history['val_loss'])) + 1
    s2_best_val = min(history_s2.history['val_loss'])
except Exception:
    s2_best_epoch = 'N/A'
    s2_best_val = 'N/A'

# Load domain shift info
try:
    ks_df = pd.read_csv(f"{RESULTS_DIR}/domain_shift_analysis.csv")
    n_ks_low = (ks_df['shift'] == 'low').sum()
    n_ks_med = (ks_df['shift'] == 'medium').sum()
    n_ks_high = (ks_df['shift'] == 'high').sum()
except Exception:
    n_ks_low = n_ks_med = n_ks_high = 'N/A'

# Session stats
try:
    session_lens = pd.Series(train_session_ids).value_counts()
    median_sess = session_lens.median()
    pct_single_sess = (session_lens == 1).mean() * 100
    pct_padded = masks_train.any(axis=1).mean() * 100
except Exception:
    median_sess = pct_single_sess = pct_padded = 'N/A'

# Threshold comparison table
threshold_table = "| Method | Threshold | FPR | Precision | Recall | F1 |\n"
threshold_table += "|--------|-----------|-----|-----------|--------|-----|\n"
for t_name in sorted(threshold_results.keys()):
    t = threshold_results[t_name]
    m_cic = metrics_cic.get(t_name, {})
    threshold_table += (f"| {t_name} | {t['threshold']:.6f} | {t['fpr']:.4f} | "
                        f"{m_cic.get('precision', 0):.4f} | {m_cic.get('recall', 0):.4f} | "
                        f"{m_cic.get('f1', 0):.4f} |\n")

report = f"""# Laporan Akhir Fase 2 — MSCNN-BiLSTM AE
Tanggal: {report_date}

## 1. Arsitektur Final
- **Stage 1 MSCNN-AE**: Multi-Scale Conv1D (kernels 3,5,7) → GAP → Dense({S1_LATENT_DIM})
- **Stage 2 BiLSTM-AE**: Bidirectional LSTM (32→16) → Dense({S2_LATENT_DIM}) → RepeatVector → BiLSTM (16→32)
- Compression ratio Stage 1: {n_features} / {S1_LATENT_DIM} = {n_features/S1_LATENT_DIM:.1f}x
- Compression ratio Stage 2: ({WINDOW_SIZE} × {S1_LATENT_DIM}) / {S2_LATENT_DIM} = {(WINDOW_SIZE * S1_LATENT_DIM)/S2_LATENT_DIM:.1f}x
- Total params Stage 1: {model_s1.count_params():,}
- Total params Stage 2: {model_s2.count_params():,}

## 2. Feature Selection Summary
- Fitur awal: {len(df_numeric.columns) if 'df_numeric' in dir() else 'N/A'}
- Setelah NZV filter: {len(df_numeric.columns) - len(low_var_features) if 'low_var_features' in dir() else 'N/A'}
- Setelah corr filter: {n_features}
- Fitur final yang digunakan: {n_features}

## 3. Domain Shift Summary
- Fitur low shift (KS < 0.1): {n_ks_low}
- Fitur medium shift (0.1 ≤ KS < 0.3): {n_ks_med}
- Fitur high shift (KS ≥ 0.3): {n_ks_high}
- Rekomendasi: Keep all features, RobustScaler + clipping for mitigation

## 4. Session Windowing Statistics
- Window size: {WINDOW_SIZE}
- Median session length: {median_sess} flows
- % Sessions with 1 flow: {pct_single_sess}%
- % Windows padded: {pct_padded}%
- Mode: {'session-based' if WINDOW_SIZE > 1 else 'per-flow (fallback)'}

## 5. Training Summary
- Stage 1: stopped at epoch {s1_best_epoch}, best val_loss = {s1_best_val}
- Stage 2: stopped at epoch {s2_best_epoch}, best val_loss = {s2_best_val}

## 6. Threshold Comparison (CIC-2017 metrics)
{threshold_table}
Selected: **{best_threshold_name}** (threshold = {best_threshold_val:.6f})

## 7. Metrik Akhir

| Metric | CIC-2017 | CSE-2018 | Gap |
|--------|----------|----------|-----|
| ROC-AUC | {roc_auc_cic:.4f} | {roc_auc_cse:.4f} | {roc_auc_cse - roc_auc_cic:+.4f} |
| PR-AUC | {pr_auc_cic:.4f} | {pr_auc_cse:.4f} | {pr_auc_cse - pr_auc_cic:+.4f} |
| F1-Score | {best_cic['f1']:.4f} | {best_cse['f1']:.4f} | {best_cse['f1'] - best_cic['f1']:+.4f} |
| Recall | {best_cic['recall']:.4f} | {best_cse['recall']:.4f} | {best_cse['recall'] - best_cic['recall']:+.4f} |
| FPR | {best_cic['fpr']:.4f} | {best_cse['fpr']:.4f} | {best_cse['fpr'] - best_cic['fpr']:+.4f} |

## 8. Verdict Generalization
**{verdict}** — {verdict_desc}

## 9. Top 3 Temuan Penting
1. Compression ratio Stage 1 = {n_features/S1_LATENT_DIM:.1f}x forces model to learn compressed representations instead of memorizing
2. Session-based windowing in latent space (W={WINDOW_SIZE}) captures temporal patterns without raw-feature noise
3. ROC-AUC gap CIC→CSE = {roc_auc_cse - roc_auc_cic:+.4f} {'indicates successful generalization' if abs(roc_auc_cse - roc_auc_cic) < 0.15 else 'indicates domain shift challenge'}

## 10. Rekomendasi Fase 3
1. **Isolation Forest ensemble**: Use Stage 1 + Stage 2 reconstruction errors as features for IF
2. **Adaptive weighting**: Optimize STAGE1_WEIGHT/STAGE2_WEIGHT based on observed error distributions
3. **Domain adaptation**: If CSE gap persists, consider feature-wise normalization or adversarial alignment
"""

with open(f"{RESULTS_DIR}/fase2_report.md", 'w') as f:
    f.write(report)

print(report)
print(f"\n\nReport saved to: {RESULTS_DIR}/fase2_report.md")
print("\n" + "=" * 70)
print("FASE 2 COMPLETE")
print("=" * 70)

# Laporan Akhir Fase 2 — MSCNN-BiLSTM AE
Tanggal: 2026-02-28 16:44

## 1. Arsitektur Final
- **Stage 1 MSCNN-AE**: Multi-Scale Conv1D (kernels 3,5,7) → GAP → Dense(8)
- **Stage 2 BiLSTM-AE**: Bidirectional LSTM (32→16) → Dense(4) → RepeatVector → BiLSTM (16→32)
- Compression ratio Stage 1: 50 / 8 = 6.2x
- Compression ratio Stage 2: (10 × 8) / 4 = 20.0x
- Total params Stage 1: 33,193
- Total params Stage 2: 40,844

## 2. Feature Selection Summary
- Fitur awal: 77
- Setelah NZV filter: N/A
- Setelah corr filter: 50
- Fitur final yang digunakan: 50

## 3. Domain Shift Summary
- Fitur low shift (KS < 0.1): 6
- Fitur medium shift (0.1 ≤ KS < 0.3): 34
- Fitur high shift (KS ≥ 0.3): 10
- Rekomendasi: Keep all features, RobustScaler + clipping for mitigation

## 4. Session Windowing Statistics
- Window size: 10
- Median session length: 50.0 flows
- % Sessions with 1 flow: 4.3229513452196%
- % Windows padded: 4.4408001911422925%
- Mode: session-based

## 5. Training Summary
- Stage 1: stopped a

## 16. PHASE 3 - GENERALIZATION EXPERIMENTS (A/B/C/D)

Eksperimen ini mengeksekusi track A/B/C/D untuk meningkatkan generalization CSE tanpa mengubah arsitektur inti.
Set `RERUN_PHASE3_EXPERIMENTS = True` untuk menjalankan.

In [39]:
# ============================================================
# 16.1 Phase 3 Experiment Orchestrator (A/B/C/D)
# ============================================================
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

PHASE3_SUMMARY_CSV = f"{RESULTS_DIR}/phase3_ablation_summary.csv"
PHASE3_BEST_CONFIG_JSON = f"{ARTIFACTS_DIR}/phase3_best_config.json"
PHASE3_REPORT_MD = f"{RESULTS_DIR}/phase3_report.md"


def _phase3_thresholds_from_benign(scores):
    mu, sigma = scores.mean(), scores.std() + 1e-12
    q1, q3 = np.percentile(scores, 25), np.percentile(scores, 75)
    iqr = q3 - q1
    med = np.median(scores)
    mad = np.median(np.abs(scores - med)) + 1e-12

    out = {}
    for k in K_SIGMA_VALUES:
        out[f'zscore_k{k}'] = {'threshold': float(mu + k * sigma), 'fpr': float((scores > (mu + k * sigma)).mean())}
    for p in PERCENTILE_VALUES:
        th = np.percentile(scores, p)
        out[f'percentile_{p}'] = {'threshold': float(th), 'fpr': float((scores > th).mean())}
    for k in [1.5, 2.0, 3.0]:
        th = q3 + k * iqr
        out[f'iqr_k{k}'] = {'threshold': float(th), 'fpr': float((scores > th).mean())}
    for k in K_SIGMA_VALUES:
        th = med + k * 1.4826 * mad
        out[f'mad_k{k}'] = {'threshold': float(th), 'fpr': float((scores > th).mean())}
    return out


def _phase3_pick_threshold(thresholds, target_fpr=0.05):
    ordered = sorted(thresholds.items(), key=lambda kv: kv[1]['threshold'])
    for n, info in ordered:
        if info['fpr'] <= target_fpr:
            return n, info['threshold']
    n, info = ordered[-1]
    return n, info['threshold']


def _map_window_to_perflow(err_windows, flow_idx, n_rows):
    arr = np.full(n_rows, np.nan, dtype=np.float32)
    for w_idx in range(len(flow_idx)):
        fi = flow_idx[w_idx]
        valid = fi[fi >= 0]
        arr[valid] = err_windows[w_idx]
    miss = np.isnan(arr)
    if miss.any():
        arr[miss] = np.nanmedian(arr)
    return arr


def _fit_error_scalers(model_s1, model_s2, X_val_local, windows_val_local, flow_idx_val_local):
    err_s1_val = compute_recon_error(model_s1, X_val_local, batch_size=S1_BATCH_SIZE)
    err_s2_win = compute_recon_error(model_s2, windows_val_local, batch_size=S2_BATCH_SIZE)
    err_s2_val = _map_window_to_perflow(err_s2_win, flow_idx_val_local, len(X_val_local))

    s1_scaler = MinMaxScaler().fit(err_s1_val.reshape(-1, 1))
    s2_scaler = MinMaxScaler().fit(err_s2_val.reshape(-1, 1))

    e1 = np.clip(s1_scaler.transform(err_s1_val.reshape(-1, 1)).ravel(), 0, 1)
    e2 = np.clip(s2_scaler.transform(err_s2_val.reshape(-1, 1)).ravel(), 0, 1)
    return {'s1': s1_scaler, 's2': s2_scaler}, e1, e2


def _extract_latent_for(model_encoder, X_arr):
    return model_encoder.predict(X_arr[..., np.newaxis], batch_size=S1_BATCH_SIZE, verbose=0)


def _prepare_cse_benign_adapt_sample(max_rows, random_seed=42):
    idx_b = np.where(y_cse == 0)[0]
    if len(idx_b) < 2:
        raise ValueError('CSE benign rows are insufficient for adaptation split (need at least 2).')

    rng = np.random.default_rng(random_seed)
    rng.shuffle(idx_b)
    idx_sel = idx_b[:min(len(idx_b), max_rows)]

    Xb = X_cse[idx_sel]
    sb = sids_cse[idx_sel].astype(str)
    tb = ts_cse[idx_sel]

    split = int(len(idx_sel) * 0.8)
    split = min(max(split, 1), len(idx_sel) - 1)

    return (
        Xb[:split], sb[:split], tb[:split],
        Xb[split:], sb[split:], tb[split:]
    )


def _train_stage1(Xtr, Xva, epochs, lr, ckpt_path):
    m1, enc = build_mscnn_ae(n_features, S1_LATENT_DIM, S1_DROPOUT)
    m1.compile(optimizer=Adam(learning_rate=lr), loss='mse')

    cb = [
        EarlyStopping(monitor='val_loss', patience=S1_ES_PATIENCE, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=S1_LR_PATIENCE, min_lr=1e-6, verbose=1),
        ModelCheckpoint(ckpt_path, monitor='val_loss', save_best_only=True, verbose=1)
    ]

    m1.fit(
        Xtr[..., np.newaxis], Xtr,
        validation_data=(Xva[..., np.newaxis], Xva),
        epochs=epochs,
        batch_size=S1_BATCH_SIZE,
        callbacks=cb,
        verbose=1,
    )
    m1.load_weights(ckpt_path)
    return m1, enc


def _train_stage2(wtr, wva, epochs, lr, ckpt_path):
    m2 = build_lstm_ae(wtr.shape[1], wtr.shape[2], S2_LATENT_DIM, S2_DROPOUT)
    m2.compile(optimizer=Adam(learning_rate=lr), loss='mse')

    cb = [
        EarlyStopping(monitor='val_loss', patience=S2_ES_PATIENCE, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=S2_LR_PATIENCE, min_lr=1e-6, verbose=1),
        ModelCheckpoint(ckpt_path, monitor='val_loss', save_best_only=True, verbose=1)
    ]

    m2.fit(
        wtr, wtr,
        validation_data=(wva, wva),
        epochs=epochs,
        batch_size=S2_BATCH_SIZE,
        callbacks=cb,
        verbose=1,
    )
    m2.load_weights(ckpt_path)
    return m2


def _evaluate_track(track_name, model_s1_t, enc_s1_t, model_s2_t, err_scalers_t, threshold_results_t, best_thr_name_t):
    scores_cic_t, err1_cic_t, err2_cic_t = compute_eval_scores(
        X_cic, sids_cic, ts_cic,
        model_s1_t, enc_s1_t, model_s2_t,
        err_scalers_t['s1'], err_scalers_t['s2'],
        WINDOW_SIZE, MIN_FLOWS_PER_SESSION
    )
    metrics_cic_t, roc_cic_t, pr_cic_t = evaluate_with_thresholds(y_cic, scores_cic_t, threshold_results_t, f'{track_name}-CIC')

    scores_cse_t, err1_cse_t, err2_cse_t = compute_eval_scores(
        X_cse, sids_cse, ts_cse,
        model_s1_t, enc_s1_t, model_s2_t,
        err_scalers_t['s1'], err_scalers_t['s2'],
        WINDOW_SIZE, MIN_FLOWS_PER_SESSION
    )
    metrics_cse_t, roc_cse_t, pr_cse_t = evaluate_with_thresholds(y_cse, scores_cse_t, threshold_results_t, f'{track_name}-CSE')

    best_cic_t = metrics_cic_t[best_thr_name_t]
    best_cse_t = metrics_cse_t[best_thr_name_t]

    return {
        'track': track_name,
        'best_threshold_name': best_thr_name_t,
        'roc_auc_cic': float(roc_cic_t),
        'pr_auc_cic': float(pr_cic_t),
        'f1_cic': float(best_cic_t['f1']),
        'recall_cic': float(best_cic_t['recall']),
        'precision_cic': float(best_cic_t['precision']),
        'fpr_cic': float(best_cic_t['fpr']),
        'roc_auc_cse': float(roc_cse_t),
        'pr_auc_cse': float(pr_cse_t),
        'f1_cse': float(best_cse_t['f1']),
        'recall_cse': float(best_cse_t['recall']),
        'precision_cse': float(best_cse_t['precision']),
        'fpr_cse': float(best_cse_t['fpr']),
        'gap_roc': float(roc_cse_t - roc_cic_t),
        'gap_f1': float(best_cse_t['f1'] - best_cic_t['f1']),
    }


def _track_objective(row):
    # Higher is better
    score = row['roc_auc_cse'] * 100 + row['f1_cse'] * 10
    if row['fpr_cic'] > 0.05:
        score -= 10
    if row['fpr_cse'] > 0.20:
        score -= 5
    return score


def _run_track_a_baseline_clean():
    print('\n=== Track A: baseline clean retrain (CIC only) ===')
    s1_ckpt = f"{CHECKPOINT_DIR}/{EXPERIMENT_TAG}_A_stage1.keras"
    s2_ckpt = f"{CHECKPOINT_DIR}/{EXPERIMENT_TAG}_A_stage2.keras"

    m1, enc = _train_stage1(X_train, X_val, S1_MAX_EPOCHS, S1_LR, s1_ckpt)
    lat_tr = _extract_latent_for(enc, X_train)
    lat_va = _extract_latent_for(enc, X_val)
    wtr, mtr, fitr = create_session_windows(lat_tr, train_session_ids, train_timestamps, WINDOW_SIZE, MIN_FLOWS_PER_SESSION)
    wva, mva, fiva = create_session_windows(lat_va, val_session_ids, val_timestamps, WINDOW_SIZE, MIN_FLOWS_PER_SESSION)

    m2 = _train_stage2(wtr, wva, S2_MAX_EPOCHS, S2_LR, s2_ckpt)

    err_scalers, e1, e2 = _fit_error_scalers(m1, m2, X_val, wva, fiva)
    comb_val = STAGE1_WEIGHT * e1 + STAGE2_WEIGHT * e2
    thr = _phase3_thresholds_from_benign(comb_val)
    best_name, best_val = _phase3_pick_threshold(thr, target_fpr=0.05)

    res = _evaluate_track('A_baseline_clean', m1, enc, m2, err_scalers, thr, best_name)
    res.update({'objective_score': _track_objective(res), 'stage1_ckpt': s1_ckpt, 'stage2_ckpt': s2_ckpt})
    return res, {'model_s1': m1, 'encoder_s1': enc, 'model_s2': m2, 'err_scalers': err_scalers, 'thresholds': thr, 'best_threshold_name': best_name}


def _run_track_b_cse_finetune(base_row):
    print('\n=== Track B: CSE benign finetune ===')
    Xtr_b, sid_tr_b, ts_tr_b, Xva_b, sid_va_b, ts_va_b = _prepare_cse_benign_adapt_sample(CSE_BENIGN_ADAPT_MAX_ROWS, RANDOM_SEED)

    s1_ckpt = f"{CHECKPOINT_DIR}/{EXPERIMENT_TAG}_B_stage1.keras"
    s2_ckpt = f"{CHECKPOINT_DIR}/{EXPERIMENT_TAG}_B_stage2.keras"

    m1, enc = build_mscnn_ae(n_features, S1_LATENT_DIM, S1_DROPOUT)
    m1.load_weights(base_row['stage1_ckpt'])
    m1.compile(optimizer=Adam(learning_rate=S1_LR * FINETUNE_LR_SCALE), loss='mse')
    m1.fit(
        Xtr_b[..., np.newaxis], Xtr_b,
        validation_data=(Xva_b[..., np.newaxis], Xva_b),
        epochs=S1_FINETUNE_EPOCHS,
        batch_size=S1_BATCH_SIZE,
        verbose=1,
        callbacks=[ModelCheckpoint(s1_ckpt, monitor='val_loss', save_best_only=True, verbose=1)]
    )
    m1.load_weights(s1_ckpt)

    lat_tr = _extract_latent_for(enc, Xtr_b)
    lat_va = _extract_latent_for(enc, Xva_b)
    sid_tr_pref = np.array([f'cse_{x}' for x in sid_tr_b])
    sid_va_pref = np.array([f'cse_{x}' for x in sid_va_b])
    wtr, _, _ = create_session_windows(lat_tr, sid_tr_pref, ts_tr_b, WINDOW_SIZE, MIN_FLOWS_PER_SESSION)
    wva, _, fiva = create_session_windows(lat_va, sid_va_pref, ts_va_b, WINDOW_SIZE, MIN_FLOWS_PER_SESSION)

    m2 = build_lstm_ae(wtr.shape[1], wtr.shape[2], S2_LATENT_DIM, S2_DROPOUT)
    m2.load_weights(base_row['stage2_ckpt'])
    m2.compile(optimizer=Adam(learning_rate=S2_LR * FINETUNE_LR_SCALE), loss='mse')
    m2.fit(
        wtr, wtr,
        validation_data=(wva, wva),
        epochs=S2_FINETUNE_EPOCHS,
        batch_size=S2_BATCH_SIZE,
        verbose=1,
        callbacks=[ModelCheckpoint(s2_ckpt, monitor='val_loss', save_best_only=True, verbose=1)]
    )
    m2.load_weights(s2_ckpt)

    if RECALIBRATE_THRESHOLD_ON_MIXED_VAL:
        X_mix_val = np.vstack([X_val, Xva_b])
        sid_mix = np.concatenate([np.array([f'cic_{s}' for s in val_session_ids.astype(str)]), sid_va_pref])
        ts_mix = np.concatenate([val_timestamps, ts_va_b])
        lat_mix = _extract_latent_for(enc, X_mix_val)
        wmix, _, fimix = create_session_windows(lat_mix, sid_mix, ts_mix, WINDOW_SIZE, MIN_FLOWS_PER_SESSION)
        err_scalers, e1, e2 = _fit_error_scalers(m1, m2, X_mix_val, wmix, fimix)
    else:
        lat_val = _extract_latent_for(enc, X_val)
        wval_cic, _, fival_cic = create_session_windows(lat_val, val_session_ids.astype(str), val_timestamps, WINDOW_SIZE, MIN_FLOWS_PER_SESSION)
        err_scalers, e1, e2 = _fit_error_scalers(m1, m2, X_val, wval_cic, fival_cic)

    comb_val = STAGE1_WEIGHT * e1 + STAGE2_WEIGHT * e2
    thr = _phase3_thresholds_from_benign(comb_val)
    best_name, _ = _phase3_pick_threshold(thr, target_fpr=0.05)

    res = _evaluate_track('B_cse_finetune', m1, enc, m2, err_scalers, thr, best_name)
    res.update({'objective_score': _track_objective(res), 'stage1_ckpt': s1_ckpt, 'stage2_ckpt': s2_ckpt})
    return res, {'model_s1': m1, 'encoder_s1': enc, 'model_s2': m2, 'err_scalers': err_scalers, 'thresholds': thr, 'best_threshold_name': best_name}


def _run_track_c_mixed_train():
    print('\n=== Track C: mixed-domain training ===')
    Xtr_b, sid_tr_b, ts_tr_b, Xva_b, sid_va_b, ts_va_b = _prepare_cse_benign_adapt_sample(CSE_BENIGN_ADAPT_MAX_ROWS, RANDOM_SEED + 7)

    n_cic_keep = int(min(len(X_train), len(Xtr_b) * (ADAPT_MIX_RATIO_CIC / max(ADAPT_MIX_RATIO_CSE, 1e-6))))
    idx_cic = np.random.default_rng(RANDOM_SEED + 11).choice(len(X_train), size=max(1, n_cic_keep), replace=False)

    Xtr_mix = np.vstack([X_train[idx_cic], Xtr_b])
    sid_mix_tr = np.concatenate([np.array([f'cic_{s}' for s in train_session_ids[idx_cic].astype(str)]), np.array([f'cse_{s}' for s in sid_tr_b])])
    ts_mix_tr = np.concatenate([train_timestamps[idx_cic], ts_tr_b])

    Xva_mix = np.vstack([X_val, Xva_b])
    sid_mix_va = np.concatenate([np.array([f'cic_{s}' for s in val_session_ids.astype(str)]), np.array([f'cse_{s}' for s in sid_va_b])])
    ts_mix_va = np.concatenate([val_timestamps, ts_va_b])

    s1_ckpt = f"{CHECKPOINT_DIR}/{EXPERIMENT_TAG}_C_stage1.keras"
    s2_ckpt = f"{CHECKPOINT_DIR}/{EXPERIMENT_TAG}_C_stage2.keras"

    m1, enc = _train_stage1(Xtr_mix, Xva_mix, S1_MAX_EPOCHS, S1_LR, s1_ckpt)

    lat_tr = _extract_latent_for(enc, Xtr_mix)
    lat_va = _extract_latent_for(enc, Xva_mix)
    wtr, _, fitr = create_session_windows(lat_tr, sid_mix_tr, ts_mix_tr, WINDOW_SIZE, MIN_FLOWS_PER_SESSION)
    wva, _, fiva = create_session_windows(lat_va, sid_mix_va, ts_mix_va, WINDOW_SIZE, MIN_FLOWS_PER_SESSION)

    m2 = _train_stage2(wtr, wva, S2_MAX_EPOCHS, S2_LR, s2_ckpt)

    err_scalers, e1, e2 = _fit_error_scalers(m1, m2, Xva_mix, wva, fiva)
    comb_val = STAGE1_WEIGHT * e1 + STAGE2_WEIGHT * e2
    thr = _phase3_thresholds_from_benign(comb_val)
    best_name, _ = _phase3_pick_threshold(thr, target_fpr=0.05)

    res = _evaluate_track('C_mixed_train', m1, enc, m2, err_scalers, thr, best_name)
    res.update({'objective_score': _track_objective(res), 'stage1_ckpt': s1_ckpt, 'stage2_ckpt': s2_ckpt})
    return res, {'model_s1': m1, 'encoder_s1': enc, 'model_s2': m2, 'err_scalers': err_scalers, 'thresholds': thr, 'best_threshold_name': best_name}


def _run_track_d_score_calibrated(base_track_name, base_bundle):
    print(f'\n=== Track D: scoring recalibration (base={base_track_name}) ===')
    if base_bundle is None:
        raise ValueError(f'Base bundle for {base_track_name} is None; cannot run Track D calibration.')
    m1 = base_bundle['model_s1']
    enc = base_bundle['encoder_s1']
    m2 = base_bundle['model_s2']
    err_scalers = base_bundle['err_scalers']

    # Build mixed validation (CIC val + CSE benign sample)
    Xtr_b, sid_tr_b, ts_tr_b, Xva_b, sid_va_b, ts_va_b = _prepare_cse_benign_adapt_sample(min(CSE_BENIGN_ADAPT_MAX_ROWS, 120000), RANDOM_SEED + 101)
    Xmix = np.vstack([X_val, Xva_b])
    sid_mix = np.concatenate([np.array([f'cic_{s}' for s in val_session_ids.astype(str)]), np.array([f'cse_{s}' for s in sid_va_b])])
    ts_mix = np.concatenate([val_timestamps, ts_va_b])

    # component norms on mixed benign
    err1_mix = compute_recon_error(m1, Xmix, batch_size=S1_BATCH_SIZE)
    lat_mix = _extract_latent_for(enc, Xmix)
    wmix, _, fimix = create_session_windows(lat_mix, sid_mix, ts_mix, WINDOW_SIZE, MIN_FLOWS_PER_SESSION)
    err2w_mix = compute_recon_error(m2, wmix, batch_size=S2_BATCH_SIZE)
    err2_mix = _map_window_to_perflow(err2w_mix, fimix, len(Xmix))

    e1_mix = np.clip(err_scalers['s1'].transform(err1_mix.reshape(-1,1)).ravel(), 0, 1)
    e2_mix = np.clip(err_scalers['s2'].transform(err2_mix.reshape(-1,1)).ravel(), 0, 1)

    # component norms on CIC/CSE for eval
    def _dataset_norms(Xd, sids_d, ts_d):
        err1 = compute_recon_error(m1, Xd, batch_size=S1_BATCH_SIZE)
        lat = _extract_latent_for(enc, Xd)
        w, _, fi = create_session_windows(lat, sids_d, ts_d, WINDOW_SIZE, MIN_FLOWS_PER_SESSION)
        err2w = compute_recon_error(m2, w, batch_size=S2_BATCH_SIZE)
        err2 = _map_window_to_perflow(err2w, fi, len(Xd))
        n1 = np.clip(err_scalers['s1'].transform(err1.reshape(-1,1)).ravel(), 0, 1)
        n2 = np.clip(err_scalers['s2'].transform(err2.reshape(-1,1)).ravel(), 0, 1)
        return n1, n2

    cic_n1, cic_n2 = _dataset_norms(X_cic, sids_cic, ts_cic)
    cse_n1, cse_n2 = _dataset_norms(X_cse, sids_cse, ts_cse)

    candidates = []
    for w1, w2 in [(0.7, 0.3), (0.5, 0.5), (0.3, 0.7)]:
        val_scores = w1 * e1_mix + w2 * e2_mix
        thr = _phase3_thresholds_from_benign(val_scores)
        best_name, _ = _phase3_pick_threshold(thr, target_fpr=0.05)

        scores_cic = w1 * cic_n1 + w2 * cic_n2
        m_cic, roc_cic, pr_cic = evaluate_with_thresholds(y_cic, scores_cic, thr, f'D-{w1:.1f}-{w2:.1f}-CIC')

        scores_cse = w1 * cse_n1 + w2 * cse_n2
        m_cse, roc_cse, pr_cse = evaluate_with_thresholds(y_cse, scores_cse, thr, f'D-{w1:.1f}-{w2:.1f}-CSE')

        bc = m_cic[best_name]
        bs = m_cse[best_name]
        row = {
            'track': f'D_score_calibrated_{w1:.1f}_{w2:.1f}',
            'best_threshold_name': best_name,
            'roc_auc_cic': float(roc_cic), 'pr_auc_cic': float(pr_cic),
            'f1_cic': float(bc['f1']), 'recall_cic': float(bc['recall']), 'precision_cic': float(bc['precision']), 'fpr_cic': float(bc['fpr']),
            'roc_auc_cse': float(roc_cse), 'pr_auc_cse': float(pr_cse),
            'f1_cse': float(bs['f1']), 'recall_cse': float(bs['recall']), 'precision_cse': float(bs['precision']), 'fpr_cse': float(bs['fpr']),
            'gap_roc': float(roc_cse - roc_cic),
            'gap_f1': float(bs['f1'] - bc['f1']),
            'stage1_weight': w1,
            'stage2_weight': w2,
        }
        row['objective_score'] = _track_objective(row)
        candidates.append(row)

    best = max(candidates, key=lambda r: r['objective_score'])
    return best, candidates


def _phase3_generate_report(df_summary, best_cfg):
    lines = []
    lines.append('# Phase 3 Generalization Report')
    lines.append(f'Date: {datetime.now().strftime("%Y-%m-%d %H:%M")}')
    lines.append('')
    lines.append('## Objectives')
    lines.append('- CSE ROC-AUC > 0.50 (minimum), target >= 0.70')
    lines.append('- CSE F1 >= 0.35')
    lines.append('- CIC FPR <= 0.05')
    lines.append('')
    lines.append('## Ablation Summary')
    lines.append(df_summary.to_markdown(index=False))
    lines.append('')
    lines.append('## Best Configuration')
    for k, v in best_cfg.items():
        lines.append(f'- {k}: {v}')
    lines.append('')
    lines.append('## Notes')
    lines.append('- Score inversion is not used as final solution.')
    lines.append('- Focus is model generalization, not schema cleanup (already stabilized).')

    text = '\n'.join(lines)
    with open(PHASE3_REPORT_MD, 'w', encoding='utf-8') as f:
        f.write(text)
    return text


if RERUN_PHASE3_EXPERIMENTS:
    # Freeze before metrics (best threshold from current run)
    before_row = {
        'track': 'before_current',
        'best_threshold_name': best_threshold_name,
        'roc_auc_cic': float(roc_auc_cic),
        'pr_auc_cic': float(pr_auc_cic),
        'f1_cic': float(best_cic['f1']),
        'recall_cic': float(best_cic['recall']),
        'precision_cic': float(best_cic['precision']),
        'fpr_cic': float(best_cic['fpr']),
        'roc_auc_cse': float(roc_auc_cse),
        'pr_auc_cse': float(pr_auc_cse),
        'f1_cse': float(best_cse['f1']),
        'recall_cse': float(best_cse['recall']),
        'precision_cse': float(best_cse['precision']),
        'fpr_cse': float(best_cse['fpr']),
        'gap_roc': float(roc_auc_cse - roc_auc_cic),
        'gap_f1': float(best_cse['f1'] - best_cic['f1']),
        'objective_score': None,
    }

    results = [before_row]

    # Track A
    res_a, bundle_a = _run_track_a_baseline_clean()
    results.append(res_a)

    # Track B
    if ENABLE_DOMAIN_ADAPTATION:
        res_b, bundle_b = _run_track_b_cse_finetune(res_a)
        results.append(res_b)
    else:
        bundle_b = None

    # Track C
    if ENABLE_DOMAIN_ADAPTATION:
        res_c, bundle_c = _run_track_c_mixed_train()
        results.append(res_c)
    else:
        bundle_c = None

    # Choose best among A/B/C
    candidates_base = [r for r in results if r['track'] in ['A_baseline_clean', 'B_cse_finetune', 'C_mixed_train']]
    if not candidates_base:
        raise ValueError('No base track result available (A/B/C) to continue with Track D.')
    best_base = max(candidates_base, key=lambda r: r['objective_score'])

    bundle_lookup = {
        'A_baseline_clean': bundle_a,
        'B_cse_finetune': bundle_b,
        'C_mixed_train': bundle_c,
    }

    # Track D
    res_d_best, res_d_all = _run_track_d_score_calibrated(best_base['track'], bundle_lookup[best_base['track']])
    results.extend(res_d_all)
    print(f"Best Track D candidate: {res_d_best['track']} (objective={res_d_best['objective_score']:.4f})")

    df_sum = pd.DataFrame(results)
    df_sum.to_csv(PHASE3_SUMMARY_CSV, index=False)

    best_overall = max([r for r in results if r['track'] != 'before_current'], key=lambda r: r['objective_score'])
    with open(PHASE3_BEST_CONFIG_JSON, 'w', encoding='utf-8') as f:
        json.dump(best_overall, f, indent=2)

    report_text = _phase3_generate_report(df_sum, best_overall)
    print(report_text)
    print(f'\nSaved: {PHASE3_SUMMARY_CSV}')
    print(f'Saved: {PHASE3_BEST_CONFIG_JSON}')
    print(f'Saved: {PHASE3_REPORT_MD}')
else:
    print('[INFO] RERUN_PHASE3_EXPERIMENTS=False. Set True to run tracks A/B/C/D.')
    print(f'Artifacts target: {PHASE3_SUMMARY_CSV}, {PHASE3_BEST_CONFIG_JSON}, {PHASE3_REPORT_MD}')


[INFO] RERUN_PHASE3_EXPERIMENTS=False. Set True to run tracks A/B/C/D.
Artifacts target: /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/results/phase3_ablation_summary.csv, /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/artifacts/v3_sessionfix/phase3_best_config.json, /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/results/phase3_report.md
